# CfC / BAOAB — **JOINT-coupled** Anisotropic Gaussian V_θ + **QK-normed** creation gate — OpenWebText d=384

## What this arm is

A fork of `colab_fock_cfc_baoab_aniso_gaussian_openwebtext_d384.ipynb`
(the additive-coupling baseline, at ~86 PPL by step 81K), changing the two
things that **cannot** be applied to that run as a continuation, because
both change parameter shapes:

| | baseline arm | this arm |
|---|---|---|
| `V_THETA_COUPLING` | `'additive'` — one bank per ξ channel, potentials summed (mixture / OR) | `'joint'` — one bank over the concatenated context (conjunction / AND) |
| `CREATION_QK_NORM` | `False` — raw `Q·K` ÷ learned per-register `log_tau`, both channels unbounded | `True` — cosine scores × clamped per-register `logit_scale`; `log_tau` not registered |
| `V_THETA_WELLS_PER_HEAD` | 8 (×5 banks = 40 attractors) | 8 (×1 bank = 8 attractors) — **parameter-matched to 0.998×**, see Cell 0 |
| `TOTAL_STEPS` / `PROBE_MAX_STEPS` | 100,000 / `None` | 150,000 / **15,000** (pilot stop) |

Everything else is held identical to the baseline for comparability:
`d=384`, `L=8`, `M=32`, `rank=4`, the `5long` ξ preset, `γ=0.10`,
`LR=3e-4`, `SEED=0`, and every stability mitigation the baseline arm
earned the hard way — `PRECISION_LR_MAX=1.0` (on from step 0 here, which
the baseline only switched on at step 47K, and which
`Analytic_Multi_Channel_Integration_in_Structured_Vtheta.md` §5
specifically recommends pairing with joint coupling), `NO_DECAY_1D`,
`clip_then_sum` on `E`/`P`, and the per-group clip overrides.

### This is a pilot first

`PROBE_MAX_STEPS = 15_000` stops cleanly at 15K **without touching**
`TOTAL_STEPS`, so the WSD windows are the production arm's. The pilot is
therefore literally the first 15,000 steps of the full run: if it passes,
set `PROBE_MAX_STEPS = None` and the same run continues from its own
checkpoint. Nothing is re-run.

It is a **falsifier, not a certificate** — §24.4's precedent (the d=768
blowup arrived at step ~37,000 after 33,000 clean steps) means 15K steps
can kill this arm but cannot bless it. See the `PROBE_MAX_STEPS` comment
in Cell 0 for the five things it *does* answer, and the
`V_THETA_COUPLING` comment for why `K` stays at 8 rather than rising to
40 as the multi-channel note's §8 suggests.

### Companion notes

- `Analytic_Multi_Channel_Integration_in_Structured_Vtheta.md` — the
  coupling ladder, closed-form product-of-experts fusion, the
  curvature-concentration caveat (§5), and predictions J1–J4.
- `Register_Temperature_Instability_in_the_Fock_Creation_Gate.md` — the
  covariance identity, the Riccati loop, why QK-norm must *replace*
  rather than multiply the temperature (§10.2–§10.4), and §13's
  deployment planning across arms.
- `CfC_BAOAB_Integrator_and_Mitigations.md` — everything the baseline arm
  learned about spikes, clipping, and the watchdog.

## Why the underlying CfC/BAOAB run exists

The `γ=0.10` and `γ=0.30` d=384 runs of
`colab_fock_aniso_gaussian_fockreg_openwebtext.ipynb` both hit chronic
gradient-spike instability under the damped velocity-Verlet integrator.
The `γ=0.10` arm eventually stalled outright: 13 watchdog reloads between
steps 10K and 17K, **zero** PPL improvement over 6,900 steps, and a
record pre-clip gradient of 263,084.

Two structural properties of the Verlet step cause this:

1. **The stiff part of the force is integrated explicitly.** A token in a
   sharp V_θ well has large local curvature `K`; the explicit update is
   stable only while `dt < 2·sqrt(m/K)`. As wells sharpen during training
   the layer step silently crosses that bound and the state amplifies
   geometrically down the remaining layers — which is what a gradient
   spike looks like from outside.
2. **V_θ sits inside the second-order `create_graph` chain**, because the
   force comes from `autograd.grad(V_θ + V_φ, create_graph=True)`.

This notebook runs the fix, in attributable stages, via a single
`INTEGRATOR` switch:

| `INTEGRATOR` | V_θ force | Integrator | What it isolates |
|---|---|---|---|
| `'verlet'` | autograd | damped velocity-Verlet | the existing baseline, bit-identical to the runs above |
| `'analytic_vtheta'` | **closed form** | damped velocity-Verlet | how much of the spiking is the V_θ half of the `create_graph` cascade |
| `'baoab'` | closed form | ABOBA split, exact `exp(-γdt)` friction | the integrator split alone, no CfC |
| `'baoab_cfc'` | closed form | ABOBA split + **closed-form harmonic propagator** for the stiff part of V_θ | the full fix |

Each arm writes to its own Drive folder (the integrator is part of the
variant tag), so arms can be run one at a time and resumed independently.

## What is guaranteed, and by what

- **`'analytic_vtheta'` is the same model, not a different one.**
  `test_cfc_baoab.py::test_analytic_vtheta_equivalence` asserts that
  switching V_θ's force to its closed form leaves the loss and *every
  parameter gradient* unchanged (worst relative error ~2e-6).
- **CfC changes how the force is integrated, not what the force is.**
  The stiff diagonal part of V_θ is propagated by its exact harmonic
  solution and the residual is kicked numerically; the two sum to the
  unmodified total force.
  `test_cfc_baoab.py::test_cfc_force_preservation` verifies the CfC and
  plain-BAOAB steps agree to O(dt³) — a second-order discrepancy would
  mean the force field had changed.
- **The propagator cannot blow up.** All Gaussian wells are attractive, so
  the stiffness is non-negative and the substep is always a bounded
  rotation in phase space. At `K = 10⁴` (ω·dt = 100), twelve explicit
  steps overflow float32 while the CfC step stays inside its initial
  orbit.

Theory: `companion_notes/Closed_Form_and_Hybrid_Integration_Strategies_for_Fock-PARFLM.md`,
`companion_notes/Blended_CfC_BAOAB_Deep_Dive.md`, and `paper_v5` §20.
Implementation: `parf/cfc_baoab.py` (propagator), `parf/model_parf_multixi.py`
(`_layer_step_langevin`), `parf/model_aniso_gaussian_vtheta.py`
(`harmonic_terms`).

## Everything else is held fixed

Same d=384 / L=16 / M=32 architecture, same 5-channel ξ, same anisotropic
depth-conditioned V_θ (5 heads × 8 wells, rank 4), same Fock coupling
regularisation, same WSD schedule, same per-group gradient clips, same
watchdog. Only the integrator changes.


In [ ]:
# == Cell 0: Configuration =============================================

# -- V_theta: Anisotropic Gaussian (diagonal + low-rank precision) -----
V_THETA_VARIANT             = 'aniso_gaussian'
V_THETA_WELLS_PER_HEAD      = 8
V_THETA_DEPTH_CONDITION     = True
V_THETA_DEPTH_CODE_INIT_STD = 0.02
ANISO_RANK                  = 4
W_SCALE                     = 1.0

# -- V_theta channel coupling (Analytic_Multi_Channel_Integration note) --
# 'additive' (the d384 baseline arm): one well bank per xi channel, the
#     per-channel potentials SUMMED. The channel sum sits outside the
#     exponent, so V is a mixture -- attracted to horizon A OR horizon B,
#     never their conjunction. Its channel-input Hessian is exactly
#     block-diagonal (that note's Figure 1), so no number of extra
#     channels can build a feature that fires only when horizons AGREE.
# 'joint'  (this arm): ONE bank whose mu_k, a_k, B_k, w_k are read from
#     the CONCATENATED context xi = [xi^(1); ...; xi^(n_ctx)], so every
#     well depends on all horizons at once. Additive is a strict special
#     case, so expressivity only rises. Analyticity is untouched -- the
#     well parameters still depend on xi only, never on h -- so the
#     closed-form force, the closed-form Hessian and the CfC-BAOAB
#     harmonic split all survive unchanged (that note's SS2/SS6).
#
# WHY K IS NOT RAISED TO n_ctx*K. That note's SS8 suggests K = n_ctx *
# K_additive to match the additive attractor budget (5*8 = 40). Measured,
# that quintuples V_theta, because the joint bank's input is n_ctx*d=1920
# wide AND K is 5x larger, so the projections scale as n_ctx^2:
#     additive K=8 : 35,512,360 V_theta params   (5 banks)
#     joint    K=8 : 35,438,600 V_theta params   (1 bank)  = 0.998x
#     joint    K=40: 177,131,560 V_theta params  (1 bank)  = 4.99x
# V_theta is already ~46% of this model's 76.8M parameters, so K=40 would
# take the whole model to ~218M -- a 2.8x bigger model, not a coupling
# ablation. K=8 is parameter-matched to the additive arm to within 0.2%
# (n_ctx x (d -> K*d) and (n_ctx*d) -> K*d are the same matrix size), so
# it isolates COUPLING from CAPACITY, which is exactly what that note's
# prediction J4 asks for. The cost is 8 joint attractors instead of 40
# additive ones -- the real price of conjunction, and the thing the A/B
# is measuring. (Note also that J1 as written -- "joint beats additive at
# equal K wells and equal params" -- is unsatisfiable: you can match
# wells or match params, not both. This arm matches params.)
#
# SS5 of that note is the risk: fusion CONCENTRATES curvature (precision
# sums across channels), which is precisely the sigma_max(B_k)^2 spike
# surface of the Mitigations note SS41. Hence PRECISION_LR_MAX is ON from
# step 0 in this arm (see below), per that note's own recommendation.
V_THETA_COUPLING = 'joint'

# -- Xi channels (5long preset from OWT notebook) ----------------------
XI_OVERRIDE     = '5long'
_XI_PRESETS_CFG = {
    5:       [0.25, 0.50, 0.75, 0.95, 0.99],
    '5long': [0.50, 0.75, 0.95, 0.99, 0.995],
    6:       [0.25, 0.50, 0.75, 0.95, 0.99, 0.995],
    '4long': [0.50, 0.75, 0.95, 0.995],
}
XI_ALPHA_INITS = _XI_PRESETS_CFG[XI_OVERRIDE]
XI_CHANNELS    = len(XI_ALPHA_INITS)
V_THETA_N_HEADS = XI_CHANNELS

# -- PARF V_phi --------------------------------------------------------
V_PHI_KIND      = 'structural_competitive'
V_PHI_MLP_HIDDEN = 128
TOP_K           = 16
V_PHI_N_HEADS   = 4
V_PHI_D_TYPE    = 32
V_PHI_D_ANGLE   = 16

# -- Reverse channel stabilisation (E5c) -------------------------------
REVERSE_CHANNEL              = True
REVERSE_CHANNEL_STABLE       = True
REVERSE_CHANNEL_PRE_LN       = True
REVERSE_CHANNEL_SOFT_NORM    = True
REVERSE_CHANNEL_WARMUP_STEPS = 4000
REVERSE_CHANNEL_PER_LAYER    = True
REVERSE_CHANNEL_RESET_SCALE  = False

# -- Creation-gate QK-normalisation (Register_Temperature note SS10-SS11) --
# QKVCreationGate_v21 never received the hardening ReverseChannel got
# under stable=True. It computes raw scores = Q.K with no normalisation of
# either factor and divides by a learned PER-REGISTER temperature, so the
# scaled score has TWO unbounded multiplicative channels:
#     |s_tilde| <= ||q|| ||k|| / tau      (both factors free)
# and the temperature gradient dL/dlog_tau = -Cov_a(s_tilde, u) is
# proportional to score magnitude. On the d384 baseline arm that produced
# a measured runaway: whichever register was coldest took ~100% of
# log_tau's entire gradient (register 14 at steps 70,522/71,194; later
# handing off to register 26 -- SS12.1).
#
# With qk_norm=True: Q and K are L2-normalised so Q.K is a cosine in
# [-1,1], and a CLAMPED per-register `logit_scale` REPLACES log_tau
# (sigma_k = min(exp(lambda_k), logit_scale_max)), giving the hard bound
#     |s_tilde| <= logit_scale_max
# for every register, token, batch and weight configuration.
#
# SS10.2/SS10.3 are the reason this REPLACES rather than multiplies: a
# bounded scale applied ON TOP of the unbounded 1/tau divisor bounds only
# the numerator of a ratio whose denominator is still free to fall, and
# leaves the identical Riccati equation (v_dot = eta*C*v^2) intact. The
# first implementation did exactly that and was no fix at all. Under
# qk_norm the model sets self.log_tau = None, so exactly one
# temperature-like knob survives -- the bounded one -- and
# `tau_create_init` below is deliberately ignored.
#
# NOT RETROFITTABLE, which is why this is a fresh arm rather than a
# setting on the continuation: it caps raw scores at 100 while the
# baseline run's coldest register reaches raw |Q.K| ~= 4,980, so enabling
# it on an existing checkpoint compresses the gate's scores ~50x in one
# step. That is not a large perturbation, it is a different function.
#
# What this does NOT do (SS10.3, stated so the pilot isn't misread): it
# does not damp the feedback loop or create an interior equilibrium --
# the same Riccati equation reappears in sigma. What it buys is that the
# loop's endpoint becomes a constant chosen in advance instead of an
# unbounded numerical accident, and that bounding ONE scalar becomes
# sufficient, which it is not while ||q||||k|| is free.
CREATION_QK_NORM             = True
CREATION_LOGIT_SCALE_INIT    = 1.0 / 0.07   # ~14.3, the CLIP-style default
CREATION_LOGIT_SCALE_MAX     = 100.0        # the hard ceiling on |s_tilde|

# -- Register repulsion (B4) -------------------------------------------
REGISTER_REPULSION       = True
REGISTER_REPULSION_COEFF = 0.05
REGISTER_REPULSION_KIND  = 'gram'

# -- Output head -------------------------------------------------------
USE_OUTPUT_BIAS = True
TIE_EMBEDDINGS  = False

# -- Optimizer ---------------------------------------------------------
OPTIMIZER = 'adamw'
GRAD_CENTRALIZATION = False

# -- LR schedule (WSD) -------------------------------------------------
LR_SCHEDULE     = 'wsd'
WSD_WARMUP_FRAC = 0.05
WSD_STABLE_FRAC = 0.60
WSD_LR_FLOOR    = None          # resolved after LR is set

# -- Batch / accumulation ----------------------------------------------
# The CfC arm carries a second anisotropic-well evaluation per layer
# (harmonic_terms at h, plus the force at the drifted h_mid), so its
# activation footprint is ~1.5x the Verlet arm's even with the well
# parameters shared between the two.  The auto-probe in Cell 5 therefore
# tends to land on a smaller per-device batch than the Verlet notebook
# does.  Leaving GRAD_ACCUM fixed would then shrink the *effective*
# batch too, which would confound a Verlet-vs-CfC comparison: the two
# runs would differ in gradient noise as well as in integrator.  So the
# probe compensates -- it keeps EFFECTIVE_BATCH pinned to the target and
# spends the difference on accumulation steps.
#
# 32 matches the Verlet aniso-Gaussian OWT run
# (colab_fock_aniso_gaussian_fockreg_openwebtext.ipynb on an 80GB card:
# batch 16 x accum 2), so PPL curves stay directly comparable.
TARGET_EFFECTIVE_BATCH = 32
GRAD_ACCUM      = 2       # fallback / lower bound; raised by the probe

# -- Fock coupling regularisation --------------------------------------
LAMBDA_FOCK_REG = 5e-3
FOCK_REG_EPS    = 1e-6

# == INTEGRATOR =========================================================
# 'verlet'          : damped velocity-Verlet, friction folded into the
#                     1/(1+dt*gamma) coefficient, V_theta force from
#                     autograd.  The historical baseline -- bit-identical
#                     to the runs this notebook is trying to improve on.
# 'analytic_vtheta' : same integrator, but -grad V_theta comes from its
#                     closed form, so V_theta leaves the second-order
#                     create_graph chain.  Same model, same gradients
#                     (asserted by test_cfc_baoab.py) -- only the way the
#                     force is obtained differs.
# 'baoab'           : palindromic ABOBA split with an exact exp(-gamma*dt)
#                     friction substep and a genuine velocity.
# 'baoab_cfc'       : as 'baoab', plus the closed-form harmonic propagator
#                     for the stiff diagonal part of V_theta.  Immune to
#                     the well-sharpening blow-up that the explicit step
#                     suffers from -- but the anisotropic OFF-diagonal
#                     coupling is still an explicit kick (an omega*dt<2 wall).
# 'baoab_cfc_lowrank': as 'baoab_cfc', but the PSD low-rank part
#                     L = sum_k g_k B_k B_k^T is ALSO integrated exactly, on
#                     its <= n_ctx*K*rank modes (impulse/RESPA fast flow), so
#                     the off-diagonal stiff channel no longer has a hard
#                     omega*dt<2 wall (only narrow damped resonances at
#                     omega*dt ~ k*pi).  Mitigation #1 of the CfC/BAOAB
#                     companion note; pair with PRECISION_LR_MAX (#2).
# INTEGRATOR = 'baoab_cfc'
INTEGRATOR = 'baoab_cfc'


# O-step thermostat temperature.  0.0 = deterministic friction only, which
# keeps this run directly comparable to the Verlet curves.  Raising it
# turns the O-step into a true FDT-locked Langevin thermostat.
LANGEVIN_T = 0.0

_INTEGRATOR_MODES = {
    #                     cfg.integrator        cfg.vtheta_analytic_force
    'verlet':            ('verlet',             False),
    'analytic_vtheta':   ('verlet',             True),
    'baoab':             ('baoab',              True),
    'baoab_cfc':         ('baoab_cfc',          True),
    'baoab_cfc_lowrank': ('baoab_cfc_lowrank',  True),
}
assert INTEGRATOR in _INTEGRATOR_MODES, (
    f'INTEGRATOR={INTEGRATOR!r} not in {sorted(_INTEGRATOR_MODES)}')
CFG_INTEGRATOR, CFG_VTHETA_ANALYTIC = _INTEGRATOR_MODES[INTEGRATOR]

# == Stiffness mitigations (CfC/BAOAB companion note, §29, §41-42) =======
# #2 -- smooth bound on the low-rank curvature sigma_max(B_k)^2 (Frobenius
#       cap).  None keeps B_k unbounded.  Tune against the SCAF Phase 7b/7c
#       Weyl audit ('Weyl frac(>2)' should drop); the Frobenius cap is
#       conservative by up to a factor `rank`, so this is not a literal
#       sigma_max^2 target.  Independent of the integrator.
#
# 2026-09-05 (companion note SS41/SS42): turned ON after
# replay_precision_cap_ablation confirmed budgets of 1.0 AND 4.0 both
# collapse all three captured spikes (step 47116: pre-clip 13,139.5 ->
# 3.98 at budget=1.0; step 48507: 203.1 -> 2.14; step 48917: 202.0 ->
# 2.79) -- including the reverse-channel-led event (48917), which
# neither budget touches directly, implying the two spike "mechanisms"
# (SS41 Findings 2-3) share this one root cause. bracket_precision_lr_max
# (Cell 6b-3) additionally showed the healthy (step 27,000) and
# spike-regime checkpoints have STATISTICALLY SIMILAR ambient
# sigma_max(B_k)^2 under a neutral batch (p50 ~280-310 across all four),
# so there is no tight, tail-only budget available here -- any cap tight
# enough to kill the exponent runaway also compresses everyday operation.
# Chose the more conservative of the two evidenced-safe budgets (1.0 over
# 4.0): cheap to loosen later (`bank._precision_lr_max` is a live,
# hot-swappable Python attribute, not part of state_dict), expensive to
# be wrong (another catastrophic reload) if too loose. Monitor val_ppl /
# dc_ratio / b_proj_sigma_max after resuming and revisit if 1.0 visibly
# hampers learning.
PRECISION_LR_MAX = 1.0
# #1 -- cap on the number of exactly-rotated low-rank modes when
#       INTEGRATOR='baoab_cfc_lowrank' (keeps the stiffest ones).  This is
#       now a genuine cost control: lowrank_modes uses a randomised truncated
#       SVD (torch.svd_lowrank) when this is set, costing O(d*P*q) instead of
#       the full O(d*P^2) per token per layer per step -- the difference
#       between a runnable arm and the batched full-SVD that stalled the run.
#       Only the few stiffest modes cross the omega*dt<2 wall; the rest are
#       demoted to the (stable) explicit kick.  None keeps ALL n_ctx*K*rank
#       (= 5*8*4 = 160 here) modes via the full SVD -- correct but very slow.
LOWRANK_MAX_MODES = 16

# -- Damping coefficient -------------------------------------------------
# gamma=0.100 chosen from the d=384, L=16 aniso-Gaussian+fock-reg gamma
# sweep (colab_fock_gamma_sweep_geodesic_aniso_gaussian_fockreg_d384.ipynb):
# best PPL (278.27) AND best geodesic R_bar (0.6708) coincide at gamma=0.100.
# gamma=0.150/0.250 are tied within ~5% (flat bowl); gamma=0.200 was an
# isolated instability outlier (PPL=2250) bracketed by good neighbours on
# both sides, not a genuine stability wall.
# Under BAOAB the friction is applied as exp(-gamma*dt) rather than
# 1/(1+gamma*dt); at gamma=0.10, dt=1 the two differ by ~0.5%, so the same
# gamma remains directly comparable across arms.
FIXED_GAMMA = 0.10

# -- Regularisation ----------------------------------------------------
LAMBDA_V       = 1e-2
BG_QUAD_EPS    = 0.0

# -- Training ----------------------------------------------------------
# 150,000 is the PRODUCTION length for this arm, set here (not 15,000) so
# the WSD warmup/stable/decay windows are the ones the full run would
# use: warmup 0->7,500, stable 7,500->97,500, decay 97,500->150,000.
# PROBE_MAX_STEPS below stops the pilot early WITHOUT touching this, so
# the pilot is literally the first 15,000 steps of the production arm --
# if it passes, clear PROBE_MAX_STEPS and the same run continues from its
# own checkpoint. Zero wasted compute; nothing is re-run.
TOTAL_STEPS   = 150_000
BLOCK_SIZE    = 512
VOCAB_SIZE    = 50257
SEED          = 0

# -- Depth side-by-side probe (2026-08-23) ------------------------------
# The g0.1 L=16 run hit a burst of large, uncaught grad-clip spikes
# (creation_gate/destruction_gate/register/reverse_ch/depth_code/V_theta,
# steps 6297-6676; see training_log.jsonl around those steps) that a real
# PPL hit (176.88->207.11 across the 6000->6500 eval). depth_code is a
# per-layer nn.Parameter (shape [L, n_ctx, d]); creation_gates/
# destruction_gates are per-layer nn.ModuleLists; reverse_ch is a SINGLE
# module reused (weight-tied) at every layer -- all three mean a smaller
# L shortens the compounding chain a spike has to propagate through both
# forward (activation state) and backward (Jacobian product depth).
# This probe pins d=384 (same as the live run) at a different L, with dt
# left untouched at 1.0 either way (see make_config below), so the ONLY
# thing that differs from the live g0.1/L=16 run is depth itself -- not
# conflated with the separate "fewer L, bigger dt for the same total
# integration time" question, which is a different experiment.
# None -> unchanged default behaviour (ARCH_TIERS ladder in Cell 5 still
# picks d=384, L=16, M=32 first). Set to an int (e.g. 8) to pin that L
# instead; the variant tag below then routes this run to its own Drive
# folder/checkpoints so it cannot collide with the live L=16 run.
L_PROBE_OVERRIDE = 8   # pinned: continuing the 2026-08-23 depth probe

# Stop the training loop cleanly (final eval + checkpoint) once this many
# steps have been taken, without touching TOTAL_STEPS -- the WSD warmup/
# stable/decay windows are fractions of TOTAL_STEPS, so changing
# TOTAL_STEPS itself would also compress/stretch the LR schedule and stop
# this from being an apples-to-apples comparison against the same
# (warmup=5000, stable=5000->65000) schedule phase the L=16 spikes were
# observed in. None = run to TOTAL_STEPS as normal (manual interrupt).
#
# 2026-08-24: the first 8,000-step slice of this probe (stopped by the
# PROBE_MAX_STEPS=8_000 below) ran clean through the entire step window
# (6,297-6,676) where the live L=16 run hit its grad-clip burst -- zero
# [spike] events, monotonically improving PPL (1476.67 -> 136.06). Per
# CfC_BAOAB_Integrator_and_Mitigations.md SS24.4's July-17 precedent
# (Tier-2 "reduce L" delayed but did not prevent the d=768 blowup, which
# hit at step ~=37,000 after 33,000 clean steps), 8,000 clean steps is not
# enough to call this resolved rather than merely delayed -- so this is
# now set to None to let the SAME run (it resumes from the Drive
# checkpoint at step 8,000 automatically; see Cell 2) continue past that
# point and find out when/whether turbulence appears. Re-set to an int to
# stop cleanly again for a snapshot.
# 2026-09-10: 15,000-step PILOT STOP for the joint-coupling + QK-norm arm.
# This is a FALSIFIER, not a certificate. The Mitigations note SS24.4's
# precedent is explicit -- the d=768 blowup hit at step ~37,000 after
# 33,000 clean steps -- and this notebook's own L=8 depth probe ran clean
# through 8,000 before that was judged "delayed, not resolved". 15,000
# steps can kill this arm; it cannot bless it.
#
# What 15,000 steps DOES answer, in descending order of value:
#   1. STABILITY (prediction J3). [spike] capture count and
#      [watchdog-hard] events vs the baseline arm's own first 15,000.
#      This is SS5's curvature-concentration risk, measured.
#   2. COST. sec/step and mem_peak. At K=8 these should be UNCHANGED vs
#      the additive arm (same matrix sizes, same FLOPs); if they are not,
#      the wiring is wrong. Known within minutes, not days.
#   3. bproj_sig slope. Joint coupling concentrates curvature, so
#      sigma_max(W_B) should climb FASTER here than additive's did.
#   4. Register_Temperature SS12 prediction 4: no single register should
#      dominate the gate's scale gradient. Watch sig_max@rN in the step
#      line (see the creation-gate monitor in Cell 6).
#   5. PPL at 8k/15k -- a sanity check ONLY, not a discriminator. Curves
#      are steep and noisy there, and the baseline arm's own first 15,000
#      steps are not a clean control (they ran with PRECISION_LR_MAX=None,
#      no NO_DECAY_1D and no clip_then_sum).
# Set to None to promote this pilot into the full 150,000-step arm.
PROBE_MAX_STEPS = 15_000

# -- Variant tag (for GDrive path and checkpoint naming) ----------------
_variant_parts = []
_variant_parts.append(f'xi{XI_OVERRIDE}')
_variant_parts.append(f'topk{TOP_K}')
_variant_parts.append(f'dt{V_PHI_D_TYPE}da{V_PHI_D_ANGLE}')
_variant_parts.append(f'mh{V_PHI_N_HEADS}')
_variant_parts.append(f'aniso_dcvt{V_THETA_N_HEADS}x{V_THETA_WELLS_PER_HEAD}')
# Both of these change the model's parameter shapes, so they MUST land in
# the tag: this arm gets its own Drive folder, its own checkpoints, its
# own training_log.jsonl and its own spikebatch ring, and can never
# collide with (or be resumed from) the additive baseline arm.
if V_THETA_COUPLING != 'additive':
    _variant_parts.append(f'vt{V_THETA_COUPLING}')
if CREATION_QK_NORM:
    _variant_parts.append('cgqk')
if L_PROBE_OVERRIDE is not None:
    _variant_parts.append(f'L{L_PROBE_OVERRIDE}probe')
_variant_parts.append('ob')
_variant_parts.append('untied')
_variant_parts.append(LR_SCHEDULE)
_variant_parts.append('e5c')
_variant_parts.append('plgate')
_variant_parts.append(f'rep{REGISTER_REPULSION_COEFF:g}')
_variant_parts.append(f'fockreg{LAMBDA_FOCK_REG:g}')
_variant_parts.append(f'g{FIXED_GAMMA:g}')
# The integrator is part of the tag, so every arm gets its own Drive
# folder, checkpoints and training_log.jsonl and can be resumed on its own.
_variant_parts.append(INTEGRATOR)
if LANGEVIN_T > 0:
    _variant_parts.append(f'T{LANGEVIN_T:g}')
_variant_tag = '_'.join(_variant_parts)

# Under 'joint' there is ONE bank, so the attractor count is K, not
# n_ctx*K -- each well is a joint function of all channels rather than
# one channel's own. Printing the additive formula here would overstate
# this arm's attractor budget by 5x and hide the real trade being made.
total_wells = (V_THETA_WELLS_PER_HEAD if V_THETA_COUPLING == 'joint'
               else V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD)
print(f'Anisotropic Gaussian V_theta on OpenWebText d=384')
if V_THETA_COUPLING == 'joint':
    print(f'  V_theta: 1 joint bank x {V_THETA_WELLS_PER_HEAD} wells = '
          f'{total_wells} total attractors '
          f'(each a joint function of all {V_THETA_N_HEADS} channels)')
else:
    print(f'  V_theta: {V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
          f'{total_wells} total attractors')
print(f'  Aniso rank r={ANISO_RANK}')
print(f'  V_theta coupling: {V_THETA_COUPLING}'
      + ('  (one joint bank over the concatenated context)'
         if V_THETA_COUPLING == 'joint' else '  (one bank per channel, summed)'))
print(f'  Creation gate QK-norm: {CREATION_QK_NORM}'
      + (f'  (logit_scale init={CREATION_LOGIT_SCALE_INIT:.2f} '
         f'max={CREATION_LOGIT_SCALE_MAX:g}; log_tau NOT registered)'
         if CREATION_QK_NORM else ''))
print(f'  Depth-conditioned: {V_THETA_DEPTH_CONDITION}')
print(f'  Fock coupling reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  Damping: fixed_gamma={FIXED_GAMMA}')
print(f'  Integrator: {INTEGRATOR}  '
      f'(cfg.integrator={CFG_INTEGRATOR}, '
      f'analytic_vtheta={CFG_VTHETA_ANALYTIC}, T={LANGEVIN_T:g})')
print(f'  Xi: {XI_CHANNELS}ch  horizons ~{[round(1/(1-a),1) for a in XI_ALPHA_INITS]} tok')
print(f'  V_phi={V_PHI_KIND} x {V_PHI_N_HEADS}h  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')
print(f'  steps={TOTAL_STEPS}  schedule={LR_SCHEDULE}  grad_accum={GRAD_ACCUM}')
print(f'  [variant] tag={_variant_tag}')

In [ ]:
# == Cell 1: Environment + Drive Mount =================================
import os, sys, gc, shutil, subprocess, json, time, math
from pathlib import Path
from dataclasses import asdict

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

REPO_URL    = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH = 'main'

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd):
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'exit {r.returncode}: {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    REPO_ROOT = Path('/content/semsimula-paper')
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}')
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: repo refresh failed ({e}); using existing checkout.')

    _gdrive_name = 'semsimula_fock_cfc_baoab_owt'
    if _variant_tag:
        _gdrive_name += f'_{_variant_tag}'
    GDRIVE_ROOT = Path(f'/content/drive/MyDrive/{_gdrive_name}')
    GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)

    DATA_DIR = GDRIVE_ROOT / 'data'
    DATA_DIR.mkdir(exist_ok=True)
    repo_data = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data.is_symlink():
        repo_data.unlink()
    elif repo_data.is_dir():
        shutil.rmtree(repo_data)
    repo_data.symlink_to(DATA_DIR)

    CKPT_DIR    = GDRIVE_ROOT / 'checkpoints'
    RESULTS_DIR = GDRIVE_ROOT / 'results'
    CKPT_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)

    _sh('pip install -q transformers huggingface_hub pyarrow matplotlib')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    DATA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    _local_phase = 'cfc_baoab_owt' + (f'_{_variant_tag}' if _variant_tag else '')
    CKPT_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase / 'ckpts'
    RESULTS_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup' / 'results' / _local_phase
    for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
        d.mkdir(parents=True, exist_ok=True)

CA_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
for sub in ['', 'parf', 'multixi', 'scaleup', 'sarf_mass_variant', 'energetic_minima']:
    d = str(CA_DIR / sub) if sub else str(CA_DIR)
    if d not in sys.path:
        sys.path.insert(0, d)

CKPT_PREFIX   = 'fock_cfc_owt' + (f'_{_variant_tag}' if _variant_tag else '')
CKPT_INTERVAL = 7_500
CKPT_STEPS    = list(range(CKPT_INTERVAL, TOTAL_STEPS + 1, CKPT_INTERVAL))

print(f'CKPT_DIR    = {CKPT_DIR}')
print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'Steps: {TOTAL_STEPS:,}  checkpoints at: {CKPT_STEPS}')

In [ ]:
# == Cell 1b: Resume-path override (same run, different INTEGRATOR) =====
#
# INTEGRATOR is deliberately part of _variant_tag (Cell 0's comment: "so
# every arm gets its own Drive folder ... and can be resumed on its own"),
# which is exactly right when INTEGRATOR is set from the start for a
# genuinely separate side-by-side arm. It is exactly WRONG when the goal
# is to keep training the SAME run and only change which integrator it
# uses from here on (e.g. baoab_cfc -> baoab_cfc_lowrank after a
# hard-watchdog burst): with no override, Cell 1 above already created a
# brand-new, empty GDRIVE_ROOT/CKPT_DIR keyed on the NEW tag, and Cell 2
# below would find no checkpoints there and start from scratch.
#
# Set this to the SOURCE run's variant tag (copy it from that run's own
# Cell 0 printout -- the `[variant] tag=...` line -- or read it off the
# existing Drive folder / checkpoint path) to redirect CKPT_DIR/
# CKPT_PREFIX/RESULTS_DIR/GDRIVE_ROOT there, while the model/training loop
# still use whatever INTEGRATOR is set to in Cell 0. None (default) is a
# no-op: normal per-arm-tag behaviour, unchanged.
#
# 2026-09-10: cleared for this arm. This cell was cloned from the
# additive-coupling baseline notebook, where it was pinned to that run's
# OWN tag (a no-op -- it just redirected to the same folder Cell 1 would
# have picked anyway). Left unedited here it would have pointed at the
# BASELINE's tag while Cell 0 above computes THIS arm's tag (with the
# vtjoint_cgqk suffix) -- i.e. it would have silently redirected this
# arm's CKPT_DIR/CKPT_PREFIX/GDRIVE_ROOT into the baseline's folder:
# load_state_dict(..., strict=False) at Cell 2 would then find a
# checkpoint whose V_theta has 5 additive banks (wrong shape for this
# arm's 1 joint bank), silently drop the mismatched keys with no error,
# and this arm's V_theta would train from random init while everything
# else loaded from the wrong architecture -- and every checkpoint this
# arm saves afterward would overwrite the baseline's own files, since
# CKPT_PREFIX would be redirected too.
# This arm has no prior checkpoint history to preserve under an old tag
# in the first place -- it is a fresh start -- so None (default, no
# override) is correct: Cell 0's own freshly-computed tag is exactly
# right and Cell 1 alone handles it correctly.
RESUME_VARIANT_TAG_OVERRIDE = None

if RESUME_VARIANT_TAG_OVERRIDE is not None:
    _old_tag = RESUME_VARIANT_TAG_OVERRIDE
    if IN_COLAB:
        GDRIVE_ROOT = Path(
            f'/content/drive/MyDrive/semsimula_fock_cfc_baoab_owt_{_old_tag}')
        GDRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        CKPT_DIR = GDRIVE_ROOT / 'checkpoints'
        RESULTS_DIR = GDRIVE_ROOT / 'results'
    else:
        _local_phase = f'cfc_baoab_owt_{_old_tag}'
        CKPT_DIR = (REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
                     / 'results' / _local_phase / 'ckpts')
        RESULTS_DIR = (REPO_ROOT / 'notebooks' / 'conservative_arch'
                        / 'scaleup' / 'results' / _local_phase)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    CKPT_PREFIX = f'fock_cfc_owt_{_old_tag}'
    print(f'[resume-override] redirected to source-run tag={_old_tag!r}')
    print(f'  CKPT_DIR    = {CKPT_DIR}')
    print(f'  CKPT_PREFIX = {CKPT_PREFIX}')
    print(f'  Cell 2 below will resume from checkpoints found there; '
          f'training continues under INTEGRATOR={INTEGRATOR!r} '
          f'(cfg.integrator={CFG_INTEGRATOR!r}). New checkpoints / '
          f'best.pt / training_log.jsonl entries land in THIS SAME '
          f'folder from now on, mixing both integrators'' history under '
          f'one filename stream -- copy the folder first if you want a '
          f'clean fork instead of an in-place continuation.')
    print(f'  (the fresh, now-unused GDRIVE_ROOT Cell 1 created for the '
          f'new tag is harmless and can be deleted later.)')


In [ ]:
# == Cell 1c: Auto-archive spikebatch/prereload snapshots before rotation ==
#
# 2026-09-10: the spikebatch ring (SPIKEBATCH_SNAPSHOT_MAX_KEEP=12) and the
# prereload rotation (PRERELOAD_SNAPSHOT_MAX_KEEP=5) both delete their OLDEST
# file the moment a new one is written, independent of whether the RUN has
# reached its final target step. A 24h Colab session captures far more than
# a ring's worth on its own -- e.g. 10 spikebatch captures across just 4,790
# steps in one observed window (mean gap ~530 steps, but bursty: as tight as
# 63-88 steps apart in a cluster) -- so waiting for the run to finish before
# archiving is not safe. This must run every session, near the top, before
# Cell 6 (training) has a chance to write a capture that evicts one you
# haven't copied out yet. Pure filesystem copy: no model, no GPU, no torch
# needed, and it is idempotent (skips anything already archived), so running
# it every session -- even ones where nothing new was captured -- costs
# nothing and needs no memory of when you last did this.
#
# Needs only CKPT_DIR / CKPT_PREFIX / GDRIVE_ROOT from Cell 1 above --
# deliberately placed before Cell 2 (GPU probe) so it runs even on a CPU
# runtime and even if Cell 2 or the model build fails later.
import shutil

_ARCHIVE_DIR = GDRIVE_ROOT / 'spikebatch_archive'
_ARCHIVE_DIR.mkdir(exist_ok=True)
_archived_now = 0
for _p in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_spikebatch.pt')):
    _dst = _ARCHIVE_DIR / _p.name
    if not _dst.exists():
        shutil.copy2(_p, _dst)
        _archived_now += 1

# Prereload snapshots (the pre-watchdog-reload weights) can't be replayed --
# no batch/RNG state -- but ARE usable by the weight-space probes
# (sigma_lr_report / bracket_precision_lr_max), so they're worth keeping too.
_PRERELOAD_ARCHIVE_DIR = GDRIVE_ROOT / 'prereload_archive'
_PRERELOAD_ARCHIVE_DIR.mkdir(exist_ok=True)
for _p in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_prereload.pt')):
    _dst = _PRERELOAD_ARCHIVE_DIR / _p.name
    if not _dst.exists():
        shutil.copy2(_p, _dst)
        _archived_now += 1

_n_sb_archive = len(list(_ARCHIVE_DIR.glob('*_spikebatch.pt')))
_n_pr_archive = len(list(_PRERELOAD_ARCHIVE_DIR.glob('*_prereload.pt')))
print(f'[archive] copied {_archived_now} new file(s) this session.')
print(f'[archive] spikebatch_archive now holds {_n_sb_archive} bundle(s) '
      f'-> {_ARCHIVE_DIR}')
print(f'[archive] prereload_archive now holds {_n_pr_archive} snapshot(s) '
      f'-> {_PRERELOAD_ARCHIVE_DIR}')


In [ ]:
# == Cell 2: GPU + Checkpoint resolution ===============================
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print('WARNING: No GPU detected. This notebook requires CUDA.')

resume_step = 0
resume_ckpt = None

for s in sorted(CKPT_STEPS, reverse=True):
    cand = CKPT_DIR / f'{CKPT_PREFIX}_step{s}.pt'
    if cand.exists():
        resume_ckpt = cand
        resume_step = s
        break

_best_candidates = []
_canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
if _canonical.exists():
    _best_candidates.append(_canonical)
for _f in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt')):
    _best_candidates.append(_f)

_best_path = None
_best_step_found = resume_step
for _cand in _best_candidates:
    try:
        _bd = torch.load(_cand, map_location='cpu', weights_only=False)
        _s = _bd.get('step', 0)
        _p = _bd.get('val_ppl', float('inf'))
        del _bd
        if _s > _best_step_found:
            _best_step_found = _s
            _best_path = _cand
            _best_ppl = _p
            print(f'  Found best candidate: {_cand.name} (step {_s:,}, PPL {_p:.2f})')
    except Exception as e:
        print(f'[warn] could not inspect {_cand.name}: {e}')

if _best_path is not None and _best_step_found > resume_step:
    print(f'Best checkpoint (step {_best_step_found:,}, PPL {_best_ppl:.2f}) is more recent '
          f'than latest periodic checkpoint (step {resume_step:,}) -- resuming from best.')
    resume_ckpt = _best_path
    resume_step = _best_step_found

# Manual checkpoints (save_manual_checkpoint, taken right before a planned
# interrupt) and pre-hard-reload forensic snapshots (_reload_best's
# `_prereload` tag, saved automatically the instant a watchdog hard-trigger
# fires -- these hold the LAST CLEAN weights immediately before the
# offending step, not the corrupted post-update ones) are both full
# (model + optimizer) checkpoints, same as `_probe_stop`. None of the three
# are covered by the periodic/`_best` search above. Left unhandled, a
# session that dies between periodic checkpoints with no new best PPL
# silently falls back to a much older best/periodic checkpoint -- exactly
# the gap that once cost a resume ~20,000 steps it didn't need to lose.
# The step number is parsed straight out of the filename (all three tags
# encode it as `_step<N>_<tag>.pt`) rather than torch.load-ing every
# candidate on disk just to check.
_tagged_re = re.compile(rf'^{re.escape(CKPT_PREFIX)}_step(\d+)_(?:manual|prereload|probe_stop)\.pt$')
_tagged_candidates = []
for _f in CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_*.pt'):
    _m = _tagged_re.match(_f.name)
    if _m:
        _tagged_candidates.append((int(_m.group(1)), _f))
_tagged_candidates.sort()
for _tag_step, _tag_path in _tagged_candidates:
    print(f'  Found manual/prereload/probe_stop candidate: {_tag_path.name} (step {_tag_step:,})')
if _tagged_candidates:
    _best_tag_step, _best_tag_path = _tagged_candidates[-1]
    if _best_tag_step > resume_step:
        print(f'{_best_tag_path.name} (step {_best_tag_step:,}) is more recent than the current '
              f'pick (step {resume_step:,}) -- resuming from it. `_prereload`/`_manual` snapshots '
              f'have no evaluated val_ppl on file; run evaluate() once right after loading if you '
              f'want a fresh reading before committing more training time.')
        resume_ckpt = _best_tag_path
        resume_step = _best_tag_step

# if you want a clean before/after comparison starting from the known-good
# state, force it explicitly rather than trusting the automatic pick — e.g.
# right after Cell 2 runs
#resume_ckpt = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
#resume_step = 27_000  # or read it back from the checkpoint's own 'step' field


if resume_ckpt is not None:
    print(f'Resuming from: {resume_ckpt.name}  (step {resume_step:,})')
    print(f'Remaining: {TOTAL_STEPS - resume_step:,} steps')
else:
    print('No checkpoint found -- training from scratch.')
    print(f'Total: {TOTAL_STEPS:,} steps  Checkpoints every {CKPT_INTERVAL:,}')

In [ ]:
# == Cell 3: Data loading (OpenWebText) ================================
from data_module import get_batch

MAX_TRAIN_TOKENS = 2_000_000_000
VAL_TOKENS       = 2_000_000
CHUNK_SIZE       = 50_000

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('gpt2')

train_cache = DATA_DIR / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
val_cache   = DATA_DIR / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'

for alt_name in [
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05',
    'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd',
    'semsimula_fock_structured_vtheta_owt_phase4',
    'semsimula_fock_gaussian_sarf_openwebtext_phase5',
    'semsimula_splm_openwebtext_phase4',
    'semsimula_splm_openwebtext_scaleup',
    'semsimula_parf_multixi_openwebtext_scaleup',
    'semsimula_fock_multixi_openwebtext_scaleup',
    'semsimula_splm_openwebtext',
    'semsimula_fock_multihead_openwebtext',
    'semsimula_fock_multicontext_vtheta_owt',
]:
    if train_cache.exists():
        break
    if IN_COLAB:
        alt = Path(f'/content/drive/MyDrive/{alt_name}/data')
    else:
        alt = Path.home() / alt_name / 'data'
    alt_train = alt / f'openwebtext_train_{MAX_TRAIN_TOKENS // 1_000_000}M.npy'
    alt_val   = alt / f'openwebtext_val_{VAL_TOKENS // 1_000_000}M.npy'
    if alt_train.exists():
        import shutil
        print(f'Reusing data cache from {alt}')
        shutil.copy2(str(alt_train), str(train_cache))
        shutil.copy2(str(alt_val), str(val_cache))
        break

if train_cache.exists() and val_cache.exists():
    print('Loading cached OpenWebText tokens ...')
    train_ids = np.load(str(train_cache))
    val_ids   = np.load(str(val_cache))
    print(f'  train: {len(train_ids):,} tokens')
    print(f'  val:   {len(val_ids):,} tokens')
else:
    from datasets import load_dataset
    print(f'Streaming OpenWebText (target: {MAX_TRAIN_TOKENS:,} train + {VAL_TOKENS:,} val tokens) ...')
    ds = load_dataset('Skylion007/openwebtext', split='train', streaming=True, trust_remote_code=True)
    all_ids = []
    total = 0
    target = MAX_TRAIN_TOKENS + VAL_TOKENS
    chunk_texts = []
    n_docs = 0
    t0 = time.time()
    for example in ds:
        chunk_texts.append(example['text'])
        n_docs += 1
        if len(chunk_texts) >= CHUNK_SIZE:
            joined = '\n\n'.join(chunk_texts)
            chunk_ids = tok.encode(joined)
            all_ids.extend(chunk_ids)
            total = len(all_ids)
            elapsed = time.time() - t0
            print(f'  {n_docs:,} docs  {total:,} tokens  ({elapsed:.0f}s)', flush=True)
            chunk_texts = []
            del joined, chunk_ids
            if total >= target:
                break
    if chunk_texts:
        joined = '\n\n'.join(chunk_texts)
        all_ids.extend(tok.encode(joined))
        del joined, chunk_texts
    all_ids = np.array(all_ids, dtype=np.uint16)
    total = len(all_ids)
    print(f'Total streamed: {total:,} tokens from {n_docs:,} documents ({time.time() - t0:.0f}s)')
    val_ids   = all_ids[-VAL_TOKENS:]
    train_ids = all_ids[:-VAL_TOKENS]
    if len(train_ids) > MAX_TRAIN_TOKENS:
        train_ids = train_ids[:MAX_TRAIN_TOKENS]
    del all_ids
    np.save(str(train_cache), train_ids)
    np.save(str(val_cache), val_ids)
    print(f'  Cached: train={len(train_ids):,} -> {train_cache}')
    print(f'  Cached: val={len(val_ids):,}   -> {val_cache}')

print(f'train: {len(train_ids):,}   val: {len(val_ids):,}')

In [ ]:
# == Cell 4: V_theta + integrator: import from the repo, then verify ====
#
# The template notebook inlined AnisotropicMixtureGaussianVTheta.  This one
# imports it instead, because the CfC propagator needs `harmonic_terms`,
# which lives with the class in parf/model_aniso_gaussian_vtheta.py.  An
# inlined copy would silently shadow it and fall back to a stale definition.

from model_aniso_gaussian_vtheta import (
    AnisotropicMixtureGaussianVTheta,
    AnisotropicMultiContextGaussianVTheta,
    AnisotropicDepthConditionedGaussianVTheta,
    install_aniso_depth_routing,
)
import model_parf_multixi as _mpm

# -- Stale-checkout guards (fail here, not 90 minutes into training) -----
assert hasattr(AnisotropicDepthConditionedGaussianVTheta, 'harmonic_terms'), (
    'STALE CHECKOUT: the anisotropic V_theta has no harmonic_terms(), which '
    'the CfC propagator needs. Restart the Colab runtime (Runtime > Restart '
    'runtime) so the freshly fetched module is re-imported.')
assert hasattr(_mpm.MultiXiPARFLM, '_layer_step_langevin'), (
    'STALE CHECKOUT: MultiXiPARFLM has no _layer_step_langevin(). Restart '
    'the Colab runtime and re-run from the top.')
assert 'integrator' in {f.name for f in
                        __import__('dataclasses').fields(_mpm.MultiXiPARFConfig)}, (
    'STALE CHECKOUT: MultiXiPARFConfig has no `integrator` field.')

# -- Run the integrator test suite (CPU, ~10 s) --------------------------
# Cheap insurance: proves on THIS checkout that the analytic V_theta force
# reproduces autograd's gradients exactly, that the CfC split preserves the
# force field to O(dt^3), and that the propagator survives stiffness that
# overflows the explicit step.
import subprocess, sys as _sys
_test = CA_DIR / 'parf' / 'test_cfc_baoab.py'
if _test.exists():
    _res = subprocess.run([_sys.executable, str(_test)],
                          capture_output=True, text=True, cwd=str(CA_DIR / 'parf'))
    print(_res.stdout[-2500:])
    if _res.returncode != 0:
        print(_res.stderr[-2500:])
        raise RuntimeError('CfC/BAOAB integrator tests FAILED -- do not train '
                           'on this checkout.')
else:
    print(f'WARNING: {_test} not found; skipping integrator self-tests.')


In [ ]:
# == Cell 5: Model config + build + aniso V_theta swap =================
import gc
import math
from model_fock_parf_multixi import FockMultiXiPARFLM, FockMultiXiPARFConfig
import model_fock_parf_v2
import model_parf_multixi
import model_parf
import model_parf_sparse

# Guard against a stale in-memory module for the classes THIS RUN
# actually instantiates -- FockMultiXiPARFLM (model_fock_parf_multixi.py)
# on top of MultiXiPARFLM (model_parf_multixi.py). These have their OWN
# copies of the per-layer checkpoint gate and the force-computation
# autograd.grad call; `model_parf.PARFLM` is a *different*, unused base
# class, so checking it (as an earlier version of this cell did) gives a
# false pass. Two independent bugs can each cause the eval-time OOM in
# forward_gathered (all L layers' buffers alive at once during
# evaluate()'s torch.enable_grad() forward, with no outer .backward()
# ever around to free them):
#  1. FockMultiXiPARFLM._stack_forward gating the per-layer checkpoint on
#     `self.training` instead of `torch.is_grad_enabled()`.
#  2. MultiXiPARFLM._layer_step hard-coding `retain_graph=True` on the
#     force autograd.grad call instead of `retain_graph=self.training`
#     (retain_graph is only needed when create_graph=True; in eval it
#     just keeps every layer's buffers alive with no backward() call to
#     ever consume/free them).
# Fail fast here instead of discovering it ~1.5h later at the first eval
# call. If this cell is re-run in a kernel that already imported these
# modules before a later `git fetch/reset`, it will (correctly) still
# fail -- sys.modules caching means only a runtime restart clears it.
import inspect as _inspect
_stack_fwd_src = _inspect.getsource(FockMultiXiPARFLM._stack_forward)
_layer_step_src = _inspect.getsource(model_parf_multixi.MultiXiPARFLM._layer_step)
assert 'torch.is_grad_enabled()' in _stack_fwd_src, (
    'STALE MODULE IN THIS KERNEL: FockMultiXiPARFLM._stack_forward still '
    'gates per-layer checkpointing on `self.training` instead of '
    '`torch.is_grad_enabled()`. Restart the Colab runtime (Runtime > '
    'Restart runtime), re-run the setup cell so it fetches the latest '
    'main, then re-run from the top -- a plain re-run of this cell '
    'cannot fix an already-imported module.'
)
assert 'retain_graph=self.training' in _layer_step_src, (
    'STALE MODULE IN THIS KERNEL: MultiXiPARFLM._layer_step still '
    'hard-codes `retain_graph=True` on the force autograd.grad call '
    '(should be `retain_graph=self.training`). Restart the Colab '
    'runtime and re-run from the top.'
)
print('Eval-time-OOM fix verified present in this kernel '
      '(FockMultiXiPARFLM checkpoint gate + MultiXiPARFLM retain_graph).')

# The BAOAB/CfC integrators return the outgoing velocity through
# _layer_step_ex; a stale model_fock_parf_multixi would still call
# _layer_step and silently drop the O-step, training a Verlet model under a
# BAOAB tag.
_fock_step_src = _inspect.getsource(FockMultiXiPARFLM._fock_layer_step)
assert '_layer_step_ex' in _fock_step_src, (
    'STALE MODULE IN THIS KERNEL: FockMultiXiPARFLM._fock_layer_step still '
    'calls _layer_step instead of _layer_step_ex, so the BAOAB/CfC velocity '
    'would be discarded every layer. Restart the Colab runtime and re-run '
    'from the top.')
print('Integrator plumbing verified (_fock_layer_step -> _layer_step_ex).')

# The V_theta component-sharing fix (context_components) is what keeps
# the CfC arm from building the anisotropic well bank twice per layer.
# A kernel that imported model_aniso_gaussian_vtheta before that commit
# would still OOM at every reasonable batch.
assert hasattr(AnisotropicDepthConditionedGaussianVTheta, 'context_components'), (
    'STALE MODULE IN THIS KERNEL: AnisotropicDepthConditionedGaussianVTheta '
    'has no context_components (the CfC memory fix). Restart the Colab '
    'runtime and re-run from the top.')


LOGFREQ_PATH = CA_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_openwebtext.npy'
DRIVE_LOGFREQ = RESULTS_DIR / 'logfreq_surprisal_openwebtext.npy'

if LOGFREQ_PATH.exists():
    LOGFREQ_FILE = LOGFREQ_PATH
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_FILE = DRIVE_LOGFREQ
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_FILE = DRIVE_LOGFREQ
    LOGFREQ_FILE.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_FILE, surprisal)

print(f'Logfreq: {LOGFREQ_FILE}')

if L_PROBE_OVERRIDE is not None:
    # Single, pinned tier: d=384 and M=32 stay identical to the live
    # g0.1/L=16 run so L is the only thing that differs. L=16 already
    # fits at d=384/M=32 on an 80GB card, so a smaller L needs strictly
    # less memory -- no OOM fallback ladder needed here.
    ARCH_TIERS = [(384, L_PROBE_OVERRIDE, 32)]
else:
    ARCH_TIERS = [
        (384, 16, 32),
        (384, 12, 16),
        (256, 16, 16),
        (256,  8, 16),
    ]


def make_config(d, L, n_registers):
    return FockMultiXiPARFConfig(
        vocab_size=VOCAB_SIZE, d=d, max_len=1024,
        L=L, v_hidden=1024, v_depth=3, dt=1.0,
        mass_mode='logfreq',
        logfreq_path=str(LOGFREQ_FILE),
        logfreq_init_alpha=0.1,
        init_gamma=1.0,
        fixed_gamma=FIXED_GAMMA,
        integrator=CFG_INTEGRATOR,
        vtheta_analytic_force=CFG_VTHETA_ANALYTIC,
        lowrank_max_modes=LOWRANK_MAX_MODES,
        langevin_T=LANGEVIN_T,
        causal_force=True,
        ln_after_step=True,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=XI_ALPHA_INITS,
        xi_learnable=True,
        xi_alpha_init_mode='explicit',
        v_phi_kind=V_PHI_KIND,
        v_phi_d_type=V_PHI_D_TYPE,
        v_phi_d_angle=V_PHI_D_ANGLE,
        v_phi_eps=0.1,
        v_phi_phi_hidden=128,
        v_phi_theta_hidden=128,
        v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
        top_k=TOP_K,
        v_phi_n_heads=V_PHI_N_HEADS,
        use_output_bias=USE_OUTPUT_BIAS,
        tie_embeddings=TIE_EMBEDDINGS,
        score_head_hidden=32,
        gumbel_tau_init=1.0,
        gumbel_tau_min=0.3,
        gumbel_noise=True,
        use_gathered_v_phi=True,
        use_layer_checkpoint=True,
        ln_before_distance=True,
        per_layer_v_phi_scale=True,
        fock_version='v2',
        n_registers=n_registers,
        register_salience_decay=0.5,
        register_salience_threshold=0.005,
        creation_gate_hidden=64,
        stack_discipline=True,
        d_k=64,
        # Ignored when creation_qk_norm=True (the model sets log_tau=None
        # and the per-register granularity moves to logit_scale); kept so
        # this notebook still works with CREATION_QK_NORM=False.
        tau_create_init=8.0,
        creation_qk_norm=CREATION_QK_NORM,
        creation_logit_scale_init=CREATION_LOGIT_SCALE_INIT,
        creation_logit_scale_max=CREATION_LOGIT_SCALE_MAX,
        reverse_channel=REVERSE_CHANNEL,
        reverse_channel_stable=REVERSE_CHANNEL_STABLE,
        reverse_channel_pre_ln=REVERSE_CHANNEL_PRE_LN,
        reverse_channel_soft_norm=REVERSE_CHANNEL_SOFT_NORM,
        reverse_channel_warmup_steps=REVERSE_CHANNEL_WARMUP_STEPS,
        reverse_channel_per_layer=REVERSE_CHANNEL_PER_LAYER,
        per_register_tau=True,
        per_register_keys=True,
        ortho_register_init=True,
        register_repulsion=REGISTER_REPULSION,
        register_repulsion_coeff=REGISTER_REPULSION_COEFF,
        register_repulsion_kind=REGISTER_REPULSION_KIND,
        prefix_causal_registers=True,
    )


model = None
model_cfg = None
for d, L, M in ARCH_TIERS:
    try:
        cfg = make_config(d, L, M)
        mdl = FockMultiXiPARFLM(cfg).to(DEVICE)
        n_v_theta_mlp = sum(p.numel() for p in mdl.V_theta.parameters())

        _init_log_prec = -math.log(d)
        _prec_max = 2.0 / d
        mdl.V_theta = AnisotropicDepthConditionedGaussianVTheta(
            d=d,
            K=V_THETA_WELLS_PER_HEAD,
            n_ctx=V_THETA_N_HEADS,
            n_layers=cfg.L,
            rank=ANISO_RANK,
            w_scale=W_SCALE,
            init_log_precision=_init_log_prec,
            precision_max=_prec_max,
            precision_lr_max=PRECISION_LR_MAX,
            code_init_std=V_THETA_DEPTH_CODE_INIT_STD,
            coupling=V_THETA_COUPLING,
        ).to(DEVICE)
        install_aniso_depth_routing(mdl)

        n = mdl.num_params()
        n_v_theta = sum(p.numel() for p in mdl.V_theta.parameters())
        print(f'Trying d={d} L={L} M={M} -> {n:,} params '
              f'(V_theta {n_v_theta_mlp:,} MLP -> {n_v_theta:,} Aniso-Gaussian)')

        if DEVICE == 'cuda':
            def _arch_probe(_m):
                rng = np.random.default_rng(42)
                xb, yb = get_batch(train_ids, 2, BLOCK_SIZE, rng)
                x = torch.from_numpy(xb).to(DEVICE)
                y = torch.from_numpy(yb).to(DEVICE)
                _, loss = _m(x, y)
                loss.backward()
                _m.zero_grad(set_to_none=True)
                del x, y, xb, yb, loss
            _arch_probe(mdl)
            gc.collect()
            torch.cuda.empty_cache()
            print(f'OOM probe passed (batch=2)')
        model = mdl
        model_cfg = cfg
        break
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            print(f'  OOM at d={d} L={L} M={M} -- trying next tier ...')
            del mdl
            gc.collect()
            if DEVICE == 'cuda':
                torch.cuda.empty_cache()
            continue
        raise

if model is None:
    raise RuntimeError('All architecture tiers OOMed.')

if USE_OUTPUT_BIAS:
    _ob_counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE)
    model.init_output_bias_from_logfreq(_ob_counts)
    print(f'Output bias <- log-unigram-freq  '
          f'(b range [{model.out_bias.min().item():.2f}, '
          f'{model.out_bias.max().item():.2f}])')

# -- Auto batch size ----------------------------------------------------
# Two measured facts this probe has to respect:
#
# 1. CUDA OOM is not recoverable in-process (60+ GB leftover after
#    traceback-clear + empty_cache).  Never start a forward that we
#    already have reason to believe will OOM.
# 2. `total_memory / 1e9` is ~85 on an 80 GiB card (decimal GB vs GiB).
#    A 92% budget of 85 GB is 78 GB -- essentially the whole device --
#    which is how batch=4 passed the probe and then died on step 1
#    at 78.6 GiB allocated.  Budget is 80% of total_memory, in bytes.
#
# Procedure: measure a train-shaped step at bs=2 (the architecture
# probe already survived this, so it will not OOM).  Larger sizes are
# attempted only when a linear-in-batch estimate of their peak plus
# the AdamW reserve sits under the 80% budget.  On an 80 GiB card the
# CfC arm's bs=2 peak is already ~half the device, so bs=4 is skipped
# and we train at 2 x 16 = 32 -- same effective batch as Verlet.
import gc

_n_params = sum(p.numel() for p in model.parameters())
_reserve_bytes = 2.5 * _n_params * 4          # AdamW exp_avg+exp_avg_sq + slack
_reserve = _reserve_bytes / 1e9


def _clear_exc():
    for _a in ('last_traceback', 'last_value', 'last_type', 'last_exc'):
        if hasattr(sys, _a):
            setattr(sys, _a, None)
    try:
        ip = get_ipython()
        if ip is not None:
            for _a in ('_last_traceback', 'last_execution_result'):
                if hasattr(ip, _a):
                    setattr(ip, _a, None)
    except NameError:
        pass


def _release():
    _clear_exc()
    model.zero_grad(set_to_none=True)
    gc.collect()
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def _rebuild_model():
    """Drop the live module and reconstruct it.  Weights are still
    random at this point (the training cell loads any checkpoint), so
    this does not discard trained state."""
    global model
    d, L, M = model_cfg.d, model_cfg.L, model_cfg.n_registers
    _clear_exc()
    try:
        model.zero_grad(set_to_none=True)
    except Exception:
        pass
    del model
    gc.collect()
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    cfg = make_config(d, L, M)
    mdl = FockMultiXiPARFLM(cfg).to(DEVICE)
    _init_log_prec = -math.log(d)
    _prec_max = 2.0 / d
    mdl.V_theta = AnisotropicDepthConditionedGaussianVTheta(
        d=d, K=V_THETA_WELLS_PER_HEAD, n_ctx=V_THETA_N_HEADS,
        n_layers=cfg.L, rank=ANISO_RANK, w_scale=W_SCALE,
        init_log_precision=_init_log_prec, precision_max=_prec_max,
        precision_lr_max=PRECISION_LR_MAX,
        code_init_std=V_THETA_DEPTH_CODE_INIT_STD,
        coupling=V_THETA_COUPLING,
    ).to(DEVICE)
    install_aniso_depth_routing(mdl)
    if USE_OUTPUT_BIAS:
        mdl.init_output_bias_from_logfreq(_ob_counts)
    model = mdl
    _held = torch.cuda.memory_allocated() / 1e9 if DEVICE == 'cuda' else 0.0
    print(f'  rebuilt model  (allocated {_held:.1f} GB)')
    return _held


def _probe_batch(bs):
    """One train-shaped step: stack + V_theta regularizer + backward.

    The training cell's forward_with_vreg is this plus the (tiny) Fock
    coupling term.  The extra V_theta call is the piece the previous
    probe omitted and that shows up as a step-1 OOM after a 'passing'
    probe.
    """
    rng = np.random.default_rng(42)
    xb, yb = get_batch(train_ids, bs, BLOCK_SIZE, rng)
    x = torch.from_numpy(xb).to(DEVICE)
    y = torch.from_numpy(yb).to(DEVICE)
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size), y.reshape(-1),
    )
    if LAMBDA_V > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        loss = loss + LAMBDA_V * (V_vals.float() ** 2).mean()
    loss.backward()
    peak = torch.cuda.max_memory_allocated() / 1e9
    del x, y, xb, yb, loss, h0, h_L, logits
    return peak


BATCH_SIZE = 2
if DEVICE == 'cuda':
    _vram_bytes = torch.cuda.get_device_properties(0).total_memory
    _vram_gb = _vram_bytes / 1e9
    _budget_gb = 0.80 * _vram_gb
    _release()
    _base_gb = torch.cuda.memory_allocated() / 1e9
    print(f'Batch probe on {_vram_gb:.1f} GB device '
          f'(budget {_budget_gb:.1f} GB = 80%): '
          f'weights {_base_gb:.1f} GB, AdamW reserve {_reserve:.1f} GB')

    # Measure the known-good size.  Do not skip this: the architecture
    # probe's batch=2 was a bare model() call, not the training step.
    _peak2 = _probe_batch(2)
    _release()
    print(f'  bs= 2: peak {_peak2:.1f} GB')
    if _peak2 + _reserve > _budget_gb:
        print(f'    even bs=2 is over budget; training at 2 anyway '
              f'(accum will be {TARGET_EFFECTIVE_BATCH // 2})')
        BATCH_SIZE = 2
    else:
        BATCH_SIZE = 2
        _act2 = max(_peak2 - _base_gb, 0.0)
        for bs in (4, 8, 16):
            _est = _base_gb + _act2 * (bs / 2.0)
            if _est + _reserve > _budget_gb:
                print(f'  bs={bs:>2}: estimated {_est:.1f} GB + reserve '
                      f'exceeds {_budget_gb:.1f} GB -- skipping '
                      f'(not attempting; an OOM here is unrecoverable)')
                break
            _peak = None
            try:
                _peak = _probe_batch(bs)
            except RuntimeError as _err:
                _is_oom = 'out of memory' in str(_err).lower()
                _err.__traceback__ = None
                _err.__context__ = None
                _err.__cause__ = None
                del _err
                if not _is_oom:
                    raise
                print(f'  bs={bs:>2}: OOM despite estimate {_est:.1f} GB '
                      f'-- keeping bs={BATCH_SIZE}')
                _rebuild_model()
                break
            _release()
            if _peak + _reserve > _budget_gb:
                print(f'  bs={bs:>2}: peak {_peak:.1f} GB + reserve '
                      f'exceeds budget -- keeping bs={BATCH_SIZE}')
                break
            BATCH_SIZE = bs
            print(f'  bs={bs:>2}: peak {_peak:.1f} GB  OK')
            _act2 = max(_peak - _base_gb, 0.0) / (bs / 2.0)

GRAD_ACCUM = max(1, TARGET_EFFECTIVE_BATCH // BATCH_SIZE)
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
print(f'Auto batch: {BATCH_SIZE} x accum={GRAD_ACCUM} '
      f'(eff={EFFECTIVE_BATCH}, target={TARGET_EFFECTIVE_BATCH})')
if EFFECTIVE_BATCH != TARGET_EFFECTIVE_BATCH:
    print(f'  NOTE: effective batch {EFFECTIVE_BATCH} != target '
          f'{TARGET_EFFECTIVE_BATCH}; PPL is not directly comparable to '
          f'the Verlet run at eff=32.')
n_params = model.num_params()
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())

d = model_cfg.d
print(f'\nModel: FockMultiXiPARFLM v2.1 + Anisotropic Gaussian V_theta')
print(f'  params: {n_params:,}  (V_theta: {n_v_theta:,})')
print(f'  d={d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  V_theta: aniso-gaussian  rank={ANISO_RANK}  '
      f'{V_THETA_N_HEADS} heads x {V_THETA_WELLS_PER_HEAD} wells = '
      f'{V_THETA_N_HEADS * V_THETA_WELLS_PER_HEAD} attractors')
print(f'  fock-reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  integrator: {INTEGRATOR}  gamma={FIXED_GAMMA}  T={LANGEVIN_T:g}')
print(f'  V_phi={V_PHI_KIND} x {V_PHI_N_HEADS} head(s)  top_k={TOP_K}  '
      f'd_type={V_PHI_D_TYPE}  d_angle={V_PHI_D_ANGLE}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')

In [ ]:
# == Cell 6d: Gradient-spike replay -- Phase 2 op/layer forensics ==========
# Root-cause diagnostic Phase 2 (CfC/BAOAB companion note SS33.3): given a
# Phase 1 '_spikebatch.pt' bundle (pre-optim.step() weights + the exact
# microbatch(es) + RNG state, captured at a GRAD_NORM_HARD_TRIGGER event),
# deterministically replay that one step's forward+backward in isolation,
# instrumented with:
#   (a) per-parameter grad norms -- finer than per-group, e.g.
#       creation_gate_qkv.log_tau vs. W_Q, or a single reverse_channel_scale.
#   (b) per-layer grad attribution -- a hook on the h tensor at each of the
#       L layer boundaries (gradient flows L -> 0; a localised spike is a
#       strong signal, and dovetails with SS27's boundary-layer finding).
#   (c) forward-activation extremes at the numerically risky ops the
#       companion note names: creation-gate softmax temperature tau,
#       cumulative-softmax denominator Z (via its alpha_max proxy),
#       reverse-channel logit_scale and the Q_force it injects, the
#       destruction-gate output, and the V_theta quadratic-form exponent
#       (recomputed from the bank's own public _components()) -- now
#       layer-resolved for creation_gate_qkv/reverse_ch (2026-08-30,
#       companion note SS38 follow-up): both are single shared modules
#       called once per layer, and _record() used to overwrite the same
#       dict key on every call, so activation_extremes silently only ever
#       reflected the LAST layer. V_theta banks turned out NOT to have
#       this bug at all once inspected against real replay output: they
#       are called ONCE per forward pass on the full depth-stacked xi
#       tensor, not per _fock_layer_step, so they're reported at the
#       fixed sentinel layer=-1 (not layer-attributable), same convention
#       as reverse_ch.logit_scale.
#   (d) the official per-layer register/gate health buffer
#       (model.set_fock_capture(True) -> model._fock_capture), giving
#       active_frac / salience / reg_cos_sim / destroy_mean /
#       create_alpha_max / create_entropy / rev_entropy / rev_alpha_max /
#       rev_scale / qforce_ratio FOR EVERY LAYER for free.
#   (e2) per-bank V_theta exponent OCCUPANCY HISTOGRAM (SS39) -- how many
#       token slots sit in the band where exp(exponent) is still alive
#       (> -10) versus underflowed to zero, since min/max/mean provably
#       could not separate the two modes but density is what the
#       localized-mode conjecture is actually about.
#   (f) attribute_spike_rows(step_tag): re-runs the captured batch one row
#       at a time to test whether a small minority of sequences owns the
#       depth_code gradient direction (SS39) -- the question that matters
#       once you notice depth_code is clipped to the same 0.25 on every
#       step regardless of how big its pre-clip norm was.
#   (e) inspect_spike_tokens(step_tag): decodes the exact offending
#       microbatch's GPT-2 tokens (via Cell 3's `tok`) and ranks rows by
#       longest same-token repeat run, to check whether a degenerate
#       OpenWebText passage correlates with the localized-blowup mode
#       (SS38.2) -- pure CPU bookkeeping, no model/GPU/RNG state touched.
#
# Non-pollution invariant: this cell saves the live model's weights,
# .grad tensors, and RNG state before disturbing any of them, and restores
# all three when done -- resuming the training loop afterward is safe
# (mirrors the SCAF GradientSpikeProbe design's save/zero/restore
# contract, docs/Gradient_Spike_Probe_Requirements_and_Design.md SS7 in
# semsimula-scaf).
#
# CAVEAT: (a) and (b) are exact -- they only touch documented, already-used
# training-loop machinery. (c)'s hooks are best-effort: they were derived
# from reading model_fock_parf_v2.py / model_fock_parf_multixi.py /
# model_aniso_gaussian_vtheta.py, not executed against the live model, so a
# `[replay][WARN] could not instrument ...` line for any one of them is
# possible and does not invalidate the others -- report which ones fire so
# they can be patched up.
#
# Usage: this cell only *defines* functions, so it is safe (and, since
# 2026-08-30 companion note SS37, intended) to run it right after Cell 5
# builds `model`, before Cell 6 starts the long-running training loop --
# that's the whole point of interrupt -> replay_all_captures() ->
# run_training(next_step, TOTAL_STEPS) needing nothing predefined elsewhere.
# The functions below still *call* into a handful of Cell-6 globals
# (forward_with_vreg, LAMBDA_V/LAMBDA_FOCK_REG/FOCK_REG_EPS, _GRAD_CLIP_CFG,
# WATCHDOG_EXCLUDE_GROUPS, CKPT_DIR/CKPT_PREFIX) looked up at call time, so
# Cell 6 must have at least started (its config/setup lines run near-
# instantly, well before the loop itself blocks) before you actually call
# replay_spike_batch(...) / replay_all_captures() -- just not before this
# cell defines them.
#   replay_spike_batch(34091)
#   replay_spike_batch(32139)
#   replay_all_captures()

import copy as _copy
from grad_clip_utils import assign_clip_group, per_group_grad_norms


def _isolated_grad_snapshot(mdl):
    """Save existing .grad tensors (or None) for every parameter."""
    return {n: (p.grad.detach().clone() if p.grad is not None else None)
            for n, p in mdl.named_parameters()}


def _isolated_grad_restore(mdl, saved):
    for n, p in mdl.named_parameters():
        g = saved.get(n)
        p.grad = g.clone() if g is not None else None


# 2026-09-09 (companion note SS47 follow-up): replay_spike_batch and
# attribute_spike_rows both predate clip_then_sum (SS45.3-45.4) and, until
# this fix, just did a plain per-microbatch `.backward()` with no clipping
# in between -- exactly `sum_then_clip` semantics. For any capture taken
# AFTER clip_then_sum went live for CLIP_THEN_SUM_GROUPS (E/P in this run),
# that mismatch showed up as a ~80-105% fidelity-check gap purely on E/P:
# replayed E/P group norms landing at ~1100+ (raw, unclipped, summed over
# all GRAD_ACCUM microbatches) against a captured top_groups list that
# didn't even mention E/P (they'd already been clipped to
# CLIP_THEN_SUM_THRESHOLD *per microbatch*, live, before ever being
# summed). Confirmed clean: every OTHER group's replayed norm matched its
# captured Phase-0 value exactly (to displayed precision) in all three of
# steps 70522/71194/71703 -- E/P were the only groups affected, and by a
# huge, easy-to-spot margin (an order of magnitude over everything else),
# not a subtle nondeterminism smell. `_cts_group_params` + the splice below
# reproduce the live loop's exact per-microbatch-clip-then-sum mechanics
# (same `nn.utils.clip_grad_norm_` call, same running-total-by-id(param)
# accumulation, same post-loop splice into `.grad`) so both replay
# functions are bit-faithful again for captures from the clip_then_sum era,
# while staying a true no-op (falls through to plain accumulation) against
# older captures/notebooks where CLIP_THEN_SUM_GROUPS is empty or absent.
def _cts_group_params(mdl):
    """{group_key: [nn.Parameter, ...]} for whichever groups this run's
    live training loop applies clip_then_sum to, resolved fresh against
    `mdl` (not the live `_CLIP_THEN_SUM_PARAMS` global, which is bound to
    `model` and PER_GROUP_CLIP at Cell 6 run time) via the same
    assign_clip_group() the watchdog/optimizer use. Returns {} (a true
    no-op downstream) if clip_then_sum isn't configured at all -- keeps
    this safe to call against pre-SS45.4 notebooks/bundles.
    """
    groups = globals().get('CLIP_THEN_SUM_GROUPS')
    cfg = globals().get('_GRAD_CLIP_CFG')
    if not groups or cfg is None or not globals().get('PER_GROUP_CLIP', False):
        return {}
    out = {}
    for n, p in mdl.named_parameters():
        if not p.requires_grad:
            continue
        key, _ = assign_clip_group(n, cfg)
        if key in groups:
            out.setdefault(key, []).append(p)
    return out


def _cts_apply_microbatch(cts_params, cts_running):
    """After ONE microbatch's backward(): jointly clip each clip_then_sum
    group's own contribution (identical mechanics to clip_grads_per_group /
    replay_clip_ablation's flattened-per-group norm), fold it into
    `cts_running` (keyed by id(param), matching Cell 6), then zero .grad so
    the normal per-microbatch accumulation below never double-counts it.
    In-place; `cts_running` is both read and written.
    """
    threshold = globals().get('CLIP_THEN_SUM_THRESHOLD')
    for gparams in cts_params.values():
        if not any(p.grad is not None for p in gparams):
            continue
        nn.utils.clip_grad_norm_(gparams, threshold)
        for p in gparams:
            if p.grad is None:
                continue
            pid = id(p)
            contrib = p.grad.detach().clone()
            cts_running[pid] = (cts_running[pid] + contrib
                                 if pid in cts_running else contrib)
            p.grad.zero_()


def _cts_splice_back(cts_params, cts_running):
    """After the full GRAD_ACCUM loop: replace each clip_then_sum group's
    (already-zeroed) `.grad` with its accumulated, per-microbatch-clipped
    running total -- identical to Cell 6's post-loop splice, so every
    downstream reader (per_group_grad_norms, clip_grads_per_group, the
    per-parameter/per-row attribution below) sees exactly what the live
    step saw.
    """
    for gparams in cts_params.values():
        for p in gparams:
            pid = id(p)
            if pid in cts_running:
                p.grad = cts_running[pid]


def replay_spike_batch(step_tag, mdl=None, top_k=12, verbose=True):
    """Deterministically replay the forward+backward captured by Phase 1
    at a GRAD_NORM_HARD_TRIGGER event, with op/layer-resolved forensics.

    step_tag : int or str -- the step number in the
        '..._step{tag}_spikebatch.pt' filename (e.g. 34091).
    mdl : the live `model` by default.

    Returns a dict report; also prints a human-readable summary. Restores
    mdl's weights/.grad/RNG state to what they were before the call.
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    # map_location='cpu' (not DEVICE): rng_state_cpu/rng_state_cuda must stay
    # plain CPU ByteTensors for torch.(cuda.)set_rng_state -- moving them to
    # DEVICE here breaks that ("RNG state must be a torch.ByteTensor"). The
    # model_state_dict tensors don't need this either: they're already
    # explicitly re-mapped with `.to(DEVICE)` per-tensor below.
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'[replay] loaded {path.name}  step={bundle["step"]}  '
          f'pre_clip_grad_norm={bundle["pre_clip_grad_norm"]}')
    if bundle.get('top_groups'):
        print('[replay] Phase-0 top groups at capture time: '
              + ', '.join(f'{k}={v}' for k, v in bundle['top_groups'].items()))

    # -- non-pollution: snapshot everything this call is about to disturb --
    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
    _restore_fns = []

    # -- (b) per-layer instrumentation: wrap whatever `_fock_layer_step` is
    #    CURRENTLY bound to (e.g. the aniso depth-routing wrapper installed
    #    by install_aniso_depth_routing), so this composes with it rather
    #    than clobbering it. Caveat: if USE_LAYER_CHECKPOINT is on, gradient
    #    checkpointing recomputes the forward during backward, so a hook
    #    may fire twice for the same layer -- the second firing overwrites
    #    the first, which is fine for peak-magnitude attribution. --
    _layer_grad_norms = {}

    def _make_layer_hook(ell):
        def _hook(grad):
            _layer_grad_norms[ell] = float(grad.detach().norm())
        return _hook

    # 2026-08-30 (companion note SS38 follow-up): creation_gate_qkv,
    # reverse_ch, and each V_theta bank are SINGLE shared modules called
    # once per _fock_layer_step invocation -- i.e. once per layer, L times
    # per forward pass -- unlike destruction_gates (a per-layer ModuleList,
    # so destruction_gates[idx] already disambiguates by construction).
    # Before this fix, _record()'s hooks on those shared modules simply
    # overwrote the same dict key on every layer's call, so
    # activation_extremes silently only ever reflected the LAST layer (7)
    # -- exactly the layers NOT implicated in the localized-blowup mode
    # (layers 0-2). _current_layer is set by the wrapper below just before
    # calling into the real layer step, so _record can tag every reading
    # with the layer that actually produced it.
    _current_layer = [-1]

    _orig_layer_step = mdl._fock_layer_step

    def _instrumented_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                  layer_idx, *args, **kwargs):
        _current_layer[0] = layer_idx
        out = _orig_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                layer_idx, *args, **kwargs)
        h_new = out[0]
        if torch.is_tensor(h_new) and h_new.requires_grad:
            h_new.register_hook(_make_layer_hook(layer_idx))
        return out

    mdl._fock_layer_step = _instrumented_layer_step
    _restore_fns.append(lambda: setattr(mdl, '_fock_layer_step', _orig_layer_step))

    # -- (c) forward-activation-extreme instrumentation (best-effort),
    #    now layer-resolved: _act_extremes[name] is {layer_idx: stats},
    #    not a single overwritten dict --
    _act_extremes = {}

    def _record(name, t, layer=None):
        # layer=None (default): tag with whatever layer is CURRENTLY
        # executing, per _current_layer above -- correct for hooks that
        # fire from inside _fock_layer_step (creation_gate_qkv, reverse_ch).
        # layer=-1 (explicit): this op is NOT layer-scoped at all -- pass
        # it explicitly rather than trusting _current_layer, which would
        # otherwise silently report whatever layer happened to run last
        # (companion note SS38 follow-up #2: V_theta banks are called
        # ONCE per forward pass on the full depth-stacked xi tensor, not
        # per _fock_layer_step -- their hook fires AFTER the layer loop
        # finishes, so _current_layer[0] would read 7, the last layer
        # processed, which is not what "layer" means for this op at all).
        if layer is None:
            layer = _current_layer[0]
        try:
            tf = t.detach().float()
            stats = {
                'min': float(tf.min()), 'max': float(tf.max()),
                'mean': float(tf.mean()),
            }
        except Exception as _e:
            stats = {'error': str(_e)}
        _act_extremes.setdefault(name, {})[layer] = stats

    # -- (c2) V_theta exponent HISTOGRAM (companion note SS39): min/max/mean
    #    cannot answer the question the localized-mode conjecture actually
    #    poses, which is one of DENSITY -- how many tokens sit in the narrow
    #    band where exp(exponent) is still numerically alive (> ~-10, i.e.
    #    exp() >= 4.5e-5) and the well therefore contributes real gradient,
    #    versus the overwhelming majority that have underflowed to exactly
    #    zero and contribute none at all. The max over ~16k tokens is a
    #    single order statistic, and it actually went the "wrong" way
    #    between the two modes (bank[3] max was -1.76/-1.73 on the two
    #    smooth captures vs. -2.48/-3.91 on the two localized ones), so
    #    per-bank occupancy counts are what's needed instead. --
    # Top edge is +inf, not 0, even though exponent = -0.5*(nonneg) can only
    # be <= 0 mathematically: at diff==0 it can come back a few 1e-12 ABOVE
    # zero in float32, and a bin list topped at 0.0 silently drops exactly
    # those slots -- which are live-band tokens, so the drop would understate
    # live_frac, the one number this histogram exists to measure. With +inf
    # the bins cover the whole real line and sum(counts) == numel always.
    _EXP_EDGES = [float('inf'), -1.0, -2.0, -5.0, -10.0, -20.0, -30.0,
                  -100.0, -1e3, -1e4, float('-inf')]
    _EXP_LIVE_BINS = 4          # bins spanning (-10, +inf]
    _exp_hist = {}

    def _accum_exponent_hist(idx, exponent):
        try:
            with torch.no_grad():
                e = exponent.detach().float().reshape(-1)
                counts = _exp_hist.setdefault(idx, [0] * (len(_EXP_EDGES) - 1))
                for b in range(len(_EXP_EDGES) - 1):
                    hi, lo = _EXP_EDGES[b], _EXP_EDGES[b + 1]
                    counts[b] += int(((e <= hi) & (e > lo)).sum())
        except Exception:
            pass

    # creation-gate temperature + cumulative-softmax Z (forward_prefix is
    # called directly, not via __call__, so a plain forward hook would
    # never fire -- monkeypatch the bound method instead).
    try:
        _cg = mdl.creation_gate_qkv
        _orig_fp = _cg.forward_prefix

        def _wrapped_fp(h, r):
            readout, alpha_max = _orig_fp(h, r)
            _record('creation_gate.alpha_max (~Z-normalised top weight)', alpha_max)
            if getattr(_cg, 'log_tau', None) is not None:
                _record('creation_gate.tau', _cg.log_tau.exp().clamp(min=1e-4))
            return readout, alpha_max

        _cg.forward_prefix = _wrapped_fp
        _restore_fns.append(lambda: setattr(_cg, 'forward_prefix', _orig_fp))
    except Exception as _e:
        print(f'[replay][WARN] could not instrument creation_gate_qkv: {_e}')

    # reverse channel: logit_scale is a static parameter (read directly,
    # no hook needed); Q_force is the batch-dependent activation the
    # companion note flags as the non-conservative injection -- hook the
    # module call directly since `self.reverse_ch(...)` goes via __call__.
    try:
        _rc = mdl.reverse_ch
        if getattr(_rc, 'logit_scale', None) is not None:
            _record('reverse_ch.logit_scale (exp, clamped)',
                     _rc.logit_scale.exp().clamp(
                         max=getattr(_rc, 'logit_scale_max', 100.0)))

        def _rc_fwd_hook(module, inputs, output):
            _record('reverse_ch.Q_force', output)

        _h_rc = _rc.register_forward_hook(_rc_fwd_hook)
        _restore_fns.append(_h_rc.remove)
    except Exception as _e:
        print(f'[replay][WARN] could not instrument reverse_ch: {_e}')

    # V_theta: exponent / diag-vs-low-rank split and the depth-code-shifted
    # xi magnitude, recomputed from each per-channel bank's own public
    # _components() using the (xi, h) this forward call actually saw.
    try:
        _banks = mdl.V_theta.banks

        def _make_vtheta_hook(idx):
            def _vt_fwd_hook(module, args, kwargs, output):
                xi_in = args[0] if len(args) > 0 else kwargs.get('xis', kwargs.get('xi'))
                h_in = args[1] if len(args) > 1 else kwargs.get('h')
                try:
                    mu, a, w, B = module._components(xi_in)
                    h_e = h_in.unsqueeze(-2)
                    diff = h_e - mu
                    diag_term = (a * diff * diff).sum(dim=-1)
                    if B.shape[-1] > 0:
                        Bt_diff = torch.einsum('...kd,...kdr->...kr', diff, B)
                        lr_term = (Bt_diff * Bt_diff).sum(dim=-1)
                    else:
                        lr_term = torch.zeros_like(diag_term)
                    exponent = -0.5 * (diag_term + lr_term)
                    _accum_exponent_hist(idx, exponent)
                    _record(f'V_theta.bank[{idx}].exponent', exponent, layer=-1)
                    _record(f'V_theta.bank[{idx}].lr_term_share',
                             lr_term / (diag_term + lr_term).clamp(min=1e-12),
                             layer=-1)
                    _record(f'V_theta.bank[{idx}].xi_shifted_norm',
                             xi_in.detach().float().norm(dim=-1), layer=-1)
                except Exception as _e2:
                    _act_extremes[f'V_theta.bank[{idx}]'] = {'error': str(_e2)}
            return _vt_fwd_hook

        for _idx, _bank in enumerate(_banks):
            _h_vt = _bank.register_forward_hook(_make_vtheta_hook(_idx), with_kwargs=True)
            _restore_fns.append(_h_vt.remove)
    except Exception as _e:
        print(f'[replay][WARN] could not instrument V_theta banks: {_e}')

    # register salience gate: capture what the destruction gates emit --
    # salience itself is thresholded at cfg.register_salience_threshold
    # (0.005) upstream of this call; the gate's own output extremes are
    # the batch-dependent signal available without touching model source.
    try:
        _dgs = mdl.destruction_gates

        def _make_dg_hook(idx):
            def _dg_fwd_hook(module, inputs, output):
                _record(f'destruction_gates[{idx}].output', output)
            return _dg_fwd_hook

        for _idx, _dg in enumerate(_dgs):
            _h_dg = _dg.register_forward_hook(_make_dg_hook(_idx))
            _restore_fns.append(_h_dg.remove)
    except Exception as _e:
        print(f'[replay][WARN] could not instrument destruction_gates: {_e}')

    # -- official per-layer diagnostic buffer (companion note SS38 follow-up):
    #    FockMultiXiPARFLM.set_fock_capture(True) makes _fock_layer_step
    #    append a @torch.no_grad() dict of register/gate health scalars
    #    (active_frac, salience_mean/std, reg_cos_sim, destroy_mean,
    #    qforce_ratio, rev_scale, create_alpha_max, create_entropy,
    #    rev_entropy, rev_alpha_max) FOR EVERY LAYER on every forward --
    #    this is the same API the eval-time causal/register diagnostics use
    #    (model.eval(); model.set_fock_capture(True)), just enabled here
    #    during the replayed training-mode forward+backward instead. Being
    #    torch.no_grad() internally, it costs almost nothing and cannot
    #    perturb the backward pass it runs alongside.
    if not hasattr(mdl, 'set_fock_capture'):
        print('[replay][WARN] model has no set_fock_capture(); skipping '
              'per-layer register/gate diagnostic buffer.')
    else:
        mdl.set_fock_capture(True)
        _restore_fns.append(lambda: mdl.set_fock_capture(False))
    _fock_capture_per_microbatch = []

    try:
        # -- load the pinned pre-step weights + RNG state --
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        torch.set_rng_state(bundle['rng_state_cpu'])
        if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])

        # -- replay every captured microbatch exactly as the original
        #    GRAD_ACCUM loop did --
        for p in mdl.parameters():
            p.grad = None
        grad_accum = bundle.get('grad_accum', len(bundle['batches']))
        mdl.train()
        _cts_params = _cts_group_params(mdl)
        _cts_running = {}
        for xb, yb in bundle['batches']:
            x = torch.from_numpy(xb).to(DEVICE)
            y = torch.from_numpy(yb).to(DEVICE)
            if getattr(mdl, '_fock_capture', None) is not None:
                mdl._fock_capture = []  # drain the previous microbatch's entries
            loss, loss_ntp, v_reg, fock_reg = forward_with_vreg(
                x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
            if REGISTER_REPULSION:
                loss = loss + mdl.pop_repulsion_loss()
            (loss / grad_accum).backward()
            # clip_then_sum (SS45.3-45.4, wired into replay 2026-09-09 --
            # see the note by _cts_group_params above): fold this
            # microbatch's own CLIP_THEN_SUM_GROUPS contribution into the
            # running total BEFORE the next microbatch's backward() adds
            # onto the same .grad tensor, exactly like the live loop.
            _cts_apply_microbatch(_cts_params, _cts_running)
            if getattr(mdl, '_fock_capture', None) is not None:
                # keyed by layer -- prefix_causal mode's _fock_layer_stats
                # already reports the last-position (full-prefix) slice, so
                # one dict per layer per microbatch is the expected shape.
                _fock_capture_per_microbatch.append(
                    {s['layer']: s for s in mdl._fock_capture})

        # Splice clip_then_sum's accumulated, per-microbatch-clipped total
        # back into .grad for CLIP_THEN_SUM_GROUPS' params, replacing the
        # already-zeroed result of normal accumulation -- same as Cell 6's
        # post-GRAD_ACCUM-loop splice, so everything below (per-parameter
        # norms, per_group_grad_norms, the fidelity check against
        # bundle['pre_clip_grad_norm']) reads exactly what the live step's
        # optimizer/watchdog actually saw.
        _cts_splice_back(_cts_params, _cts_running)

        # -- (a) per-parameter grad norms (finer than per-group) --
        _param_norms = {n: float(p.grad.detach().norm())
                         for n, p in mdl.named_parameters() if p.grad is not None}
        _top_params = sorted(_param_norms.items(), key=lambda kv: kv[1],
                              reverse=True)[:top_k]
        _total_sq_excl, _total_sq_all = 0.0, 0.0
        for n, v in _param_norms.items():
            _total_sq_all += v * v
            _key, _ = assign_clip_group(n, _GRAD_CLIP_CFG)
            if _key not in WATCHDOG_EXCLUDE_GROUPS:
                _total_sq_excl += v * v
        _total_norm_excl = _total_sq_excl ** 0.5
        _total_norm_all = _total_sq_all ** 0.5
        _replayed_pg_norms = per_group_grad_norms(mdl, _GRAD_CLIP_CFG)

        _recorded = bundle['pre_clip_grad_norm']
        _fidelity_pct = 100 * abs(_total_norm_excl - _recorded) / max(_recorded, 1e-9)

        report = {
            'step': bundle['step'],
            'pre_clip_grad_norm_recorded': _recorded,
            'pre_clip_grad_norm_replayed_matching': _total_norm_excl,
            'pre_clip_grad_norm_replayed_all_groups': _total_norm_all,
            'fidelity_gap_pct': _fidelity_pct,
            'top_params': _top_params,
            'replayed_group_norms': _replayed_pg_norms,
            'layer_grad_norms': dict(sorted(_layer_grad_norms.items())),
            'activation_extremes': _act_extremes,
            'fock_capture_per_microbatch': _fock_capture_per_microbatch,
            'vtheta_exponent_hist': {'edges': list(_EXP_EDGES),
                                      'counts': {k: list(v)
                                                 for k, v in _exp_hist.items()}},
        }

        if verbose:
            print(f'\n[replay] fidelity check: replayed (matching-groups) total='
                  f'{_total_norm_excl:.1f}  vs. recorded pre_clip_grad_norm='
                  f'{_recorded:.1f}  (diff {_fidelity_pct:.1f}%)')
            if _fidelity_pct > 5.0:
                print('[replay][WARN] fidelity gap > 5% -- replay may not be '
                      'bit-exact (cuDNN nondeterminism, an unaccounted RNG '
                      'consumer, or a mismatched batch/weight pairing). '
                      'Treat the attribution below as approximate.')
            print(f'[replay] replayed total incl. reverse-channel groups='
                  f'{_total_norm_all:.1f}  (excl.={_total_norm_excl:.1f}; the gap '
                  f'is what the watchdog aggregate would have missed, SS33.3 Phase 0)')
            print('[replay] replayed per-group norms (top 8):')
            for k, v in sorted(_replayed_pg_norms.items(), key=lambda kv: kv[1],
                                reverse=True)[:8]:
                print(f'    {v:10.2f}  {k}')
            print('[replay] top per-parameter grad norms:')
            for n, v in _top_params:
                print(f'    {v:10.2f}  {n}')
            print('[replay] per-layer grad norm (gradient flowing INTO each layer boundary):')
            for ell, v in sorted(_layer_grad_norms.items()):
                print(f'    layer {ell:2d}: {v:10.2f}')
            print('[replay] forward-activation extremes at risky ops (by layer; '
                  '-1 = not layer-attributable):')
            for k, by_layer in _act_extremes.items():
                for ell, v in sorted(by_layer.items()):
                    print(f'    {k}  layer {ell:2d}: {v}')
            if _exp_hist:
                print('[replay] V_theta exponent occupancy per bank (token-slot '
                      'counts summed over all microbatches; "live" = exponent '
                      '> -10, where exp() >= 4.5e-5 and the well still '
                      'contributes gradient):')
                _lbl = [f'({_EXP_EDGES[b+1]:g},{_EXP_EDGES[b]:g}]'
                        for b in range(len(_EXP_EDGES) - 1)]
                print('    bank  ' + '  '.join(f'{s:>13}' for s in _lbl)
                      + f'  {"live_n":>10}  {"live_frac":>11}')
                for _bi in sorted(_exp_hist):
                    _c = _exp_hist[_bi]
                    _tot = sum(_c) or 1
                    _live = sum(_c[:_EXP_LIVE_BINS])
                    print(f'    {_bi:4d}  ' + '  '.join(f'{v:13d}' for v in _c)
                          + f'  {_live:10d}  {_live/_tot:11.4e}')
            if _fock_capture_per_microbatch:
                # 2026-08-30 (companion note SS38 follow-up): print the LAST
                # microbatch's per-layer register/gate health table -- the
                # one whose forward+backward is what pre_clip_grad_norm
                # above was actually measured on when grad_accum==1 (the
                # common case for these captures); earlier microbatches
                # (if grad_accum>1) are still in the returned report dict.
                _last_fc = _fock_capture_per_microbatch[-1]
                print(f'[replay] per-layer register/gate health '
                      f'(microbatch {len(_fock_capture_per_microbatch)-1} of '
                      f'{len(_fock_capture_per_microbatch)}, via set_fock_capture):')
                _fc_cols = ['active_frac', 'salience_mean', 'reg_cos_sim',
                            'destroy_mean', 'create_alpha_max', 'create_entropy',
                            'rev_entropy', 'rev_alpha_max', 'rev_scale',
                            'qforce_ratio']
                _fc_cols = [c for c in _fc_cols
                            if any(c in s for s in _last_fc.values())]
                print('    layer  ' + '  '.join(f'{c:>16}' for c in _fc_cols))
                for ell in sorted(_last_fc):
                    _row = _last_fc[ell]
                    print(f'    {ell:5d}  ' + '  '.join(
                        f'{_row.get(c, float("nan")):16.4g}' for c in _fc_cols))
    finally:
        # -- restore everything (non-pollution invariant), even on error --
        for _fn in reversed(_restore_fns):
            try:
                _fn()
            except Exception:
                pass
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    return report


def inspect_spike_tokens(step_tag, batch_idx=None, show_n=3, snippet_chars=240):
    """Decode the exact offending microbatch(es) captured at `step_tag` and
    print/return degeneracy stats (companion note SS38 follow-up SS38.4: no
    forward-activation signal separated the two localized events from the
    smooth-cascade ones, so this looks at the OTHER half of the forward
    pass -- what the model was actually reading -- for the handful of rows
    in the batch with the largest per-row max-token-repeat-run, on the
    theory that a degenerate run (e.g. a long whitespace/punctuation/
    boilerplate repeat straight out of OpenWebText) could be what triggers
    a localized early-layer blowup.

    This is pure CPU bookkeeping -- no model, no GPU, no RNG/weight state
    touched -- so it is safe to call at any time, interleaved with training
    or other replays, and does not require Cell 6 to have run at all
    (only CKPT_DIR/CKPT_PREFIX and the `tok` GPT-2 tokenizer from Cell 3).

    step_tag  : int or str, as in replay_spike_batch.
    batch_idx : which microbatch in the capture to inspect (default: all).
    show_n    : how many of the batch's rows (ranked by max-repeat-run,
                descending) to print a decoded snippet for.
    """
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    mb_indices = range(len(bundle['batches'])) if batch_idx is None else [batch_idx]
    all_rows = []
    for mb in mb_indices:
        xb, _yb = bundle['batches'][mb]
        for row in range(xb.shape[0]):
            ids = xb[row].tolist()
            n = len(ids)
            uniq = len(set(ids))
            # longest run of the same token id back-to-back.
            best_run, cur_run, cur_id = 1, 1, ids[0]
            for t in ids[1:]:
                if t == cur_id:
                    cur_run += 1
                else:
                    cur_run, cur_id = 1, t
                best_run = max(best_run, cur_run)
            all_rows.append({
                'microbatch': mb, 'row': row, 'seq_len': n,
                'unique_token_ratio': uniq / n, 'max_repeat_run': best_run,
                'ids': ids,
            })

    all_rows.sort(key=lambda r: r['max_repeat_run'], reverse=True)
    print(f'[tokens] step {bundle["step"]}: {len(all_rows)} row(s) across '
          f'{len(list(mb_indices))} microbatch(es), pre_clip_grad_norm='
          f'{bundle["pre_clip_grad_norm"]}')
    print(f'{"mb":>3} {"row":>4} {"len":>5} {"uniq_ratio":>10} {"max_repeat_run":>14}')
    for r in all_rows:
        print(f'{r["microbatch"]:3d} {r["row"]:4d} {r["seq_len"]:5d} '
              f'{r["unique_token_ratio"]:10.3f} {r["max_repeat_run"]:14d}')

    print(f'\n[tokens] decoded snippet for the top {min(show_n, len(all_rows))} '
          f'row(s) by max_repeat_run (most likely to show degenerate text):')
    for r in all_rows[:show_n]:
        text = tok.decode(r['ids'])
        snippet = text[:snippet_chars]
        print(f'\n  -- microbatch {r["microbatch"]}, row {r["row"]} '
              f'(len={r["seq_len"]}, unique_ratio={r["unique_token_ratio"]:.3f}, '
              f'max_repeat_run={r["max_repeat_run"]}) --')
        print('  ' + snippet.replace('\n', '\\n'))

    return all_rows


def attribute_spike_rows(step_tag, mdl=None, batch_idx=None,
                          track=('V_theta.depth_code',), verbose=True):
    """Per-row (per-sequence) gradient attribution for a captured spike.

    Companion note SS39. The premise: per-group clipping already caps
    `depth_code` at 0.25 on EVERY step, quiet or spiking (Cell 6's
    GRAD_CLIP_OVERRIDES, tightened 0.5 -> 0.25 on 2026-08-23 precisely
    because it was saturating its ceiling on every quiet step). So a
    localized event's applied `depth_code` update is exactly the same SIZE
    as a quiet step's -- 0.25 either way, whether the pre-clip norm was 88
    or 425. Its damage therefore cannot be magnitude; it has to be
    DIRECTION. The conjecture is that the direction gets dictated by a
    small minority of rows whose tokens land near a sharp V_theta well at
    layers 0-2 -- the only layers carrying meaningful salience
    (~0.32/0.14-0.22/0.06-0.15 at layers 0-2 vs. ~1e-3-1e-4 at layers 5-6,
    identically across all four replays so far, SS38.7).

    This tests that head-on: re-run the captured batch ONE ROW AT A TIME,
    each row scaled by 1/(grad_accum * rows_per_microbatch) so its number
    is its own share of the aggregate, and report per-row gradient norms
    for the tracked parameters plus the layer-0 h-boundary gradient. If the
    conjecture holds, the localized captures (39,983 / 41,837) should show
    one or two rows owning most of the total while the smooth-cascade
    captures (37,763 / 41,318) should look flat.

    Two honest caveats about what "per row" can mean here:

    1. The BAOAB O-step draws noise, so a row replayed alone does not see
       the noise realisation it saw inside its batch of 8. The RNG is
       therefore reset to the capture's pinned state before EVERY row, so
       all rows are compared under an identical draw -- which is what a
       relative attribution needs -- rather than each reproducing its exact
       in-batch contribution.
    2. Gradient norms don't add: ||sum_i g_i|| != sum_i ||g_i||. The
       concentration metric is therefore max_i ||g_i|| / sum_i ||g_i||,
       which sits at 1/n_rows when every row contributes equally and tends
       to 1 when a single row owns the gradient. ||sum_i g_i|| is also
       reported as a rough cross-check against replay_spike_batch's figure
       for the same parameter -- rough, not exact, because of caveat 1 and
       because the register-repulsion term (a parameter regulariser, not a
       per-row data term) is drained but excluded here.
    3. For any `track` name in CLIP_THEN_SUM_GROUPS (E/P in this run,
       SS45.3-45.4): `norm_of_summed_grad` here is NOT expected to match
       replay_spike_batch's (2026-09-09-fixed) group norm for that
       parameter, and that's fine, not a bug. clip_then_sum's joint clip
       operates on a whole MICROBATCH's gradient (all its rows already
       summed together internally by autograd); this function forwards one
       row completely alone, so there is no multi-row microbatch gradient
       here for a threshold-based clip to act on in the first place --
       reproducing clip_then_sum per single row would be a different,
       not-live-representative computation, so this function deliberately
       still reports each row's raw, unclipped contribution (fine for the
       top1_share/top3_share concentration metrics this function exists
       for; just don't expect the summed/clipped total to reconcile
       against replay_spike_batch's for these two parameters).

    Same non-pollution invariant as replay_spike_batch: weights, .grad
    tensors and RNG state are snapshotted up front and restored in a
    finally block, so this is safe to interleave with training.
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'[rows] loaded {path.name}  step={bundle["step"]}  '
          f'pre_clip_grad_norm={bundle["pre_clip_grad_norm"]}')

    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
    _restore_fns = []

    _layer0_grad = [float('nan')]
    _orig_layer_step = mdl._fock_layer_step

    def _instrumented_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                  layer_idx, *args, **kwargs):
        out = _orig_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                layer_idx, *args, **kwargs)
        h_new = out[0]
        if layer_idx == 0 and torch.is_tensor(h_new) and h_new.requires_grad:
            h_new.register_hook(
                lambda g: _layer0_grad.__setitem__(0, float(g.detach().norm())))
        return out

    mdl._fock_layer_step = _instrumented_layer_step
    _restore_fns.append(lambda: setattr(mdl, '_fock_layer_step', _orig_layer_step))

    _name2param = dict(mdl.named_parameters())
    _absent = [n for n in track if n not in _name2param]
    if _absent:
        print(f'[rows][WARN] not in named_parameters(), skipping: {_absent}')
    track = [n for n in track if n in _name2param]

    def _reset_rng():
        torch.set_rng_state(bundle['rng_state_cpu'])
        if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])

    rows = []
    _accum = {n: None for n in track}
    try:
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        grad_accum = bundle.get('grad_accum', len(bundle['batches']))
        mb_indices = (list(range(len(bundle['batches']))) if batch_idx is None
                      else [batch_idx])
        mdl.train()
        for mb in mb_indices:
            xb, yb = bundle['batches'][mb]
            n_rows = xb.shape[0]
            for row in range(n_rows):
                for p in mdl.parameters():
                    p.grad = None
                _layer0_grad[0] = float('nan')
                _reset_rng()
                x = torch.from_numpy(xb[row:row + 1]).to(DEVICE)
                y = torch.from_numpy(yb[row:row + 1]).to(DEVICE)
                loss, loss_ntp, v_reg, fock_reg = forward_with_vreg(
                    x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                if REGISTER_REPULSION:
                    mdl.pop_repulsion_loss()   # drained, deliberately not added
                (loss / (grad_accum * n_rows)).backward()
                _tot_sq = 0.0
                for p in mdl.parameters():
                    if p.grad is not None:
                        _tot_sq += float(p.grad.detach().norm()) ** 2
                _rec = {
                    'microbatch': mb, 'row': row,
                    'ntp': float(loss_ntp.detach()),
                    'layer0_h_grad': _layer0_grad[0],
                    'total_grad_norm': _tot_sq ** 0.5,
                }
                for n in track:
                    g = _name2param[n].grad
                    _rec[n] = float(g.detach().norm()) if g is not None else 0.0
                    if g is not None:
                        _accum[n] = (g.detach().clone() if _accum[n] is None
                                     else _accum[n] + g.detach())
                rows.append(_rec)
    finally:
        for _fn in reversed(_restore_fns):
            try:
                _fn()
            except Exception:
                pass
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    summary = {}
    for n in list(track) + ['layer0_h_grad', 'total_grad_norm']:
        vals = [r[n] for r in rows if r.get(n) is not None
                and not (isinstance(r[n], float) and math.isnan(r[n]))]
        _s = sum(vals)
        _ranked = sorted(vals, reverse=True)
        summary[n] = {
            'sum_of_row_norms': _s,
            'max_row_norm': _ranked[0] if _ranked else 0.0,
            'top1_share': (_ranked[0] / _s) if _s > 0 else float('nan'),
            'top3_share': (sum(_ranked[:3]) / _s) if _s > 0 else float('nan'),
            'uniform_share_baseline': (1.0 / len(vals)) if vals else float('nan'),
            'norm_of_summed_grad': (float(_accum[n].norm())
                                     if _accum.get(n) is not None else None),
        }

    report = {
        'step': bundle['step'],
        'pre_clip_grad_norm_recorded': bundle['pre_clip_grad_norm'],
        'rows': rows,
        'summary': summary,
    }

    if verbose:
        _key = track[0] if track else 'total_grad_norm'
        print(f'\n[rows] per-row attribution, ranked by {_key}  (each row scaled '
              f'by 1/(grad_accum*rows_per_mb), i.e. its own share of the aggregate):')
        _cols = ['ntp', 'layer0_h_grad', 'total_grad_norm'] + list(track)
        print(f'{"mb":>3} {"row":>4}  ' + '  '.join(f'{c:>20}' for c in _cols))
        for r in sorted(rows, key=lambda r: r.get(_key, 0.0), reverse=True):
            print(f'{r["microbatch"]:3d} {r["row"]:4d}  ' + '  '.join(
                f'{r.get(c, float("nan")):20.6g}' for c in _cols))
        for n, s in summary.items():
            print(f'\n[rows] concentration for {n}:')
            print(f'    top-1 row / sum-of-row-norms : {s["top1_share"]:.4f}'
                  f'   (flat-batch baseline {s["uniform_share_baseline"]:.4f})')
            print(f'    top-3 rows / sum-of-row-norms : {s["top3_share"]:.4f}')
            print(f'    sum of ||row grad||           : {s["sum_of_row_norms"]:.4f}')
            if s['norm_of_summed_grad'] is not None:
                print(f'    ||sum of row grads||          : '
                      f'{s["norm_of_summed_grad"]:.4f}   (rough cf. '
                      f'replay_spike_batch for this param; see caveats)')
    return report


def replay_precision_cap_ablation(step_tag, budgets=(1.0, 4.0, None), mdl=None,
                                   top_k=8, verbose=True):
    """Companion note SS41.6/SS41.7 item 1: re-run a captured spike batch
    under several `precision_lr_max` budgets -- weights, batch, and RNG
    state held bit-identical to the capture via the same snapshot/restore
    invariant `replay_spike_batch` uses -- to check whether capping
    sigma_max(B_k)^2 (model_aniso_gaussian_vtheta.py's `_bound_lowrank`)
    tames the exponent blow-up and the resulting grad-norm, BEFORE
    committing to resuming live training with the cap switched on.

    Mirrors `replay_integrator_ablation` (companion note SS40), but swaps
    `bank._precision_lr_max` across arms instead of `cfg.integrator` --
    `_bound_lowrank` reads `self._precision_lr_max` fresh on every forward
    call (it is a plain Python attribute, not a buffer or nn.Parameter),
    so this is a live-model attribute swap, not a new checkpoint.

    budgets : iterable of float or None. `None` disables `_bound_lowrank`
        entirely (uncapped).
        NOTE (2026-09-10): this used to describe None as "the value
        PRECISION_LR_MAX has actually been training with" -- true when
        written (SS41/SS42), but PRECISION_LR_MAX has been 1.0 since
        SS42.6, so None is now a COUNTERFACTUAL arm, not the baseline.
        The summary below takes whichever arm matches the live
        PRECISION_LR_MAX as its baseline, marks it [AS TRAINED], and
        checks it against bundle['pre_clip_grad_norm'] -- that comparison
        is this helper's fidelity check, so keep the live value in
        `budgets`. A float `b` sets
        `sigma_max(B_k)^2 <= b` on EVERY V_theta bank
        (`bank._precision_lr_max = b`, via `mdl.V_theta.banks`) for the
        duration of that arm only.
    mdl : the live `model` by default.

    Same non-pollution invariant as replay_spike_batch: weights, .grad
    tensors, RNG state, and every bank's `_precision_lr_max` are
    snapshotted up front and restored in a `finally` block, so this is
    safe to call on the live model mid-training.

    Returns {arm_label: {budget, pre_clip_grad_norm, replayed_group_norms,
    per_layer_h_grad, vtheta_exponent_min}}.
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'[precap] loaded {path.name}  step={bundle["step"]}  '
          f'pre_clip_grad_norm={bundle["pre_clip_grad_norm"]}')

    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
    _banks = mdl.V_theta.banks
    _saved_budgets = [b._precision_lr_max for b in _banks]
    _orig_layer_step = mdl._fock_layer_step
    _vt_hooks = []

    _layer_grad = {}

    def _instrumented_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                  layer_idx, *args, **kwargs):
        out = _orig_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                layer_idx, *args, **kwargs)
        h_new = out[0]
        if torch.is_tensor(h_new) and h_new.requires_grad:
            def _layer_hook(g, li=layer_idx):
                # NOTE: must not `return` the setdefault(...) result --
                # torch treats any non-None tensor-hook return value as a
                # gradient replacement, and dict.setdefault(...) returns
                # the (float) value, which crashes autograd with
                # "expected Variable, but hook returned 'float'".
                _layer_grad.setdefault(li, float(g.detach().norm()))
            h_new.register_hook(_layer_hook)
        return out

    # per-bank exponent min -- the direct check that the cap is actually
    # biting (bank._components() already applies _bound_lowrank internally,
    # so this reads the SAME B the forward pass used, capped or not).
    _exp_min = {}

    def _make_vtheta_hook(idx):
        def _vt_fwd_hook(module, args, kwargs, output):
            xi_in = args[0] if len(args) > 0 else kwargs.get('xis', kwargs.get('xi'))
            h_in = args[1] if len(args) > 1 else kwargs.get('h')
            try:
                mu, a, w, B = module._components(xi_in)
                h_e = h_in.unsqueeze(-2)
                diff = h_e - mu
                diag_term = (a * diff * diff).sum(dim=-1)
                if B.shape[-1] > 0:
                    Bt_diff = torch.einsum('...kd,...kdr->...kr', diff, B)
                    lr_term = (Bt_diff * Bt_diff).sum(dim=-1)
                else:
                    lr_term = torch.zeros_like(diag_term)
                exponent = -0.5 * (diag_term + lr_term)
                _cur = _exp_min.get(idx, float('inf'))
                _exp_min[idx] = min(_cur, float(exponent.detach().min()))
            except Exception:
                pass
        return _vt_fwd_hook

    results = {}
    try:
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        mdl._fock_layer_step = _instrumented_layer_step
        for _idx, _bank in enumerate(_banks):
            _h_vt = _bank.register_forward_hook(_make_vtheta_hook(_idx), with_kwargs=True)
            _vt_hooks.append(_h_vt)
        grad_accum = bundle.get('grad_accum', len(bundle['batches']))
        mdl.train()

        for budget in budgets:
            _live_budget = globals().get('PRECISION_LR_MAX', None)
            label = ('uncapped (None)' if budget is None
                      else f'precision_lr_max={budget}')
            if budget == _live_budget:
                label += ' [AS TRAINED]'
            for b in _banks:
                b._precision_lr_max = budget
            _layer_grad.clear()
            _exp_min.clear()
            for p in mdl.parameters():
                p.grad = None
            torch.set_rng_state(bundle['rng_state_cpu'])
            if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
                torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])

            # 2026-09-10 (SS48.2/SS48.3, applied here belatedly): mirror the
            # live loop's clip_then_sum mechanics AND its register-repulsion
            # term. Both were spliced into replay_spike_batch but never into
            # this helper, so any bundle captured after clip_then_sum went
            # live (~step 52,550) replayed E/P raw, unclipped and fully
            # summed (~1,100+) -- large enough to swamp events that are
            # themselves only 100-500, and roughly arm-independent, so every
            # arm would look alike and the cap would look useless.
            # _cts_group_params returns {} when clip_then_sum isn't
            # configured, so this is a true no-op on pre-SS45.4 bundles
            # (47,116 / 48,507 / 48,917) and SS42's results stand.
            _cts_params = _cts_group_params(mdl)
            _cts_running = {}
            for xb, yb in bundle['batches']:
                x = torch.from_numpy(xb).to(DEVICE)
                y = torch.from_numpy(yb).to(DEVICE)
                loss, *_ = forward_with_vreg(
                    x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                if REGISTER_REPULSION:
                    loss = loss + mdl.pop_repulsion_loss()
                (loss / grad_accum).backward()
                _cts_apply_microbatch(_cts_params, _cts_running)
            _cts_splice_back(_cts_params, _cts_running)

            _pg = per_group_grad_norms(mdl, _GRAD_CLIP_CFG)
            _total_sq_excl = sum(v * v for k, v in _pg.items()
                                   if k not in WATCHDOG_EXCLUDE_GROUPS)
            results[label] = {
                'budget': budget,
                'pre_clip_grad_norm': _total_sq_excl ** 0.5,
                'replayed_group_norms': _pg,
                'per_layer_h_grad': dict(sorted(_layer_grad.items())),
                'vtheta_exponent_min': dict(sorted(_exp_min.items())),
            }
            if verbose:
                _top = sorted(_pg.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
                print(f'\n[precap] {label}:')
                print(f'    pre_clip_grad_norm={results[label]["pre_clip_grad_norm"]:.2f}')
                print('    top groups: '
                      + ', '.join(f'{k}={v:.2f}' for k, v in _top))
                print('    per-layer h-grad: '
                      + ', '.join(f'L{ell}={v:.4f}' for ell, v in
                                  results[label]['per_layer_h_grad'].items()))
                print('    V_theta exponent min per bank: '
                      + ', '.join(f'bank{idx}={v:.1f}' for idx, v in
                                  results[label]['vtheta_exponent_min'].items()))
    finally:
        mdl._fock_layer_step = _orig_layer_step
        for _h in _vt_hooks:
            _h.remove()
        for b, budget in zip(_banks, _saved_budgets):
            b._precision_lr_max = budget
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    if verbose and results:
        _recorded = bundle['pre_clip_grad_norm']
        # Baseline = whichever arm matches the LIVE PRECISION_LR_MAX, i.e. the
        # configuration this bundle was actually captured under. Pre-SS42.6
        # bundles were captured at None, so this still resolves correctly for
        # them provided PRECISION_LR_MAX matches what that run used.
        _live_budget = globals().get('PRECISION_LR_MAX', None)
        _base_label = next((lbl for lbl, r in results.items()
                            if r['budget'] == _live_budget), None)
        _base = (results[_base_label]['pre_clip_grad_norm']
                 if _base_label is not None else None)
        print(f'\n[precap] summary (step={bundle["step"]}, recorded pre-clip='
              f'{_recorded:.1f} at capture time):')
        print(f'    {"arm":<36} {"pre_clip":>10}  {"vs as-trained":>14}')
        for label, r in results.items():
            _ratio = (f'{r["pre_clip_grad_norm"] / _base:.3f}x'
                       if _base else 'n/a')
            print(f'    {label:<36} {r["pre_clip_grad_norm"]:10.2f}  {_ratio:>14}')
        if _base_label is None:
            print(f'    (no arm at the live PRECISION_LR_MAX={_live_budget!r} '
                  '-- add it to `budgets` for a like-for-like baseline and '
                  'for the fidelity check)')
        elif _recorded:
            _fid = abs(_base - _recorded) / abs(_recorded) * 100.0
            print(f'    fidelity vs capture: {_fid:.1f}%  ({_base:.2f} '
                  f'replayed vs {_recorded:.2f} recorded)'
                  + ('' if _fid <= 5.0 else
                     '   <-- WARNING: >5%, do NOT read the arms until this '
                     'is explained (SS48.1 was exactly this signature)'))
    return results


def replay_curvature_rebalance_ablation(step_tag, precision_max_grid=None,
                                         precision_lr_max_grid=(1.0, 0.25),
                                         mdl=None, top_k=8, verbose=True):
    """2-D sweep over (precision_max, precision_lr_max) -- the 'degree of
    anisotropy' question (2026-09-11 discussion): under baoab_cfc the
    DIAGONAL precision a_k is integrated EXACTLY by the CfC harmonic
    propagator (immune to the well-sharpening blow-up by construction),
    while the LOW-RANK term B_k B_k^T is still an explicit kick with a
    genuine omega*dt<2 stability wall. Yet `precision_max` (the diagonal
    cap) is set from `2.0/d` in Cell 5 -- exactly the Verlet-era
    omega*dt<2 fingerprint (summed over d coordinates: d*(2/d)=2), a
    constraint that stopped applying to this channel the moment baoab_cfc
    was adopted, and nobody moved it since. Measured: init a ~ 0.0027,
    ceiling 2/d ~ 0.0052 (only 1.93x headroom), while ambient uncapped
    sigma_max(B_k)^2 runs ~283-310 (companion note SS42.4) -- the
    reachable LOW-RANK curvature is ~54,000x the diagonal ceiling, even
    though the low-rank channel is the only one of the two that is
    numerically dangerous. So essentially all curvature has nowhere to go
    but the dangerous channel, which is also consistent with SS9's
    finding that >99.9% of well-token pairs are numerically dead (a
    rank-4 razor ridge in 384 dimensions has almost no measure).

    This helper asks directly: does shifting curvature INTO the
    exactly-integrated diagonal channel (raising precision_max) collapse
    the spike as well as capping the low-rank channel alone
    (precision_lr_max, already covered by replay_precision_cap_ablation)
    does -- and does it un-saturate the wells (vtheta_exponent_min moving
    off ~-10^5) rather than merely suppressing the grad-norm symptom?

    Same mechanics as replay_precision_cap_ablation (its docstring has
    the full rationale for the snapshot/restore invariant, the
    clip_then_sum splice, and the exponent-min hook), generalised to two
    independent live attributes instead of one -- both `_precision_max`
    and `_precision_lr_max` are plain Python attributes read fresh every
    forward call (model_aniso_gaussian_vtheta.py's `_components` /
    `_bound_lowrank`), so this is a live-model attribute swap on both
    axes, not a new checkpoint.

    precision_max_grid : iterable of float, or None (default) to resolve
        to (2/d, 20/d, 200/d) using the live model's own d -- the exact
        sweep proposed in the 2026-09-11 discussion. Unlike
        precision_lr_max, precision_max has no legitimate `None`
        (uncapped) arm here: Cell 5 always constructs every bank with a
        concrete value, so there is no "as the model was built" state to
        compare against other than the live value itself.
    precision_lr_max_grid : iterable of float or None, crossed with every
        precision_max_grid entry. Default (1.0, 0.25) mirrors the two
        budgets already evidenced by replay_precision_cap_ablation.
    mdl : the live `model` by default.

    Returns {arm_label: {precision_max, precision_lr_max,
    pre_clip_grad_norm, replayed_group_norms, per_layer_h_grad,
    vtheta_exponent_min}}.
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'[rebal] loaded {path.name}  step={bundle["step"]}  '
          f'pre_clip_grad_norm={bundle["pre_clip_grad_norm"]}')

    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
    _banks = mdl.V_theta.banks
    _saved_lr_max = [b._precision_lr_max for b in _banks]
    _saved_prec_max = [b._precision_max for b in _banks]
    # The value Cell 5 actually built this model with -- the single
    # source of truth for the [AS TRAINED] comparison below, rather than
    # recomputing 2.0/d again and risking the two silently diverging if
    # Cell 5's formula ever changes.
    _live_precision_max = _saved_prec_max[0]
    _d = getattr(mdl.V_theta, 'd', None)
    if precision_max_grid is None:
        if _d is None:
            raise RuntimeError('mdl.V_theta has no .d; pass precision_max_grid '
                               'explicitly.')
        precision_max_grid = (2.0 / _d, 20.0 / _d, 200.0 / _d)
    _orig_layer_step = mdl._fock_layer_step
    _vt_hooks = []

    _layer_grad = {}

    def _instrumented_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                  layer_idx, *args, **kwargs):
        out = _orig_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                layer_idx, *args, **kwargs)
        h_new = out[0]
        if torch.is_tensor(h_new) and h_new.requires_grad:
            def _layer_hook(g, li=layer_idx):
                # must not `return` the setdefault(...) result -- see
                # replay_precision_cap_ablation's twin hook for why.
                _layer_grad.setdefault(li, float(g.detach().norm()))
            h_new.register_hook(_layer_hook)
        return out

    _exp_min = {}

    def _make_vtheta_hook(idx):
        def _vt_fwd_hook(module, args, kwargs, output):
            xi_in = args[0] if len(args) > 0 else kwargs.get('xis', kwargs.get('xi'))
            h_in = args[1] if len(args) > 1 else kwargs.get('h')
            try:
                mu, a, w, B = module._components(xi_in)
                h_e = h_in.unsqueeze(-2)
                diff = h_e - mu
                diag_term = (a * diff * diff).sum(dim=-1)
                if B.shape[-1] > 0:
                    Bt_diff = torch.einsum('...kd,...kdr->...kr', diff, B)
                    lr_term = (Bt_diff * Bt_diff).sum(dim=-1)
                else:
                    lr_term = torch.zeros_like(diag_term)
                exponent = -0.5 * (diag_term + lr_term)
                _cur = _exp_min.get(idx, float('inf'))
                _exp_min[idx] = min(_cur, float(exponent.detach().min()))
            except Exception:
                pass
        return _vt_fwd_hook

    results = {}
    try:
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        mdl._fock_layer_step = _instrumented_layer_step
        for _idx, _bank in enumerate(_banks):
            _h_vt = _bank.register_forward_hook(_make_vtheta_hook(_idx), with_kwargs=True)
            _vt_hooks.append(_h_vt)
        grad_accum = bundle.get('grad_accum', len(bundle['batches']))
        mdl.train()

        for precision_max in precision_max_grid:
            for precision_lr_max in precision_lr_max_grid:
                _is_live_pm = math.isclose(precision_max, _live_precision_max,
                                            rel_tol=1e-6)
                _is_live_lr = precision_lr_max == globals().get('PRECISION_LR_MAX', None)
                label = (f'precision_max={precision_max:.5f} (={precision_max*_d:.0f}/d)'
                         f'  precision_lr_max={precision_lr_max}')
                if _is_live_pm and _is_live_lr:
                    label += ' [AS TRAINED]'
                for b in _banks:
                    b._precision_max = precision_max
                    b._precision_lr_max = precision_lr_max
                _layer_grad.clear()
                _exp_min.clear()
                for p in mdl.parameters():
                    p.grad = None
                torch.set_rng_state(bundle['rng_state_cpu'])
                if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
                    torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])

                # clip_then_sum + register-repulsion (SS48.2/48.3): see
                # replay_precision_cap_ablation for why this is mandatory,
                # not optional, on any bundle captured after clip_then_sum
                # went live (~step 52,550) -- built in from the start here
                # rather than retrofitted.
                _cts_params = _cts_group_params(mdl)
                _cts_running = {}
                for xb, yb in bundle['batches']:
                    x = torch.from_numpy(xb).to(DEVICE)
                    y = torch.from_numpy(yb).to(DEVICE)
                    loss, *_ = forward_with_vreg(
                        x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                    if REGISTER_REPULSION:
                        loss = loss + mdl.pop_repulsion_loss()
                    (loss / grad_accum).backward()
                    _cts_apply_microbatch(_cts_params, _cts_running)
                _cts_splice_back(_cts_params, _cts_running)

                _pg = per_group_grad_norms(mdl, _GRAD_CLIP_CFG)
                _total_sq_excl = sum(v * v for k, v in _pg.items()
                                       if k not in WATCHDOG_EXCLUDE_GROUPS)
                results[label] = {
                    'precision_max': precision_max,
                    'precision_lr_max': precision_lr_max,
                    'pre_clip_grad_norm': _total_sq_excl ** 0.5,
                    'replayed_group_norms': _pg,
                    'per_layer_h_grad': dict(sorted(_layer_grad.items())),
                    'vtheta_exponent_min': dict(sorted(_exp_min.items())),
                }
                if verbose:
                    _top = sorted(_pg.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
                    print(f'\n[rebal] {label}:')
                    print(f'    pre_clip_grad_norm='
                          f'{results[label]["pre_clip_grad_norm"]:.2f}')
                    print('    top groups: '
                          + ', '.join(f'{k}={v:.2f}' for k, v in _top))
                    print('    V_theta exponent min per bank: '
                          + ', '.join(f'bank{idx}={v:.1f}' for idx, v in
                                      results[label]['vtheta_exponent_min'].items()))
    finally:
        mdl._fock_layer_step = _orig_layer_step
        for _h in _vt_hooks:
            _h.remove()
        for b, lr_max, prec_max in zip(_banks, _saved_lr_max, _saved_prec_max):
            b._precision_lr_max = lr_max
            b._precision_max = prec_max
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    if verbose and results:
        _recorded = bundle['pre_clip_grad_norm']
        _base_label = next((lbl for lbl in results if lbl.endswith('[AS TRAINED]')),
                           None)
        _base = (results[_base_label]['pre_clip_grad_norm']
                 if _base_label is not None else None)
        print(f'\n[rebal] summary (step={bundle["step"]}, recorded pre-clip='
              f'{_recorded:.1f} at capture time):')
        print(f'    {"arm":<62} {"pre_clip":>10}  {"exp_min(bank0)":>15}')
        for label, r in results.items():
            _e0 = next(iter(r['vtheta_exponent_min'].values()), float('nan'))
            print(f'    {label:<62} {r["pre_clip_grad_norm"]:10.2f}  {_e0:15.1f}')
        if _base_label is None:
            print(f'    (no arm matched the live (precision_max, '
                  f'precision_lr_max) exactly -- include '
                  f'{_live_precision_max:.5f} and '
                  f'{globals().get("PRECISION_LR_MAX", None)!r} in the grids '
                  f'for a fidelity check)')
        elif _recorded:
            _fid = abs(_base - _recorded) / abs(_recorded) * 100.0
            print(f'    fidelity vs capture: {_fid:.1f}%  ({_base:.2f} '
                  f'replayed vs {_recorded:.2f} recorded)'
                  + ('' if _fid <= 5.0 else
                     '   <-- WARNING: >5%, do NOT read the arms until this '
                     'is explained (SS48.1 was exactly this signature)'))
    return results


def replay_clip_ablation(step_tag, groups=('E', 'P'),
                          thresholds=(1.0, 0.3, 0.1, 0.03), mdl=None,
                          verbose=True):
    """Companion note SS44 follow-up: test whether *when* per-group clipping
    is applied matters for `groups` (default the two largest, most
    consistent raw contributors across every capture to date -- SS44.2),
    not just *how tight* it is.

    Important correctness note this helper exists to make concrete:
    `clip_grads_per_group` (grad_clip_utils.py) is a pure post-hoc rescale
    of `.grad` -- `nn.utils.clip_grad_norm_` returns the norm computed
    BEFORE it rescales, so the value it returns (what the watchdog compares
    against `hard_trigger`, and what `replay_spike_batch` /
    `replay_precision_cap_ablation` report as "pre_clip_grad_norm") is
    IDENTICAL no matter what threshold you pass in. Unlike
    `precision_lr_max` (SS42), which changes the forward pass itself
    (`_bound_lowrank` reads `self._precision_lr_max` live), a clip
    threshold cannot be "ablated" by replaying the same accumulated
    gradient under different values -- `min(threshold, raw_norm)` is
    already exactly known once `raw_norm` is known, no replay needed.

    What genuinely differs, and what this helper actually measures, is
    WHERE in the accumulation the clip is applied:

    - `sum_then_clip` (the live training loop's current order, Cell 6):
      accumulate raw grads across all `GRAD_ACCUM` microbatches via
      successive `.backward()` calls, THEN clip the accumulated total
      once. One outlier microbatch/row can dominate that accumulated sum
      before the clip ever sees it (`attribute_spike_rows`, SS44.3, found
      39-72% of a spike's `total_grad_norm` sitting in 1-3 of 32 rows) --
      the clip can only shrink the *result* uniformly, it cannot stop the
      outlier from setting the update's *direction*.
    - `clip_then_sum` (the alternative being tested): clip each
      microbatch's own `groups` gradient to `threshold` BEFORE adding it
      into the running total. This bounds any one microbatch's
      contribution to the final update at the source, which
      `sum_then_clip` structurally cannot do regardless of how tight
      `threshold` is set.

    Both arms replay the SAME captured microbatches/RNG state (the
    non-pollution snapshot/restore invariant `replay_spike_batch` uses),
    so the only difference between them is clip order, not data.

    Returns {'sum_then_clip': {group: {threshold: applied_norm}},
             'clip_then_sum': {group: {threshold: applied_norm}},
             'cosine_vs_sum_then_clip': {group: {threshold: cosine}}}.
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'[clipabl] loaded {path.name}  step={bundle["step"]}  '
          f'pre_clip_grad_norm={bundle["pre_clip_grad_norm"]}')

    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None

    # -- resolve each requested group to its actual parameter list once,
    #    via the SAME assign_clip_group() the live training loop uses, so
    #    "group='E'" means exactly the parameters the watchdog/optimizer
    #    would put in clip-group 'E', not a name guess. --
    _group_params = {g: [] for g in groups}
    for n, p in mdl.named_parameters():
        key, _ = assign_clip_group(n, _GRAD_CLIP_CFG)
        if key in _group_params:
            _group_params[key].append(p)
    for g, ps in _group_params.items():
        if not ps:
            print(f"[clipabl][WARN] group '{g}' matched no parameters -- "
                  f"check the name against assign_clip_group()'s grouping "
                  f"(top-level attribute name, or an override substring).")

    grad_accum = bundle.get('grad_accum', len(bundle['batches']))
    sum_then_clip_raw = {g: None for g in groups}   # torch.Tensor, flat-concat
    clip_then_sum_acc = {g: {t: None for t in thresholds} for g in groups}

    def _flat_grad(ps):
        return torch.cat([p.grad.detach().reshape(-1) for p in ps
                           if p.grad is not None])

    try:
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        torch.set_rng_state(bundle['rng_state_cpu'])
        if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])
        mdl.train()

        for p in mdl.parameters():
            p.grad = None
        for xb, yb in bundle['batches']:
            x = torch.from_numpy(xb).to(DEVICE)
            y = torch.from_numpy(yb).to(DEVICE)
            for g, ps in _group_params.items():
                for p in ps:
                    p.grad = None
            loss, *_ = forward_with_vreg(
                x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
            (loss / grad_accum).backward()

            for g, ps in _group_params.items():
                if not ps:
                    continue
                g_flat = _flat_grad(ps)
                # -- sum_then_clip: just keep accumulating the raw grad,
                #    exactly like the live loop's successive .backward()
                #    calls into the same .grad tensor would. --
                sum_then_clip_raw[g] = (
                    g_flat if sum_then_clip_raw[g] is None
                    else sum_then_clip_raw[g] + g_flat)
                # -- clip_then_sum: clip THIS microbatch's contribution to
                #    `threshold` before adding it to that threshold's
                #    running total. --
                g_norm = float(g_flat.norm())
                for t in thresholds:
                    scale = min(1.0, t / max(g_norm, 1e-12))
                    contrib = g_flat * scale
                    clip_then_sum_acc[g][t] = (
                        contrib if clip_then_sum_acc[g][t] is None
                        else clip_then_sum_acc[g][t] + contrib)

        results = {'sum_then_clip': {}, 'clip_then_sum': {},
                   'cosine_vs_sum_then_clip': {}}
        for g in groups:
            if sum_then_clip_raw[g] is None:
                continue
            raw = sum_then_clip_raw[g]
            raw_norm = float(raw.norm())
            results['sum_then_clip'][g] = {
                t: min(t, raw_norm) for t in thresholds}
            results['clip_then_sum'][g] = {}
            results['cosine_vs_sum_then_clip'][g] = {}
            for t in thresholds:
                acc = clip_then_sum_acc[g][t]
                acc_norm = float(acc.norm())
                results['clip_then_sum'][g][t] = acc_norm
                # direction of the FINAL applied update in each regime:
                # sum_then_clip's applied grad is raw rescaled uniformly,
                # so its direction is just raw/raw_norm regardless of t.
                cos = float(torch.dot(raw, acc) / (raw_norm * acc_norm + 1e-12))
                results['cosine_vs_sum_then_clip'][g][t] = cos

        if verbose:
            print(f'\n[clipabl] step={bundle["step"]}  '
                  f'{len(bundle["batches"])} microbatch(es), '
                  f'grad_accum={grad_accum}')
            for g in groups:
                if g not in results['sum_then_clip']:
                    continue
                raw_norm = float(sum_then_clip_raw[g].norm())
                print(f"\n[clipabl] group '{g}'  (raw accumulated norm = "
                      f'{raw_norm:.2f}, {len(_group_params[g])} param(s))')
                print(f'    {"threshold":>10}  {"sum_then_clip":>14}  '
                      f'{"clip_then_sum":>14}  {"ratio":>8}  '
                      f'{"cos(applied dirs)":>18}')
                for t in thresholds:
                    stc = results['sum_then_clip'][g][t]
                    cts = results['clip_then_sum'][g][t]
                    cos = results['cosine_vs_sum_then_clip'][g][t]
                    ratio = cts / stc if stc else float('nan')
                    print(f'    {t:10.3g}  {stc:14.4f}  {cts:14.4f}  '
                          f'{ratio:8.3f}  {cos:18.4f}')
            print('\n[clipabl] "ratio" > 1 means clip_then_sum applies a '
                  '*larger* update than sum_then_clip at that threshold '
                  '(the outlier microbatch alone can already saturate '
                  'that threshold, so clipping earlier costs it nothing '
                  'extra); "ratio" < 1 means clip_then_sum is *more* '
                  'conservative (multiple microbatches each partially '
                  'saturate the threshold, so summing their capped '
                  'contributions still ends up smaller than capping the '
                  'sum once). "cos" close to 1.0 means clip order barely '
                  "changes the update's direction at that threshold "
                  '(clipping earlier vs. later mainly just rescales the '
                  'same direction); a noticeably lower cosine means the '
                  'two regimes disagree about which way the parameter '
                  'should actually move -- that would be the concrete '
                  'case for preferring clip_then_sum.')
    finally:
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    return results


def replay_integrator_ablation(step_tag, lowrank_layers=frozenset({0, 1, 2}),
                                mdl=None, top_k=8, verbose=True):
    """Companion note SS40.4: re-run a captured spike batch under two
    integrator configs -- the recorded 'baoab_cfc' and 'baoab_cfc_lowrank'
    restricted to `lowrank_layers` -- with weights, batch, and RNG state
    held bit-identical to the capture, to check whether the exact
    (SVD-based) low-rank substep suppresses the localized-mode spike that
    plain `baoab_cfc`'s approximate off-diagonal kick produces, BEFORE
    committing to a live layer-restricted trial.

    Sibling of `replay_precision_cap_ablation`: same snapshot/restore
    invariant, but swaps `mdl.cfg.integrator` / `mdl.cfg.lowrank_layers`
    across arms instead of `bank._precision_lr_max` --
    `model_parf_multixi.py` reads both as plain runtime branches
    (`use_cfc = cfg.integrator == 'baoab_cfc'`, `use_lowrank = cfg.integrator
    == 'baoab_cfc_lowrank'`), so this is a live-model attribute swap, not a
    new checkpoint -- the weights (B_proj, mu_proj, etc.) are identical
    either way.

    lowrank_layers : which layers run the exact low-rank substep in the
        second arm. Defaults to {0, 1, 2} per SS38.7's salience profile
        (the localized mode lives almost entirely in L0-2; SS34.3's cost
        floor does not amortize cleanly with layer count, so wider sets
        cost more than proportionally).
    mdl : the live `model` by default.

    Same non-pollution invariant as replay_spike_batch: weights, .grad
    tensors, RNG state, and cfg.integrator / cfg.lowrank_layers are
    snapshotted up front and restored in a `finally` block, so this is
    safe to call on the live model mid-training.

    Returns {arm_label: {integrator, lowrank_layers, pre_clip_grad_norm,
    replayed_group_norms, per_layer_h_grad}}.
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'[intgab] loaded {path.name}  step={bundle["step"]}  '
          f'pre_clip_grad_norm={bundle["pre_clip_grad_norm"]}')

    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
    _saved_integrator = mdl.cfg.integrator
    _saved_lowrank_layers = getattr(mdl.cfg, 'lowrank_layers', None)
    _orig_layer_step = mdl._fock_layer_step

    _layer_grad = {}

    def _instrumented_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                  layer_idx, *args, **kwargs):
        out = _orig_layer_step(h, h_prev, r, salience, m_b, gamma, dt,
                                layer_idx, *args, **kwargs)
        h_new = out[0]
        if torch.is_tensor(h_new) and h_new.requires_grad:
            def _layer_hook(g, li=layer_idx):
                # NOTE: see replay_precision_cap_ablation's twin hook --
                # must not `return` the setdefault(...) result or torch's
                # autograd engine raises "expected Variable, but hook
                # returned 'float'".
                _layer_grad.setdefault(li, float(g.detach().norm()))
            h_new.register_hook(_layer_hook)
        return out

    _arms = [
        ('baoab_cfc as captured', 'baoab_cfc', None),
        (f'baoab_cfc_lowrank layers {sorted(lowrank_layers)}',
         'baoab_cfc_lowrank', lowrank_layers),
    ]
    results = {}
    try:
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        mdl._fock_layer_step = _instrumented_layer_step
        grad_accum = bundle.get('grad_accum', len(bundle['batches']))
        mdl.train()

        for label, integrator, layers in _arms:
            mdl.cfg.integrator = integrator
            mdl.cfg.lowrank_layers = layers
            _layer_grad.clear()
            for p in mdl.parameters():
                p.grad = None
            torch.set_rng_state(bundle['rng_state_cpu'])
            if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
                torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])

            # 2026-09-10 (SS48.2/SS48.3, applied here belatedly): mirror the
            # live loop's clip_then_sum mechanics AND its register-repulsion
            # term. Both were spliced into replay_spike_batch but never into
            # this helper, so any bundle captured after clip_then_sum went
            # live (~step 52,550) replayed E/P raw, unclipped and fully
            # summed (~1,100+) -- large enough to swamp events that are
            # themselves only 100-500, and roughly arm-independent, so every
            # arm would look alike and the cap would look useless.
            # _cts_group_params returns {} when clip_then_sum isn't
            # configured, so this is a true no-op on pre-SS45.4 bundles
            # (47,116 / 48,507 / 48,917) and SS42's results stand.
            _cts_params = _cts_group_params(mdl)
            _cts_running = {}
            for xb, yb in bundle['batches']:
                x = torch.from_numpy(xb).to(DEVICE)
                y = torch.from_numpy(yb).to(DEVICE)
                loss, *_ = forward_with_vreg(
                    x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                if REGISTER_REPULSION:
                    loss = loss + mdl.pop_repulsion_loss()
                (loss / grad_accum).backward()
                _cts_apply_microbatch(_cts_params, _cts_running)
            _cts_splice_back(_cts_params, _cts_running)

            _pg = per_group_grad_norms(mdl, _GRAD_CLIP_CFG)
            _total_sq_excl = sum(v * v for k, v in _pg.items()
                                   if k not in WATCHDOG_EXCLUDE_GROUPS)
            results[label] = {
                'integrator': integrator,
                'lowrank_layers': layers,
                'pre_clip_grad_norm': _total_sq_excl ** 0.5,
                'replayed_group_norms': _pg,
                'per_layer_h_grad': dict(sorted(_layer_grad.items())),
            }
            if verbose:
                _top = sorted(_pg.items(), key=lambda kv: kv[1], reverse=True)[:top_k]
                print(f'\n[intgab] {label}:')
                print(f'    pre_clip_grad_norm={results[label]["pre_clip_grad_norm"]:.2f}')
                print('    top groups: '
                      + ', '.join(f'{k}={v:.2f}' for k, v in _top))
                print('    per-layer h-grad: '
                      + ', '.join(f'L{ell}={v:.4f}' for ell, v in
                                  results[label]['per_layer_h_grad'].items()))
    finally:
        mdl._fock_layer_step = _orig_layer_step
        mdl.cfg.integrator = _saved_integrator
        mdl.cfg.lowrank_layers = _saved_lowrank_layers
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    if verbose and len(results) == 2:
        _recorded = bundle['pre_clip_grad_norm']
        _labels = list(results.keys())
        _base = results[_labels[0]]['pre_clip_grad_norm']
        print(f'\n[intgab] summary (step={bundle["step"]}, recorded pre-clip='
              f'{_recorded:.1f} at capture time):')
        print(f'    {"arm":<42} {"pre_clip":>10}  {"vs baoab_cfc":>12}')
        for label, r in results.items():
            _ratio = f'{r["pre_clip_grad_norm"] / _base:.3f}x' if _base else 'n/a'
            print(f'    {label:<42} {r["pre_clip_grad_norm"]:10.2f}  {_ratio:>12}')
        if _recorded:
            # The 'as captured' arm re-runs the recorded configuration, so it
            # is this helper's fidelity check -- the role SS48.4's 0.0% table
            # plays for replay_spike_batch.
            _fid = abs(_base - _recorded) / abs(_recorded) * 100.0
            print(f'    fidelity vs capture: {_fid:.1f}%  ({_base:.2f} '
                  f'replayed vs {_recorded:.2f} recorded)'
                  + ('' if _fid <= 5.0 else
                     '   <-- WARNING: >5%, do NOT read the arms until this '
                     'is explained (SS48.1 was exactly this signature)'))
    return results


def replay_all_captures(mdl=None, top_k=5, verbose=False):
    """Replay every '_spikebatch.pt' currently on disk and print one
    summary attribution table across all of them.

    Companion note SS36: with CAPTURE_SPIKE_THRESHOLD (Cell 6) decoupled
    from the rare GRAD_NORM_HARD_TRIGGER, Phase 1 now harvests the far
    more frequent moderate-band spikes (the ones actually correlated with
    the run's convergence plateau, not just the occasional catastrophic
    reload). A consistent leading-group/per-layer signature across many
    of these is much stronger evidence than any single replay -- this is
    the batch version of calling replay_spike_batch(step_tag) by hand for
    every capture.

    Set verbose=True to also print each individual replay's full report
    (per-parameter / per-layer / activation-extreme detail); default only
    prints the summary table below.
    """
    mdl = mdl if mdl is not None else model
    _prefix = f'{CKPT_PREFIX}_step'
    _suffix = '_spikebatch.pt'
    paths = sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_spikebatch.pt'))
    if not paths:
        print('[replay-all] no captures found on disk.')
        return []
    reports = []
    for p in paths:
        step_tag = int(p.name[len(_prefix):-len(_suffix)])
        print(f'[replay-all] replaying step {step_tag} ({p.name}) ...')
        try:
            reports.append(replay_spike_batch(step_tag, mdl=mdl, top_k=top_k,
                                                verbose=verbose))
        except KeyboardInterrupt:
            # 2026-08-30 (companion note SS37): KeyboardInterrupt is a
            # BaseException, not an Exception, so the `except Exception`
            # below never caught it -- it used to propagate all the way to
            # IPython's top-level uncaught-exception handler, which pins
            # the interrupted replay's entire forward/backward graph via
            # sys.last_traceback (the same OOM failure mode as interrupting
            # Cell 6 itself, just triggered from in here instead). Catching
            # it here keeps that graph scoped to this frame, freed on
            # return, same fix as run_training() in Cell 6.
            print(f'\n[replay-all] interrupted while replaying step {step_tag}; '
                  f'stopping here (kept {len(reports)} report(s) already '
                  f'replayed). Run the GPU memory-cleanup cell before '
                  f'retrying, just in case.')
            break
        except Exception as e:
            print(f'[replay-all][WARN] failed on {p.name}: {e}')

    print(f'\n[replay-all] summary across {len(reports)} captured spikes '
          f'(sorted by step):')
    print(f'{"step":>8} {"pre_clip":>9} {"fidelity%":>10}  leader (norm)'
          f'  |  next-2 groups')
    for r in sorted(reports, key=lambda r: r['step']):
        _pg = sorted(r['replayed_group_norms'].items(), key=lambda kv: kv[1],
                      reverse=True)
        _leader = f'{_pg[0][0]}={_pg[0][1]:.1f}' if _pg else '(none)'
        _next2 = '  '.join(f'{k}={v:.1f}' for k, v in _pg[1:3])
        print(f'{r["step"]:8d} {r["pre_clip_grad_norm_recorded"]:9.1f} '
              f'{r["fidelity_gap_pct"]:10.4f}  {_leader:<28}  |  {_next2}')
    return reports


print('Cell 6d ready (helpers defined; call sites need Cell 6 to have at least')
print('started -- see the usage note atop this cell): e.g. replay_spike_batch(34091)')
print('              or replay_all_captures() to replay + summarize every capture on disk')
print('              or attribute_spike_rows(34091) for per-row depth_code attribution')
print('              or inspect_spike_tokens(34091) to decode the offending microbatch')
print('              (inspect_spike_tokens needs only Cell 3\'s `tok`, not Cell 6 at all)')
print('              or replay_precision_cap_ablation(34091, budgets=[1.0, 4.0, None])')
print('              to test precision_lr_max caps before enabling it live (SS41.6/41.7)')
print('              or replay_integrator_ablation(34091, lowrank_layers={0,1,2})')
print('              to test baoab_cfc vs baoab_cfc_lowrank on the same capture (SS40.4)')
print('Available captures on disk:')
for _p in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_spikebatch.pt')):
    print(f'  {_p.name}')


In [ ]:
# == Cell 6d-2: Spike inspection -- steps 70522/71194/71703 (SS48) =========
# ============================================================
# Spike inspection: steps 70522, 71194 (hard-triggers) and 71703
# (reverse_channel_scale-exceeds-total anomaly). Safe to run live --
# replay_spike_batch/attribute_spike_rows restore weights/.grad/RNG
# on exit, so no need to interrupt training first.
# ============================================================

STEPS_TO_INSPECT = {
    70522: 'hard-trigger #1 -- depth_code-dominant (887.9)',
    71194: 'hard-trigger #2 -- three-way register/creation_gate/reverse_channel_scale (691.0)',
    71703: 'reverse_channel_scale-exceeds-total anomaly, cleanest example (361.3 vs 233.1)',
}

reports = {}
for step, label in STEPS_TO_INSPECT.items():
    print(f'\n{"="*78}\n[inspect] step {step} -- {label}\n{"="*78}')
    reports[step] = replay_spike_batch(step)

# -----------------------------------------------------------------
# 1+2. Per-row attribution for the two hard-trigger events. Track
#      whichever parameters replay_spike_batch itself found on top
#      for THAT step -- no guessing at exact module/parameter paths
#      (e.g. creation_gate_qkv.* vs. register_embed vs. V_theta.depth_code).
# -----------------------------------------------------------------
row_reports = {}
for step in (70522, 71194):
    top_names = [n for n, _ in reports[step]['top_params'][:4]]
    print(f'\n{"="*78}\n[rows] step {step} -- tracking top-4 params: {top_names}\n{"="*78}')
    row_reports[step] = attribute_spike_rows(step, track=tuple(top_names))

# -----------------------------------------------------------------
# 3. reverse_channel_scale-exceeds-total anomaly at 71703: pull the
#    two totals replay_spike_batch already computed side by side.
#    (Cell 6's WATCHDOG_EXCLUDE_GROUPS comment, fixed 2026-09-09,
#    explains WHY these differ -- this just re-confirms the size of
#    the gap on the cleanest capture so far.)
# -----------------------------------------------------------------
r = reports[71703]
excl = r['pre_clip_grad_norm_replayed_matching']       # what the watchdog sees
incl = r['pre_clip_grad_norm_replayed_all_groups']     # incl. reverse_channel_scale/reverse_ch
rc_norm = r['replayed_group_norms'].get('override:reverse_channel_scale', float('nan'))
rch_norm = r['replayed_group_norms'].get('override:reverse_ch', float('nan'))
print(f'\n{"="*78}\n[anomaly] step 71703 -- WATCHDOG_EXCLUDE_GROUPS blind-spot size\n{"="*78}')
print(f'  watchdog-visible total (excl. reverse groups):     {excl:10.1f}')
print(f'  true total incl. reverse_channel_scale/reverse_ch: {incl:10.1f}')
print(f'  gap the watchdog never sees:                       {incl - excl:10.1f}')
print(f'  reverse_channel_scale group norm alone:             {rc_norm:10.1f}')
print(f'  reverse_ch group norm alone:                        {rch_norm:10.1f}')
print(f'  recorded (live, captured) pre_clip_grad_norm:       '
      f'{r["pre_clip_grad_norm_recorded"]:10.1f}')

# -----------------------------------------------------------------
# Quick cross-event concentration comparison (top-1 row / sum-of-rows):
# high top1_share = one row's tokens dominated the update direction;
# near-uniform = the localized-blowup conjecture (SS39) doesn't hold here.
# -----------------------------------------------------------------
print(f'\n{"="*78}\n[summary] per-row concentration, tracked params only\n{"="*78}')
for step in (70522, 71194):
    for n, s in row_reports[step]['summary'].items():
        if n in ('layer0_h_grad', 'total_grad_norm'):
            continue
        print(f'  step {step:6d}  {n:30s}  top1_share={s["top1_share"]:.4f}'
              f'  (uniform baseline {s["uniform_share_baseline"]:.4f})')

In [ ]:
# == Cell 6d-3: Hot-row token/element deconstruction (SS48 follow-up) =====
# ============================================================
# §48 follow-up: deconstructing the two new mechanisms.
#   (1) decode the rows that gradient attribution NAMED as hot
#   (2) per-element gradient map: log_tau (one entry per REGISTER)
#       and reverse_channel_scale (one entry per LAYER)
#   (3) creation-gate pre-softmax score magnitudes, per row
# Read-only: snapshots/restores weights, .grad and RNG.
# ============================================================
import io as _io
import contextlib as _ctx
import model_fock_parf_v2 as _mfv2

HOT_ROWS = {70522: [(3, 2)], 71194: [(2, 2), (1, 5)]}
PROBE_PARAMS = ('creation_gate_qkv.log_tau', 'reverse_channel_scale')

def decode_hot_rows(step_tag, hot_rows, snippet_chars=400):
    """(1) The §38.4/§39 degeneracy test, run in the correct direction:
    those ranked rows by repeat-run and hoped the culprit was among them.
    Here attribution already names the culprit, so we just read it -- and
    report where it RANKS on the degeneracy metrics, which is the actual
    falsifier.
    """
    buf = _io.StringIO()
    with _ctx.redirect_stdout(buf):          # it prints a 32-row table; not needed
        all_rows = inspect_spike_tokens(step_tag, show_n=0)
    n = len(all_rows)
    # inspect_spike_tokens returns rows already sorted by max_repeat_run desc.
    rank_repeat = {(r['microbatch'], r['row']): i for i, r in enumerate(all_rows)}
    rank_uniq = {(r['microbatch'], r['row']): i for i, r in enumerate(
        sorted(all_rows, key=lambda r: r['unique_token_ratio']))}
    med_uniq = sorted(r['unique_token_ratio'] for r in all_rows)[n // 2]
    med_rep = sorted(r['max_repeat_run'] for r in all_rows)[n // 2]
    print(f'\n[hot-tokens] step {step_tag}: {n} rows; batch medians: '
          f'unique_token_ratio={med_uniq:.3f}, max_repeat_run={med_rep}')
    for mb, row in hot_rows:
        r = next(x for x in all_rows
                 if x['microbatch'] == mb and x['row'] == row)
        print(f'\n  -- GRADIENT-HOT row mb={mb} row={row} --')
        print(f'     max_repeat_run={r["max_repeat_run"]:5d}   '
              f'rank {rank_repeat[(mb, row)] + 1}/{n} (1 = most repetitive)')
        print(f'     unique_token_ratio={r["unique_token_ratio"]:.3f}   '
              f'rank {rank_uniq[(mb, row)] + 1}/{n} (1 = most degenerate)')
        print('     text: '
              + tok.decode(r['ids'])[:snippet_chars].replace('\n', '\\n'))
    return all_rows

def probe_hot_rows(step_tag, hot_rows, mdl=None, track=PROBE_PARAMS,
                   verbose=True):
    """(2)+(3). `log_tau` is shape (M,) -- one entry per REGISTER -- and
    `reverse_channel_scale` is (n_gate,) -- one per LAYER. Their reported
    norms (345.4 / 361.3 at 71194) aggregate over those axes, which we
    have never looked inside. This dumps the per-element breakdown
    batch-wide and per row, alongside the creation gate's pre-softmax
    score magnitudes (captured by patching the module-level readout, so
    no Q/K math is duplicated here).
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'\n[probe] loaded {path.name}  step={bundle["step"]}  '
          f'pre_clip_grad_norm={bundle["pre_clip_grad_norm"]}')

    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None

    _name2param = dict(mdl.named_parameters())
    targets = [n for n in track if n in _name2param]
    missing = [n for n in track if n not in _name2param]
    if missing:
        print(f'[probe][WARN] not in named_parameters(), skipping: {missing}')

    # -- (3) score capture: patch the module-level readouts the creation
    #    gate calls, so we see the exact `scores` tensor it passes in
    #    (post-temperature-division, pre-softmax). The prefix-causal
    #    variant stabilises with a CONSTANT shift of 40.0 rather than a
    #    running max, so absolute score scale is meaningful -- reported
    #    below as a fraction of that constant.
    _READOUT_CLAMP = 40.0
    _score_calls = []

    def _make_patched(orig):
        def _patched(scores, V, *a, **kw):
            with torch.no_grad():
                s = scores.detach().float()
                _score_calls.append({'max_abs': float(s.abs().max()),
                                     'mean': float(s.mean()),
                                     'std': float(s.std())})
            return orig(scores, V, *a, **kw)
        return _patched

    _orig_pref = _mfv2._prefix_causal_creation_readout
    _orig_caus = _mfv2._causal_creation_readout
    _mfv2._prefix_causal_creation_readout = _make_patched(_orig_pref)
    _mfv2._causal_creation_readout = _make_patched(_orig_caus)

    def _reset_rng():
        torch.set_rng_state(bundle['rng_state_cpu'])
        if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])

    full_grads, row_recs = {}, []
    try:
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        mdl.train()
        grad_accum = bundle.get('grad_accum', len(bundle['batches']))

        # ---- full-batch pass (mirrors replay_spike_batch, incl. clip_then_sum)
        _reset_rng()
        for p in mdl.parameters():
            p.grad = None
        _cts_params = _cts_group_params(mdl)
        _cts_running = {}
        _score_calls.clear()
        for xb, yb in bundle['batches']:
            x = torch.from_numpy(xb).to(DEVICE)
            y = torch.from_numpy(yb).to(DEVICE)
            loss, *_ = forward_with_vreg(
                x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
            if REGISTER_REPULSION:
                loss = loss + mdl.pop_repulsion_loss()
            (loss / grad_accum).backward()
            _cts_apply_microbatch(_cts_params, _cts_running)
        _cts_splice_back(_cts_params, _cts_running)
        for n in targets:
            g = _name2param[n].grad
            full_grads[n] = (g.detach().float().reshape(-1).cpu().clone()
                             if g is not None else None)
        full_score_max = max((c['max_abs'] for c in _score_calls), default=float('nan'))

        # ---- per-row pass (mirrors attribute_spike_rows' isolation)
        for mb, (xb, yb) in enumerate(bundle['batches']):
            n_rows = xb.shape[0]
            for row in range(n_rows):
                for p in mdl.parameters():
                    p.grad = None
                _reset_rng()
                _score_calls.clear()
                x = torch.from_numpy(xb[row:row + 1]).to(DEVICE)
                y = torch.from_numpy(yb[row:row + 1]).to(DEVICE)
                loss, loss_ntp, *_ = forward_with_vreg(
                    x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                if REGISTER_REPULSION:
                    mdl.pop_repulsion_loss()      # drained, deliberately unused
                (loss / (grad_accum * n_rows)).backward()
                rec = {'microbatch': mb, 'row': row,
                       'ntp': float(loss_ntp.detach()),
                       'score_max_abs': max((c['max_abs'] for c in _score_calls),
                                            default=float('nan')),
                       'score_std_max': max((c['std'] for c in _score_calls),
                                            default=float('nan'))}
                for n in targets:
                    g = _name2param[n].grad
                    rec[n] = (g.detach().float().reshape(-1).cpu().clone()
                              if g is not None else None)
                row_recs.append(rec)
    finally:
        _mfv2._prefix_causal_creation_readout = _orig_pref
        _mfv2._causal_creation_readout = _orig_caus
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    if verbose:
        axis = {'creation_gate_qkv.log_tau': 'register',
                'reverse_channel_scale': 'layer'}
        for n in targets:
            g = full_grads[n]
            if g is None:
                continue
            tot_sq = float((g ** 2).sum()) or 1.0
            print(f'\n[probe] {n}: per-{axis.get(n, "element")} breakdown of the '
                  f'full-batch gradient (norm={tot_sq ** 0.5:.2f}, '
                  f'{g.numel()} entries)')
            print(f'    {"idx":>4}  {"grad":>12}  {"share_of_norm^2":>16}')
            for i in torch.argsort(g.abs(), descending=True).tolist():
                print(f'    {i:4d}  {float(g[i]):12.4f}  '
                      f'{float(g[i]) ** 2 / tot_sq:16.4f}')

        print(f'\n[probe] creation-gate score magnitude, full batch: '
              f'max|scaled score|={full_score_max:.2f}  '
              f'({full_score_max / _READOUT_CLAMP:.2f}x the readout\'s '
              f'constant shift of {_READOUT_CLAMP:g})')
        scores = sorted(r['score_max_abs'] for r in row_recs)
        n = len(scores)
        print(f'[probe] per-row max|scaled score|: median={scores[n // 2]:.2f}  '
              f'p90={scores[int(0.9 * n)]:.2f}  max={scores[-1]:.2f}')

        hot = set(hot_rows)
        print(f'\n[probe] gradient-hot rows vs. batch (the §48.8 test -- is the '
              f'hot row a SCORE outlier, or only a backward-signal outlier?):')
        for r in sorted(row_recs, key=lambda r: r['score_max_abs'], reverse=True):
            key = (r['microbatch'], r['row'])
            rank = 1 + sum(1 for x in row_recs
                           if x['score_max_abs'] > r['score_max_abs'])
            if key not in hot:
                continue
            print(f'    mb={key[0]} row={key[1]}  ntp={r["ntp"]:.3f}  '
                  f'max|scaled score|={r["score_max_abs"]:.2f}  '
                  f'(rank {rank}/{len(row_recs)}, 1 = largest in batch)')
            for nm in targets:
                g = r[nm]
                if g is None:
                    continue
                top = int(torch.argmax(g.abs()))
                print(f'        {nm}: peak {axis.get(nm, "element")} '
                      f'{top} (grad={float(g[top]):.4f}, '
                      f'row norm={float(g.norm()):.4f})')

    return {'step': bundle['step'], 'full_grads': full_grads,
            'rows': row_recs, 'full_score_max': full_score_max}

probe_reports = {}
for _step, _rows in HOT_ROWS.items():
    print(f'\n{"=" * 78}\n[deconstruct] step {_step}\n{"=" * 78}')
    decode_hot_rows(_step, _rows)
    probe_reports[_step] = probe_hot_rows(_step, _rows)

In [ ]:
# == Cell 6d-4: Gate saturation + log_tau history sweep (SS48.8) ==========
# ============================================================
# §48.8 follow-up: is the creation gate's score distribution
# sitting above the readout's hard clamp, and is register 14 the
# one that isn't?
#
#   _prefix_causal_creation_readout does:
#       s32 = scores.clamp(max=40.0) - 40.0
#   clamp(max=) passes ZERO gradient above the ceiling, so only
#   score entries BELOW 40 can reach log_tau / W_Q / W_K at all.
#
# PREDICTION (stated before running): register 14 should have the
# largest tau -- hence the smallest scaled scores, hence the most
# un-clamped mass -- and therefore be the only register still able
# to receive temperature gradient. If instead register 14's
# un-clamped fraction is unremarkable, the clamp story is wrong.
# ============================================================
import re as _re
import model_fock_parf_v2 as _mfv2

READOUT_CLAMP = 40.0
FOCUS_REGISTER = 14

def probe_gate_saturation(step_tag, mdl=None, clamp=READOUT_CLAMP,
                          focus=FOCUS_REGISTER, verbose=True):
    """Per-register x per-layer clamp saturation for the creation gate,
    plus per-register salience/active-fraction, from ONE replayed pass.

    Both are forward-side quantities, but this still runs the backward:
    a grad-enabled forward with no backward leaves the whole graph pinned
    (§43), which is exactly the eval-time OOM this notebook already fought.
    Same snapshot/restore invariant as every other Cell 6d helper.
    """
    mdl = mdl if mdl is not None else model
    path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
    bundle = torch.load(path, map_location='cpu', weights_only=False)
    print(f'\n[sat] loaded {path.name}  step={bundle["step"]}')

    _saved_grads = _isolated_grad_snapshot(mdl)
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _saved_rng_cpu = torch.get_rng_state()
    _saved_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
    _restore = []

    # Track which layer is executing, so each creation-gate call (the gate
    # is ONE shared module invoked once per layer) can be tagged -- same
    # pattern replay_spike_batch uses for its layer-resolved captures.
    _cur_layer = [-1]
    _orig_step = mdl._fock_layer_step
    _sal_by_layer, _act_by_layer = {}, {}

    def _instrumented_step(h, h_prev, r, salience, m_b, gamma, dt,
                           layer_idx, *a, **kw):
        _cur_layer[0] = layer_idx
        with torch.no_grad():
            # salience is (B, Tr, M) in prefix-causal mode, (B, M) otherwise;
            # reduce everything except the trailing register axis.
            for name, t, store in (('sal', salience, _sal_by_layer),
                                   ('act', m_b, _act_by_layer)):
                if torch.is_tensor(t) and t.shape[-1] == mdl.cfg.n_registers:
                    v = t.detach().float().reshape(-1, t.shape[-1]).mean(0).cpu()
                    store[layer_idx] = v if layer_idx not in store else (
                        store[layer_idx] + v) / 2
        return _orig_step(h, h_prev, r, salience, m_b, gamma, dt,
                          layer_idx, *a, **kw)

    mdl._fock_layer_step = _instrumented_step
    _restore.append(lambda: setattr(mdl, '_fock_layer_step', _orig_step))

    # Per-layer, per-register saturation counts.
    _n_ge, _n_tot, _smax = {}, {}, {}

    def _make_patched(orig):
        def _patched(scores, V, *a, **kw):
            with torch.no_grad():
                s = scores.detach().float()          # (B, M, T)
                ell = _cur_layer[0]
                ge = (s >= clamp).sum(dim=(0, 2)).cpu()          # (M,)
                tot = s.shape[0] * s.shape[2]
                mx = s.amax(dim=(0, 2)).cpu()                    # (M,)
                _n_ge[ell] = ge if ell not in _n_ge else _n_ge[ell] + ge
                _n_tot[ell] = tot if ell not in _n_tot else _n_tot[ell] + tot
                _smax[ell] = mx if ell not in _smax else torch.maximum(_smax[ell], mx)
            return orig(scores, V, *a, **kw)
        return _patched

    _orig_pref = _mfv2._prefix_causal_creation_readout
    _orig_caus = _mfv2._causal_creation_readout
    _mfv2._prefix_causal_creation_readout = _make_patched(_orig_pref)
    _mfv2._causal_creation_readout = _make_patched(_orig_caus)
    _restore.append(lambda: setattr(
        _mfv2, '_prefix_causal_creation_readout', _orig_pref))
    _restore.append(lambda: setattr(
        _mfv2, '_causal_creation_readout', _orig_caus))

    try:
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in bundle['model_state_dict'].items()},
            strict=False)
        torch.set_rng_state(bundle['rng_state_cpu'])
        if bundle.get('rng_state_cuda') is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(bundle['rng_state_cuda'])
        mdl.train()
        for p in mdl.parameters():
            p.grad = None
        grad_accum = bundle.get('grad_accum', len(bundle['batches']))
        xb, yb = bundle['batches'][-1]           # one microbatch is enough
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        loss, *_ = forward_with_vreg(x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
        if REGISTER_REPULSION:
            loss = loss + mdl.pop_repulsion_loss()
        (loss / grad_accum).backward()

        log_tau = dict(mdl.named_parameters()).get('creation_gate_qkv.log_tau')
        tau = (log_tau.detach().float().exp().clamp(min=1e-4).cpu()
               if log_tau is not None else None)
    finally:
        for fn in reversed(_restore):
            try:
                fn()
            except Exception:
                pass
        mdl.load_state_dict(_saved_sd)
        _isolated_grad_restore(mdl, _saved_grads)
        torch.set_rng_state(_saved_rng_cpu)
        if _saved_rng_cuda is not None and DEVICE == 'cuda':
            torch.cuda.set_rng_state_all(_saved_rng_cuda)

    layers = sorted(_n_ge)
    ge_all = torch.stack([_n_ge[l] for l in layers]).sum(0).float()
    tot_all = float(sum(_n_tot[l] for l in layers))
    frac_ge = ge_all / max(tot_all, 1.0)                  # per register
    unclamped = 1.0 - frac_ge

    if verbose:
        M = frac_ge.numel()
        print(f'[sat] clamp={clamp:g}; {len(layers)} layers x '
              f'{int(tot_all / max(len(layers), 1))} (B*T) entries per layer')
        print(f'[sat] batch-wide: {float(frac_ge.mean()) * 100:.2f}% of all '
              f'score entries are AT OR ABOVE the clamp (zero gradient path)')
        order = torch.argsort(unclamped, descending=True).tolist()
        print(f'\n[sat] per-register, ranked by UN-clamped mass '
              f'(the only mass that can carry gradient):')
        print(f'    {"reg":>4}  {"unclamped_frac":>15}  {"max_score":>11}  '
              f'{"tau":>9}  {"salience":>10}  {"active":>8}')
        smax_all = torch.stack([_smax[l] for l in layers]).amax(0)
        sal_all = (torch.stack([_sal_by_layer[l] for l in layers]).mean(0)
                   if _sal_by_layer else torch.full((M,), float('nan')))
        act_all = (torch.stack([_act_by_layer[l] for l in layers]).mean(0)
                   if _act_by_layer else torch.full((M,), float('nan')))
        for k in order[:8] + ([focus] if focus not in order[:8] else []):
            mark = '  <-- FOCUS' if k == focus else ''
            print(f'    {k:4d}  {float(unclamped[k]):15.6f}  '
                  f'{float(smax_all[k]):11.1f}  '
                  f'{float(tau[k]) if tau is not None else float("nan"):9.3f}  '
                  f'{float(sal_all[k]):10.4f}  {float(act_all[k]):8.4f}{mark}')
        rank = 1 + order.index(focus)
        print(f'\n[sat] register {focus}: un-clamped rank {rank}/{M}  '
              f'(1 = most un-clamped). PREDICTION was rank 1.')
        print(f'[sat] register {focus} tau={float(tau[focus]):.3f} vs '
              f'median tau={float(tau.median()):.3f}, max={float(tau.max()):.3f}')

        print(f'\n[sat] per-layer un-clamped fraction for register {focus} '
              f'(layer 0 is where reverse_channel_scale lives, §48.6):')
        for l in layers:
            f_l = 1.0 - float(_n_ge[l][focus]) / max(_n_tot[l], 1)
            print(f'    layer {l:2d}: {f_l:.6f}   (all-register mean '
                  f'{1.0 - float(_n_ge[l].sum()) / max(_n_tot[l] * len(_n_ge[l]), 1):.6f})')

    return {'step': bundle['step'], 'frac_ge_clamp': frac_ge,
            'unclamped': unclamped, 'tau': tau,
            'per_layer_ge': _n_ge, 'per_layer_tot': _n_tot,
            'salience': _sal_by_layer, 'active': _act_by_layer}

def sweep_log_tau_history(focus=FOCUS_REGISTER, extra_ckpts=(), verbose=True):
    """`log_tau` and register-embedding history for the focus register.

    Reads the `_spikebatch.pt` bundles already on disk rather than the
    real checkpoints: each bundle carries a full `model_state_dict` but no
    optimizer state, so this is far cheaper than a checkpoint sweep over
    Drive. Pass `extra_ckpts=[...]` (paths) to widen the span.
    """
    rows = []
    for p in sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_spikebatch.pt')):
        m = _re.search(r'_step(\d+)_spikebatch\.pt$', p.name)
        if m:
            rows.append((int(m.group(1)), p, 'spikebatch'))
    for p in extra_ckpts:
        p = Path(p)
        m = _re.search(r'_step(\d+)', p.name)
        rows.append((int(m.group(1)) if m else -1, p, 'checkpoint'))
    rows.sort()

    out = []
    for step, p, kind in rows:
        try:
            b = torch.load(p, map_location='cpu', weights_only=False)
            sd = b.get('model_state_dict', b)
            lt = sd.get('creation_gate_qkv.log_tau')
            re_emb = sd.get('register_embed')
            if lt is None:
                continue
            lt = lt.float()
            tau = lt.exp().clamp(min=1e-4)
            rec = {'step': step, 'kind': kind,
                   'log_tau_focus': float(lt[focus]),
                   'tau_focus': float(tau[focus]),
                   'tau_median': float(tau.median()),
                   'tau_max': float(tau.max()),
                   'tau_focus_rank': 1 + int((tau > tau[focus]).sum()),
                   'emb_focus_norm': (float(re_emb[focus].float().norm())
                                       if re_emb is not None else float('nan')),
                   'emb_median_norm': (float(re_emb.float().norm(dim=-1).median())
                                        if re_emb is not None else float('nan'))}
            out.append(rec)
            del b, sd
        except Exception as e:
            print(f'[hist][WARN] {p.name}: {e}')

    if verbose and out:
        print(f'\n[hist] register {focus} across {len(out)} snapshot(s):')
        print(f'    {"step":>7}  {"kind":>11}  {"tau[k]":>9}  {"rank":>5}  '
              f'{"tau_med":>9}  {"tau_max":>9}  {"|emb[k]|":>9}  {"|emb|_med":>10}')
        for r in out:
            print(f'    {r["step"]:7d}  {r["kind"]:>11}  {r["tau_focus"]:9.4f}  '
                  f'{r["tau_focus_rank"]:5d}  {r["tau_median"]:9.4f}  '
                  f'{r["tau_max"]:9.4f}  {r["emb_focus_norm"]:9.4f}  '
                  f'{r["emb_median_norm"]:10.4f}')
    return out

sat_reports = {s: probe_gate_saturation(s) for s in (70522, 71194)}
tau_history = sweep_log_tau_history()

In [ ]:
# == Cell 6: Training loop =============================================

LR            = 3e-4
WEIGHT_DECAY  = 0.01
# 2026-09-09 (companion note SS51): exclude 1-D parameters (biases, norm
# gains, and every scale/temperature-like scalar) from weight decay --
# the standard practice this notebook had simply never applied, because
# AdamW was handed a flat parameter list. See split_decay_params() for
# the measurement that made this urgent: decoupled decay on the
# LOG-parameterised creation temperature pulls tau toward 1 from an init
# of 8.0, and explains ~70% of the drift SS49.5 attributed to a
# self-reinforcing runaway.
# Changing this splits AdamW into two param groups, which re-indexes the
# optimizer state; load_optim_state() below remaps pre-SS51 checkpoints
# so Adam moments survive the change. Set False to restore old behaviour.
NO_DECAY_1D   = True
WARMUP_STEPS  = int(WSD_WARMUP_FRAC * TOTAL_STEPS) if LR_SCHEDULE == 'wsd' else 4000
GRAD_CLIP     = 1.0
GRAD_CLIP_VPHI = 0.3

PER_GROUP_CLIP = True
GRAD_CLIP_OVERRIDES = {
    'V_phi': GRAD_CLIP_VPHI,
    # 2026-09-09 (companion note SS49.8): log_tau split out of the
    # 'creation_gate' group. This entry MUST stay ABOVE 'creation_gate':
    # assign_clip_group returns the FIRST substring hit while iterating
    # this dict, and 'creation_gate_qkv.log_tau' matches both keys, so
    # placing it any lower would make it dead config that silently
    # never fires.
    # Why split: clip_grads_per_group clips each group JOINTLY. At step
    # 71,194 log_tau alone was 345.38 of the creation_gate group's 432.16
    # norm, so the rescale factor 0.3/432.16 was applied to W_Q/W_K/W_V
    # too -- their own contribution was sqrt(432.16^2 - 345.38^2) = 259.8,
    # so they received an effective 0.180 instead of the 0.300 they would
    # have got on their own (~60%). Negligible at step 70,522 (log_tau
    # only 1.65 of 429.52), i.e. this is an emerging tax that grows with
    # the SS49.5 runaway, and it is collateral damage to parameters that
    # are not themselves misbehaving.
    # NOT a fix for the runaway itself: Adam is close to scale-invariant
    # per parameter in steady state, so clipping log_tau's gradient
    # changes its own step size far less than the factor suggests.
    # Halting the drift needs the forward-side change in SS49.9. What
    # this does buy: the projections stop being starved, and
    # override:log_tau becomes a first-class line in the per-group log
    # and in every future capture's top_groups.
    'log_tau': 0.3,
    # 2026-09-10: the qk_norm analogue of the 'log_tau' split above, and
    # it needs the SAME ordering care -- it must precede 'creation_gate'
    # because 'creation_gate_qkv.logit_scale' matches both keys and
    # assign_clip_group returns the FIRST substring hit in dict order.
    # Spelled out in full rather than as a bare 'logit_scale' key on
    # purpose: ReverseChannel ALSO has a `logit_scale` parameter, and a
    # bare key placed here would capture it too and silently move it out
    # of the 'reverse_ch' group (0.1) it belongs to.
    'creation_gate_qkv.logit_scale': 0.3,
    'creation_gate': 0.3,
    'destruction_gate': 0.3,
    'reverse_channel_scale': 0.1,
    'reverse_ch': 0.1,
    'register': 0.3,
    # 2026-08-23: tightened 0.5 -> 0.25. depth_code was already saturating
    # its old ceiling on *every* quiet step (top[override:depth_code] ~
    # 0.5-1.6) and was the single largest pre-clip contributor (1592.8 at
    # the worst step-6435 spike) in the first CfC/BAOAB grad-clip burst
    # seen on the g0.1 OWT run (steps 6297-6616, val_ppl 176.88->207.11).
    # Halving its ceiling costs little in the already-saturated quiet
    # regime and caps its burst contribution proportionally.
    'depth_code': 0.25,
}
# NOTE: excluding reverse_channel_scale/reverse_ch keeps the watchdog EMA
# from false-triggering on their own normal warmup ramp, but it also means
# BOTH watchdog layers below are structurally blind to them -- they showed
# up in the top-4 of nearly every spike in the 2026-08-23 burst (e.g.
# reverse_channel_scale=403.8, reverse_ch=363.1 at step 6435).
# 2026-09-09 correction: this comment used to end with "...GRAD_NORM_HARD_
# TRIGGER below is a fallback that does see them" -- that was wrong.
# GRAD_NORM_HARD_TRIGGER checks `_raw_gn = float(grad_norm)`, the exact
# same excluded aggregate the EMA uses (both trace back to
# clip_grads_per_group's `total_sq`, which skips any group listed here) --
# confirmed live: at steps 70,660 and 71,703 the printed
# reverse_channel_scale per-group norm (242.1 / 361.3) was LARGER than the
# printed pre-clip total grad (118.8 / 233.1) for that same step, because
# the "total" is summed over every OTHER group only. So an isolated
# reverse_channel_scale/reverse_ch spike, however large, cannot fire the
# EMA soft-trigger or the GRAD_NORM_HARD_TRIGGER hard reload -- only
# CAPTURE_SPIKE_THRESHOLD's bundle-capture check (same excluded aggregate)
# happens to still catch it, and only when some other group in the same
# step also clears that threshold. Left as-is for now (per-group clipping
# still bounds the actual applied update to these params regardless of
# whether the watchdog aggregate sees them) -- this is a monitoring-
# coverage gap, not a confirmed training-risk gap. replay_spike_batch()
# reports both totals side by side (`pre_clip_grad_norm_replayed_matching`
# = excl., `pre_clip_grad_norm_replayed_all_groups` = incl.) for exactly
# this reason -- use it to see the size of the blind spot on any capture.
WATCHDOG_EXCLUDE_GROUPS = {'override:reverse_channel_scale', 'override:reverse_ch'}

# 2026-09-09 (companion notes SS49.5/SS50): floor on the creation gate's
# per-register temperature, enforced as a PROJECTION after optim.step()
# rather than a clamp inside forward(). Two reasons for the projection
# form: (a) it is not a forward change at all, so it is resume-safe on the
# live 72k-step checkpoint and needs no fresh arm; (b) a forward clamp
# would zero the gradient at the boundary -- the exact gradient-killing
# pattern the readout's own clamp(max=40.0) has (SS49.2) -- whereas
# projected gradient descent lets the gradient flow normally and simply
# pushes the parameter back into the box.
# Why a floor at all: SS50.3 shows the inverse temperature v = 1/tau obeys
# v_dot = eta*C*v^2 under gradient flow in the diffuse regime -- a Riccati
# equation with FINITE-TIME blow-up, not merely exponential growth. The
# measured drift (tau[14] 5.3409 -> 5.2104 over steps 68,313-71,985, rank
# 30 -> 32 of 32, both sampled gradients positive) is the early, slow part
# of that trajectory.
# tau was initialised at 8.0 for all 32 registers; the pool median is now
# 6.45 and register 14 sits at 5.21. A floor of 4.0 is therefore 23% below
# today's minimum -- a no-op at present values, roughly 20k steps of
# headroom at the observed drift rate, and a hard bound on how far the
# tau channel can amplify scores (SS50.5: it bounds ONE of the two
# multiplicative channels; the ||Q||*||K|| channel needs qk_norm, which is
# fresh-arm-only).
# Set to None to disable.
TAU_CREATE_MIN = 4.0

GRAD_SPIKE_DEBUG     = True
GRAD_SPIKE_THRESHOLD = 100.0
GRAD_SPIKE_COOLDOWN  = 0
EVAL_INTERVAL = 500
EVAL_ITERS    = 40
LOG_INTERVAL  = 50
CAUSAL_PROBE_INTERVAL = 10_000
TRAINED_LEAK_PROBE_INTERVAL = 10_000
TRAINED_LEAK_PROBE_K = 256
TRAINED_LEAK_PROBE_PAIRS = 2

# 2026-09-07 (companion note SS46): Colab enforces a hard ~24h runtime
# lifetime cap independent of anything this notebook does -- when it hits,
# the kernel is torn down with zero warning, mid-step, no chance to save.
# Checkpoints otherwise only happen on val-improvement or every
# EVAL_INTERVAL steps' periodic save, which can leave a multi-thousand-step
# gap if the wall-clock cutoff lands between saves (the run this was born
# from lost 4,252 steps -- interrupted at 56,752, last verified-good
# checkpoint at 52,500 -- to exactly that gap, compounded by an unrelated
# CUDA-session corruption that also silently truncated the one manual
# checkpoint taken mid-incident). This is a pure wall-clock safety net,
# independent of step count and of how many times run_training() has been
# (re)called this session: fires at most once per process, as soon as VM
# uptime crosses AUTOSAVE_WALLCLOCK_HOURS. Set to None to disable.
AUTOSAVE_WALLCLOCK_HOURS = 23.5
_autosave_ckpt_done_this_process = False


def _vm_uptime_seconds():
    """Seconds since this VM booted -- NOT since this kernel/notebook
    started, and NOT since run_training() was last (re)called (`t0`
    inside run_training resets on every call, so it can't see across an
    interrupt-and-resume within the same still-alive session). Read fresh
    from the kernel every call rather than cached, so it stays correct no
    matter how the training loop gets interrupted/resumed around it."""
    try:
        with open('/proc/uptime') as _f:
            return float(_f.read().split()[0])
    except Exception:
        return 0.0

if WSD_LR_FLOOR is None:
    WSD_LR_FLOOR = LR * 0.05

GRAD_NORM_EMA_ALPHA = 0.05
GRAD_NORM_EMA_THRESHOLD = 50.0
GRAD_NORM_EMA_PATIENCE = 200
# Fast, EMA-independent safety net added 2026-08-23. With alpha=0.05 and
# patience=200, an isolated spike decays back out of the EMA within ~15-20
# steps and never accumulates 200 *consecutive* above-threshold steps, so
# a short burst of huge-but-brief spikes (pre-clip total=2401.6 at step
# 6435, =1815.9 at step 6407, ...) can do lasting damage (val_ppl
# 176.88->207.11 over the 6000->6500 eval window) without ever tripping
# the patience-gated reload. This reloads best immediately on any single
# step whose raw (pre-EMA) grad_norm exceeds it, regardless of EMA/patience
# state. Set to None to disable.
# 2026-08-30 Stage-1 smoke test value, used once to validate Phase 1/2
# end-to-end (see companion note SS35) -- restored to the production
# value below afterward. Left here for reference, not active.
# GRAD_NORM_HARD_TRIGGER = 110.0
GRAD_NORM_HARD_TRIGGER = 500.0


# How many '_prereload' snapshots (the about-to-be-discarded state AT the
# moment the watchdog fires, before it is overwritten by the last best)
# to keep on disk at once; 0 disables the feature entirely. A dense
# reload cluster can produce a lot of these in a short span, so this
# rotates rather than growing unbounded -- see docs/Example_Stiffness_
# Audit_OWT_g0.1_Anisotropic_Gaussian.md sec 9 in the SCAF repo for why
# this exists: without it, a run whose best_ppl stops improving for
# thousands of steps during a bad cluster leaves NO checkpoint at all in
# that window for a later stiffness audit to look at, because both of
# this file's other save triggers (a new best; the CKPT_INTERVAL grid)
# can go arbitrarily long without firing precisely while the cluster is
# active. This one is unconditional on val_ppl, so it cannot have that
# failure mode.
PRERELOAD_SNAPSHOT_MAX_KEEP = 5

# Root-cause diagnostic Phase 1 (CfC/BAOAB companion note SS33.3): alongside
# the '_prereload' snapshot, also capture the PRE-optim.step() weights, the
# exact microbatch(es), and the torch RNG state for any step whose pre-clip
# grad norm crosses CAPTURE_SPIKE_THRESHOLD, so the offending forward+
# backward can be replayed deterministically in isolation (Phase 2).
#
# 2026-08-30 (companion note SS36): CAPTURE_SPIKE_THRESHOLD is deliberately
# a SEPARATE, lower knob from GRAD_NORM_HARD_TRIGGER below, not the same
# value -- the two answer different questions. GRAD_NORM_HARD_TRIGGER is
# 'how bad does a step have to be before we discard progress and reload
# the best checkpoint' (rare, >=500 in this run's history); the plateau
# analysis in SS36 showed the run can sit stuck for 1,000+ steps with
# ZERO hard triggers, driven instead by the far more frequent 100-450
# 'grad_spike' band (GRAD_SPIKE_THRESHOLD above), which is the actual
# convergence question. Gating capture on GRAD_NORM_HARD_TRIGGER meant
# Phase 1 could only ever harvest the rare, possibly-unrepresentative
# catastrophic events; gating it on this separate, lower threshold
# instead harvests the frequent moderate band WITHOUT touching the
# reload behaviour at all (CAPTURE_SPIKE_THRESHOLD plays no role in the
# watchdog logic below, so lowering it cannot cause extra reloads).
#
# 2026-09-06 (companion note SS42): lowered 200.0 -> 100.0 (matching
# GRAD_SPIKE_THRESHOLD, so every printed '[spike]' line also gets a
# capturable bundle) now that PRECISION_LR_MAX=1.0 has suppressed the
# chronic V_theta/low-rank mechanism (SS41/SS42): the interesting
# population worth harvesting moved down into the 100-200 band, e.g. the
# first post-cap event led by 'register' rather than 'depth_code' or
# 'reverse_channel_scale' (step 47,142, pre-clip 147.9 -- under the old
# 200.0 floor, so it printed but was never saved). An OOM at step ~47,530
# the same session was investigated and is NOT attributed to this: the
# capture code path only runs when a step actually crosses the threshold,
# and the log in the steps immediately before that crash shows no
# '[spike]'/'[spike-capture]' line at all, so the extra state_dict copy
# never fired in that window -- steady-state memory was already at
# mem_resv=57.6GB/79.25GB (73%) beforehand, which is the more likely
# explanation. Revisit upward if a genuine capture-linked OOM shows up.
CAPTURE_SPIKE_THRESHOLD = 100.0
# Only the hard trigger and this threshold have a single well-defined
# 'offending batch' to snapshot; the slow EMA trigger is a drift, not a
# one-step event, and stays a Phase-0 log-mining question. Same rotation
# policy as PRERELOAD_SNAPSHOT_MAX_KEEP; 0/False disables.
CAPTURE_SPIKE_BATCH = True
# Raised 5 -> 12 (SS36): with capture now firing on the frequent moderate
# band rather than only the rare hard trigger, a deeper rotation keeps a
# small *library* of captures on disk to replay across (see the new
# replay_all_captures() in Cell 6d) instead of just the single latest one.
SPIKEBATCH_SNAPSHOT_MAX_KEEP = 12

# Permanent archive config (see _archive_bundle below), separate from the
# live ring above. 2026-09-11: unconditionally archiving every capture
# forever does not scale to the planned 150,000-step continuation --
# observed rates project to ~362 spikebatch captures (1 per ~414 steps)
# and ~71 hard-trigger prereload events (1 per ~2,115 steps) over that
# span, and spikebatch bundles are ~300MB / prereload snapshots ~900MB
# (full weights + optimizer state), i.e. ~173GB unconditionally. Most
# spikebatch captures are also the "routine, self-limiting" smooth-
# cascade events the diagnostic programme's own taxonomy (companion note
# SS8) already reads as not individually interesting -- worth the write
# for offline replay, not worth permanent disk.
# So: spikebatch is gated by severity AND capped by count; prereload
# (rarer, and every one already implies a genuine watchdog reload, so
# always meaningful) is capped by count only. ARCHIVE_MIN_GRAD_NORM=200
# (2x CAPTURE_SPIKE_THRESHOLD) clears the ~100-190 routine band while
# keeping moderately severe events -- e.g. the 220.5/269.6 pair already
# used for the rank-spectrum analysis (Cell 6b-4) would still qualify.
# At these defaults the worst case over the RUN'S ENTIRE LIFE (not per
# session) is 60*300MB + 40*900MB ~= 54GB. Given the projected ~362/~71
# event counts, both caps are expected to start evicting (oldest first)
# before the continuation ends -- that is the bound doing its job, not a
# bug; lower either number if your Drive quota is tighter.
ARCHIVE_MIN_GRAD_NORM       = 2 * CAPTURE_SPIKE_THRESHOLD  # = 200.0
ARCHIVE_SPIKEBATCH_MAX_KEEP = 60
ARCHIVE_PRERELOAD_MAX_KEEP  = 40

# 2026-09-06: two back-to-back OOMs (companion note SS42 follow-up) both
# happened ~400-450 iterations after a FRESH resume from the same
# step-47,116 checkpoint -- one with a spike-capture event in that window,
# one without -- ruling out spike-capture as the cause and pointing
# instead at ordinary CUDA-allocator warm-up creep: right after a fresh
# model/optimizer build, the caching allocator's reserved pool tends to
# grow for its first few hundred iterations as it meets the real
# distribution of tensor shapes, before plateauing. Steady-state here is
# already mem_resv=57.6GB/79.25GB (73%), leaving only ~22GB of margin for
# that creep -- not enough, twice in a row. Proactively releasing unused
# cached blocks (torch.cuda.empty_cache() does not free anything still
# referenced, only hands idle cached blocks back to the driver) during
# just that early window costs a small sync every CUDA_EMPTY_CACHE_INTERVAL
# steps but gives the allocator repeated chances to coalesce/shrink
# instead of only ever growing. Relative to `start_step` (the argument
# run_training() was actually called with), not the absolute step number,
# so it re-arms on every fresh resume, not just the very first one ever.
# 0/None disables.
CUDA_EMPTY_CACHE_WARMUP_STEPS = 1000
CUDA_EMPTY_CACHE_INTERVAL = 50

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)


def lr_schedule(step):
    if LR_SCHEDULE == 'wsd':
        warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
        stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
        if step < warmup_end:
            return LR * (step + 1) / max(warmup_end, 1)
        elif step < stable_end:
            return LR
        else:
            decay_steps = TOTAL_STEPS - stable_end
            progress = (step - stable_end) / max(decay_steps, 1)
            cos_decay = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
            return WSD_LR_FLOOR + (LR - WSD_LR_FLOOR) * cos_decay
    else:
        if step < WARMUP_STEPS:
            return LR * (step + 1) / WARMUP_STEPS
        progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
        return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def fock_coupling_reg(mdl, lam, eps):
    alphas = mdl.xi_module.alpha
    return -lam * torch.log(alphas + eps).sum()


def forward_with_vreg(x, targets, lambda_v, lambda_fock, fock_eps):
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    logits = model.compute_logits(h_L)
    loss_ntp = F.cross_entropy(
        logits.reshape(-1, model_cfg.vocab_size),
        targets.reshape(-1),
    )
    v_reg_value = torch.tensor(0.0, device=x.device)
    fock_reg_value = torch.tensor(0.0, device=x.device)
    loss = loss_ntp

    if lambda_v > 0:
        xis = model.xi_module(h_L.detach())
        V_vals = model.V_theta(xis, h_L)
        v_reg_value = (V_vals.float() ** 2).mean()
        loss = loss + lambda_v * v_reg_value

    if lambda_fock > 0:
        fock_reg_value = fock_coupling_reg(model, lambda_fock, fock_eps)
        loss = loss + fock_reg_value

    return loss, loss_ntp, v_reg_value, fock_reg_value


@torch.no_grad()
def evaluate():
    model.eval()
    # 2026-08-30 (companion note SS37): try/finally so an interrupt landing
    # mid-eval (e.g. while inspecting spike captures) can't leave `model`
    # stuck in .eval() mode for the run_training() call that resumes after it.
    try:
        losses = []
        _mem_pre_gb = torch.cuda.memory_allocated() / 1e9
        torch.cuda.reset_peak_memory_stats()
        _mem_per_iter = []
        for _i in range(EVAL_ITERS):
            xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
            x = torch.from_numpy(xb).to(DEVICE)
            y = torch.from_numpy(yb).to(DEVICE)
            # 2026-09-06 (companion note SS43): the gc.collect()-every-iter
            # fix (SS42 follow-up, below) was falsified empirically -- two
            # more OOMs at the *identical* mem_alloc=47.47GB at iter 0,
            # bit-for-bit unchanged by adding gc.collect(). That refuted
            # the "uncollected create_graph=True reference cycle" theory
            # behind it: grep confirms MultiXiPARFLM._layer_forces gates
            # the only torch.autograd.grad() call in the analytic-V_theta
            # path with create_graph=self.training / retain_graph=
            # self.training, and self.training is False here (model.eval()
            # above) -- so that call is create_graph=False and produces a
            # plain detached grad_phi, exactly as the code's own docstring
            # says. There is no hidden create_graph=True anywhere in this
            # path; gc.collect() had nothing collectible to find, which is
            # exactly why the number never moved.
            #
            # The real graph that survives eval is the *analytic V_theta
            # force* itself: f_theta = -V_theta.analytical_grad(xis, h_in)
            # is ordinary differentiable tensor arithmetic (matmuls against
            # V_theta's own parameters), not a discrete autograd.grad()
            # call -- so it is NOT gated by create_graph/retain_graph at
            # all and stays connected to h_in regardless of train/eval.
            # Chained across L checkpointed layers this reaches the same
            # ~50-57GB peak the training loop's own mem_peak print already
            # shows for one forward+backward microbatch (batch=8) -- training
            # only survives it because loss.backward() immediately follows
            # and lets use_layer_checkpoint's non-reentrant recompute-then-
            # free machinery actually run. evaluate() never calls
            # .backward(), so that same per-iteration peak is simply never
            # reclaimed -- no Python object is leaked, so del + gc.collect()
            # correctly find nothing to do, which is the direct explanation
            # for why the previous fix made no difference.
            #
            # Fix: give eval a real (but weight-inert) backward() so the
            # checkpoint machinery frees each layer's segment exactly as it
            # does in training. self.training stays False (model.eval()),
            # so create_graph/retain_graph for grad_phi are still False and
            # nothing here is more expensive than a normal training step's
            # forward+backward at the same batch size. The resulting
            # .grad tensors are discarded immediately -- optimizer.step()
            # is never called on them, so this cannot corrupt training.
            with torch.enable_grad():
                _, loss = model(x, y)
                losses.append(loss.item())
                loss.backward()
            model.zero_grad(set_to_none=True)
            del loss, x, y
            gc.collect()
            torch.cuda.empty_cache()
            if _i % 10 == 0 or _i == EVAL_ITERS - 1:
                _mem_now_gb = torch.cuda.memory_allocated() / 1e9
                _mem_per_iter.append(_mem_now_gb)
                # Printed live (not just in the end-of-function summary
                # below) so the trend is visible in the log even if this
                # call still OOMs before reaching that summary.
                print(f'    [evaluate] iter {_i}/{EVAL_ITERS}  '
                      f'mem_alloc={_mem_now_gb:.2f}GB  '
                      f'mem_resv={torch.cuda.memory_reserved()/1e9:.2f}GB')
        _mem_post_gb = torch.cuda.memory_allocated() / 1e9
        _mem_peak_gb = torch.cuda.max_memory_allocated() / 1e9
        _mem_trend = ', '.join(f'{m:.2f}' for m in _mem_per_iter)
        print(f'    [evaluate] mem before={_mem_pre_gb:.2f}GB  '
              f'after={_mem_post_gb:.2f}GB  peak_during={_mem_peak_gb:.2f}GB  '
              f'trend(every 10 iters)=[{_mem_trend}]GB')
    finally:
        model.train()
    return float(np.mean(losses))


def run_causal_probe(step_num):
    import math as _math
    # Use the same V_theta family and the same integrator as the run, so
    # this probe certifies prefix-causality for what is actually training.
    from model_aniso_gaussian_vtheta import (
        AnisotropicDepthConditionedGaussianVTheta as _DCMCGVT,
        install_aniso_depth_routing as _idr,
    )
    _PROBE_VOCAB, _PROBE_D, _PROBE_L = 101, 32, 4
    _PROBE_T, _PROBE_M, _PROBE_XI = 48, 8, 3
    _PROBE_WELLS = 4

    _logfreq_probe = Path('/tmp/causal_probe_logfreq.npy')
    np.save(_logfreq_probe, np.full(_PROBE_VOCAB, 5.0, dtype=np.float32))

    _probe_cfg = FockMultiXiPARFConfig(
        vocab_size=_PROBE_VOCAB, d=_PROBE_D, max_len=64, L=_PROBE_L,
        v_hidden=64, v_depth=3, dt=1.0,
        mass_mode='logfreq', logfreq_path=str(_logfreq_probe),
        logfreq_init_alpha=0.1, init_gamma=1.0, fixed_gamma=0.30,
        causal_force=True, ln_after_step=True,
        xi_channels=_PROBE_XI, xi_alpha_inits=[0.5, 0.9, 0.99],
        xi_learnable=True, xi_alpha_init_mode='explicit',
        v_phi_kind='structural_competitive',
        v_phi_d_type=8, v_phi_d_angle=4, v_phi_eps=0.1,
        v_phi_phi_hidden=16, v_phi_theta_hidden=16, v_phi_mlp_hidden=16,
        top_k=8, v_phi_n_heads=2,
        use_output_bias=True, tie_embeddings=False,
        score_head_hidden=8,
        gumbel_tau_init=1.0, gumbel_tau_min=0.3, gumbel_noise=True,
        use_gathered_v_phi=True, use_layer_checkpoint=False,
        ln_before_distance=True, per_layer_v_phi_scale=True,
        fock_version='v2', n_registers=_PROBE_M,
        register_salience_decay=0.5, register_salience_threshold=0.005,
        creation_gate_hidden=16, stack_discipline=True,
        d_k=16, tau_create_init=8.0,
        reverse_channel=True, reverse_channel_stable=True,
        reverse_channel_pre_ln=True, reverse_channel_soft_norm=True,
        reverse_channel_warmup_steps=4000, reverse_channel_per_layer=True,
        per_register_tau=True, per_register_keys=True,
        ortho_register_init=True, register_repulsion=False,
        prefix_causal_registers=True,
        integrator=CFG_INTEGRATOR,
        vtheta_analytic_force=CFG_VTHETA_ANALYTIC,
        lowrank_max_modes=LOWRANK_MAX_MODES,
        langevin_T=0.0,          # noise would swamp the leak signal
    )
    torch.manual_seed(1234)
    _probe_model = FockMultiXiPARFLM(_probe_cfg)
    _probe_model.V_theta = _DCMCGVT(
        d=_PROBE_D, K=_PROBE_WELLS, n_ctx=_PROBE_XI, n_layers=_PROBE_L,
        rank=2, w_scale=1.0, init_log_precision=-_math.log(_PROBE_D),
        precision_max=2.0/_PROBE_D, precision_lr_max=PRECISION_LR_MAX,
        code_init_std=0.02,
    )
    _idr(_probe_model)
    _probe_model.double().eval()

    with torch.no_grad():
        _probe_model.reverse_channel_scale.fill_(1.0)
        _probe_model.reverse_warmup_step.fill_(4000)

    _t_p = _PROBE_T // 2
    _prng = np.random.default_rng(7)
    _x1 = torch.from_numpy(_prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T))).long()
    _x2 = _x1.clone()
    _x2[:, _t_p:] = torch.from_numpy(
        _prng.integers(0, _PROBE_VOCAB, (2, _PROBE_T - _t_p))).long()

    with torch.enable_grad():
        _la = _probe_model(_x1)[0].detach()
        _lb = _probe_model(_x2)[0].detach()
    _max_delta = float((_la[:, :_t_p] - _lb[:, :_t_p]).abs().max().item())

    _probe_model.train()
    torch.manual_seed(99)
    with torch.enable_grad():
        _lta = _probe_model(_x1)[0].detach()
    torch.manual_seed(99)
    with torch.enable_grad():
        _ltb = _probe_model(_x2)[0].detach()
    _max_delta = max(_max_delta,
                     float((_lta[:, :_t_p] - _ltb[:, :_t_p]).abs().max().item()))

    _passed = (_max_delta == 0.0)
    del _probe_model, _la, _lb, _lta, _ltb, _x1, _x2
    gc.collect()

    status = 'PASS' if _passed else '*** FAIL ***'
    print(f'\n[causal probe] step {step_num:,}  max|dlogit|={_max_delta:.3e}  [{status}]')
    if not _passed:
        print('[causal probe] WARNING: nonzero future sensitivity detected!')
    return _passed, _max_delta


def run_trained_leak_probe(step_num):
    _debug_dir = str(CA_DIR / 'scaleup' / 'debug')
    if _debug_dir not in sys.path:
        sys.path.insert(0, _debug_dir)
    from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

    print(f'\n{"="*64}')
    print(f'[trained leak probe] step {step_num:,} -- running on live model')
    print(f'{"="*64}')

    probe_res = probe_trained_leak(
        model, val_ids, device=DEVICE, context=BLOCK_SIZE,
        n_pairs=TRAINED_LEAK_PROBE_PAIRS, use_float64=False)
    honest_res = honest_ppl_test(
        model, val_ids, k=TRAINED_LEAK_PROBE_K,
        context=BLOCK_SIZE, batch=BATCH_SIZE, device=DEVICE)
    model.train()

    result = {
        'step': step_num,
        'probe_max_dlogit_past': probe_res['max_dlogit_past'],
        'probe_mean_dnll_past_nats': round(probe_res['mean_dnll_past'], 6),
        'probe_gate_zero_control': probe_res['gate_zero_control'],
        'honest_k': honest_res['k'],
        'ppl_mid_window_standard': round(honest_res['ppl_mid_window'], 4),
        'ppl_last_pos_leak_free': round(honest_res['ppl_last_pos'], 4),
        'paired_diff_nats': round(honest_res['paired_diff_nats'], 6),
        'paired_diff_se': round(honest_res['paired_diff_se'], 6),
    }

    _leak_status = 'CLEAN' if result['paired_diff_nats'] < 0.1 else 'LEAK DETECTED'
    print(f'\n[trained leak probe] step {step_num:,}  '
          f'honest_PPL={result["ppl_last_pos_leak_free"]:.2f}  '
          f'standard_PPL={result["ppl_mid_window_standard"]:.2f}  '
          f'diff={result["paired_diff_nats"]:+.4f} nats  [{_leak_status}]')
    return result


def save_checkpoint(step_num, val_loss_val, tag_suffix=''):
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'model_cfg': asdict(model_cfg),
        'train_cfg': {
            'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE,
            'grad_accum': GRAD_ACCUM, 'effective_batch': EFFECTIVE_BATCH,
            'steps': TOTAL_STEPS, 'lr': LR, 'weight_decay': WEIGHT_DECAY,
            'warmup_steps': WARMUP_STEPS, 'grad_clip': GRAD_CLIP,
            'grad_clip_vphi': GRAD_CLIP_VPHI,
            'optimizer': OPTIMIZER, 'grad_centralization': GRAD_CENTRALIZATION,
            'lambda_v': LAMBDA_V, 'v_theta_variant': V_THETA_VARIANT,
            'lr_schedule': LR_SCHEDULE,
            'v_theta_n_heads': V_THETA_N_HEADS,
            'v_theta_wells_per_head': V_THETA_WELLS_PER_HEAD,
            'v_theta_depth_condition': V_THETA_DEPTH_CONDITION,
            'v_theta_depth_code_init_std': V_THETA_DEPTH_CODE_INIT_STD,
            'aniso_rank': ANISO_RANK,
            'lambda_fock_reg': LAMBDA_FOCK_REG,
            'integrator': INTEGRATOR,
            'cfg_integrator': CFG_INTEGRATOR,
            'vtheta_analytic_force': CFG_VTHETA_ANALYTIC,
            'langevin_T': LANGEVIN_T,
        },
        'step': step_num,
        'val_loss': val_loss_val,
        'val_ppl': math.exp(val_loss_val),
        'gamma': model.gamma.item(),
        'xi_alphas': model.xi_alpha_values(),
        'variant': (f'fock_parf_multixi_v2.1_aniso_gaussian_'
                f'dcvt{V_THETA_N_HEADS}_{INTEGRATOR}'),
        'corpus': 'openwebtext',
        'phase': 7,
        'seed': SEED,
    }
    fname = f'{CKPT_PREFIX}_step{step_num}{tag_suffix}.pt'
    path = CKPT_DIR / fname
    for _attempt in range(2):
        try:
            torch.save(ckpt, path)
            break
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error saving checkpoint; remounting... ({_e})')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}')
                    print(f'[WARN] Checkpoint NOT saved: {path}')
                    return None
            else:
                print(f'[WARN] Checkpoint save failed: {_e}')
                return None
    print(f'  Checkpoint saved: {path}  (PPL={math.exp(val_loss_val):.2f})')
    if '_best' in tag_suffix:
        canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        import shutil
        shutil.copy2(path, canonical)
        print(f'  Canonical best: {canonical}')
    return path


# -- Optimizer --
# _trainable is ALSO the legacy flat parameter order, i.e. exactly the
# ordering every pre-SS51 optimizer_state_dict on disk was indexed by.
# _remap_optim_state below depends on that, so do not reorder it.
_trainable = [p for p in model.parameters() if p.requires_grad]


def split_decay_params(mdl):
    """(decay, no_decay) -- standard practice: weight-decay only >=2-D.

    2026-09-09 (companion note SS51). Until now AdamW was handed a flat
    list, so WEIGHT_DECAY=0.01 hit every parameter including the 1-D
    scale/temperature-like ones. That is actively harmful for
    creation_gate_qkv.log_tau, which is LOG-parameterised: decoupled decay
    pulls log_tau toward 0, i.e. pulls tau toward 1, from an
    initialisation of tau_create_init=8.0. Integrating the real WSD
    schedule with ZERO loss gradient, decay alone takes tau 8.0 -> 5.61 by
    step 65k -> 5.42 by step 72k -> 5.11 by step 100k, and the observed
    pool at step ~72k is min 5.21 / median 6.48 / max 8.95 -- i.e. the
    pure-decay line runs straight through the middle of the measured
    pool, and accounts for ~70% of the register-14 drift that SS49.5 read
    as a self-reinforcing runaway.
    The same argument applies to the other 1-D parameters that dominate
    every captured spike: reverse_channel_scale (initialised at zeros),
    ReverseChannel.logit_scale, and V_theta.depth_code.
    """
    decay, no_decay = [], []
    for _n, p in mdl.named_parameters():
        if not p.requires_grad:
            continue
        (decay if p.ndim >= 2 else no_decay).append(p)
    return decay, no_decay


def _remap_optim_state(old_sd, old_order, new_optim):
    """Re-index a flat-param-list optimizer state_dict onto param groups.

    torch keys optimizer state by each parameter's POSITION in the
    flattened param_groups. Splitting one group into two therefore
    invalidates every key: load_state_dict() raises on the group-count
    mismatch (which the callers below used to swallow, silently discarding
    all Adam moments), and -- worse -- a hand-rolled sequential re-index
    with the right group SIZES loads without error while attaching most
    moment buffers to the wrong parameters. Verified on a toy model: the
    sequential version mis-assigns 4 of 6 parameters silently; this
    version reproduces every exp_avg and step counter exactly.

    The emitted param_groups carry `new_optim`'s OWN hyperparameters,
    because torch's load_state_dict keeps the *saved* group's
    hyperparameters and only substitutes the live group's `params` list --
    so emitting bare {'params': ...} groups would silently throw away the
    per-group weight_decay this whole change exists to introduce.
    """
    pos_of, idx = {}, 0
    for g in new_optim.param_groups:
        for p in g['params']:
            pos_of[id(p)] = idx
            idx += 1
    new_state, dropped = {}, 0
    for old_idx, st in old_sd['state'].items():
        old_idx = int(old_idx)
        p = old_order[old_idx] if old_idx < len(old_order) else None
        if p is None or id(p) not in pos_of:
            dropped += 1
            continue
        new_state[pos_of[id(p)]] = st
    new_pgs, n = [], 0
    for g in new_optim.param_groups:
        ng = {k: v for k, v in g.items() if k != 'params'}
        ng['params'] = list(range(n, n + len(g['params'])))
        n += len(g['params'])
        new_pgs.append(ng)
    return {'state': new_state, 'param_groups': new_pgs}, dropped


def load_optim_state(state_dict, label=''):
    """Restore optimizer state, remapping across a param-group change.

    Tries the direct load first (correct and cheapest for any checkpoint
    written under the same grouping), then falls back to the explicit
    old-position -> Parameter -> new-position remap for pre-SS51
    checkpoints. Only if BOTH fail do we start with fresh moments, and
    unlike the previous behaviour that outcome is now reported loudly
    rather than hidden inside an `except ValueError: pass`.
    """
    try:
        optim.load_state_dict(state_dict)
        print(f'  Optimizer state restored{label}.')
        return True
    except (ValueError, KeyError) as e:
        try:
            remapped, dropped = _remap_optim_state(
                state_dict, _trainable, optim)
            optim.load_state_dict(remapped)
            print(f'  Optimizer state restored{label} via SS51 param-group '
                  f'remap ({len(remapped["state"])} params carried over, '
                  f'{dropped} dropped).')
            return True
        except Exception as e2:
            print(f'  [WARN] Optimizer state could NOT be restored{label} '
                  f'-- Adam moments start fresh. direct={e}  remap={e2}')
            return False


if OPTIMIZER == 'adamw':
    if NO_DECAY_1D:
        _decay, _no_decay = split_decay_params(model)
        optim = torch.optim.AdamW(
            [{'params': _decay, 'weight_decay': WEIGHT_DECAY},
             {'params': _no_decay, 'weight_decay': 0.0}],
            lr=LR, betas=(0.9, 0.95))
        _n_d = sum(p.numel() for p in _decay)
        _n_nd = sum(p.numel() for p in _no_decay)
        print(f'[no-decay-1d] weight_decay={WEIGHT_DECAY} on {len(_decay)} '
              f'tensors ({_n_d:,} params); 0.0 on {len(_no_decay)} 1-D '
              f'tensors ({_n_nd:,} params, incl. log_tau / '
              f'reverse_channel_scale / logit_scale / depth_code)')
    else:
        optim = torch.optim.AdamW(_trainable, lr=LR,
                                  weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lamb':
    try:
        import torch_optimizer
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torch_optimizer'])
        import torch_optimizer
    optim = torch_optimizer.Lamb(_trainable, lr=LR,
                                weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
elif OPTIMIZER == 'lion':
    try:
        from lion_pytorch import Lion
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lion-pytorch'])
        from lion_pytorch import Lion
    optim = Lion(_trainable, lr=LR, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.99))
else:
    raise ValueError(f'Unknown OPTIMIZER={OPTIMIZER!r}; choose adamw / lamb / lion')
print(f'Optimizer: {type(optim).__name__}')
if DEVICE == 'cuda':
    _free, _total = torch.cuda.mem_get_info()
    print(f'  CUDA after optimiser: {_free/1e9:.1f} GB free / {_total/1e9:.1f} GB')

# -- Resume --
if resume_ckpt is not None and resume_step < TOTAL_STEPS:
    print(f'Resuming from checkpoint at step {resume_step:,}: {resume_ckpt}')
    ckpt_data = torch.load(resume_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_data['model_state_dict'], strict=False)
    if (REVERSE_CHANNEL and REVERSE_CHANNEL_STABLE and REVERSE_CHANNEL_RESET_SCALE
            and getattr(model, 'reverse_channel_scale', None) is not None):
        with torch.no_grad():
            model.reverse_channel_scale.zero_()
            if hasattr(model, 'reverse_warmup_step'):
                model.reverse_warmup_step.zero_()
        print('  [E5c] reverse_channel_scale re-zeroed + warmup reset')
    if 'optimizer_state_dict' in ckpt_data:
        load_optim_state(ckpt_data['optimizer_state_dict'])
    prev_ppl = ckpt_data.get('val_ppl', float('nan'))
    print(f'  Model loaded. Previous PPL: {prev_ppl:.2f}')
    del ckpt_data
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# -- Training state --
log_path = RESULTS_DIR / 'training_log.jsonl'
_log_fh = [log_path.open('a')]
_log_write_count = [0]
# Google Drive's FUSE mount buffers writes locally and only reliably syncs
# them to Drive on file-descriptor close. A long training run keeps a single
# handle open for its whole (up to 24h) session, so if Colab kills the
# runtime abruptly (session timeout, disconnect, OOM) any writes since the
# last close can be silently lost even though flush() succeeded locally.
# Forcing an fsync + periodic close/reopen bounds how much log history can
# be lost to roughly _LOG_REOPEN_EVERY * LOG_INTERVAL steps.
_LOG_REOPEN_EVERY = 10


def _log_write(record_str):
    for _attempt in range(2):
        try:
            _log_fh[0].write(record_str)
            _log_fh[0].flush()
            try:
                os.fsync(_log_fh[0].fileno())
            except OSError:
                pass  # fsync isn't guaranteed to be meaningful on FUSE mounts
            _log_write_count[0] += 1
            if _log_write_count[0] % _LOG_REOPEN_EVERY == 0:
                _log_fh[0].close()
                _log_fh[0] = log_path.open('a')
            return
        except OSError as _e:
            if _e.errno == 107 and _attempt == 0:
                print(f'[WARN] Drive transport error on log write; remounting...')
                try:
                    from google.colab import drive as _drv
                    _drv.mount('/content/drive', force_remount=True)
                    try:
                        _log_fh[0].close()
                    except Exception:
                        pass
                    _log_fh[0] = log_path.open('a')
                except Exception as _re:
                    print(f'[WARN] Drive remount failed: {_re}; log record lost.')
                    return
            else:
                print(f'[WARN] Log write failed (attempt {_attempt+1}): {_e}')
                return


t0 = time.time()
model.train()
run_ntp = 0.0
run_vreg = 0.0
run_fock_reg = 0.0
n_run = 0
n_skipped = 0

best_val_ppl = float('inf')
_best_ckpt_path = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'

if not _best_ckpt_path.exists():
    _step_bests = sorted(CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_best.pt'))
    if _step_bests:
        _best_ckpt_path = _step_bests[-1]
        import shutil
        _canonical = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
        shutil.copy2(_best_ckpt_path, _canonical)
        _best_ckpt_path = _canonical

if _best_ckpt_path.exists():
    try:
        _bd = torch.load(_best_ckpt_path, map_location='cpu', weights_only=False)
        best_val_ppl = _bd.get('val_ppl', float('inf'))
        print(f'Restored running best PPL: {best_val_ppl:.2f}')
        del _bd
    except Exception as e:
        print(f'[warn] {e}')

_grad_norm_ema = 0.0
_grad_norm_above_thresh = 0


def _archive_bundle(path, archive_dirname, max_keep=None,
                     min_grad_norm=None, grad_norm=None):
    """Copy a just-written spikebatch/prereload file into its permanent
    GDRIVE_ROOT archive AT WRITE TIME, rather than relying on the next
    session's Cell 1c sweep to catch it -- Cell 1c protects the gap
    BETWEEN sessions, but Colab is single-kernel (Cell 6 blocks it for
    as long as run_training() runs), so it cannot be re-run mid-session
    without interrupting training, and one session has already produced
    more captures (13) than a full ring rotation (12) before hitting its
    24h limit. Cell 1c stays as a belt-and-suspenders sweep for anything
    from before this fix, or if this copy fails silently below.

    2026-09-11: BOUNDED, not unconditional -- archiving every capture
    forever does not scale to a 150,000-step continuation (see the
    ARCHIVE_* config comment above CAPTURE_SPIKE_THRESHOLD). Two
    independent, optional bounds:

    min_grad_norm/grad_norm : if both given, skip archiving entirely
        when grad_norm < min_grad_norm -- a routine, sub-threshold
        capture not worth the permanent disk cost. Pass neither (both
        None, the default) to archive unconditionally, e.g. for
        prereload snapshots, which are already rare by construction
        (every one implies a genuine watchdog reload).
    max_keep : if given, evict the OLDEST (by mtime) archived file(s)
        of this same suffix once the archive holds more than max_keep --
        same eviction policy already used for the live CKPT_DIR ring
        (SPIKEBATCH_SNAPSHOT_MAX_KEEP / PRERELOAD_SNAPSHOT_MAX_KEEP), so
        the archive can never grow unbounded even under a sustained
        stretch of qualifying events.

    Best-effort and non-blocking throughout: a Drive error here must
    never take down the training loop, same as the eviction unlink()
    calls elsewhere in this cell.
    """
    if path is None:
        return
    if min_grad_norm is not None and (grad_norm is None
                                       or grad_norm < min_grad_norm):
        return
    try:
        _adir = GDRIVE_ROOT / archive_dirname
        _adir.mkdir(exist_ok=True)
        _dst = _adir / path.name
        if not _dst.exists():
            import shutil as _shutil_arch
            _shutil_arch.copy2(path, _dst)
        if max_keep:
            _suffix = path.name.rsplit('_', 1)[-1]  # 'spikebatch.pt' / 'prereload.pt'
            _existing_a = sorted(_adir.glob(f'{CKPT_PREFIX}_step*_{_suffix}'),
                                 key=lambda p: p.stat().st_mtime)
            for _stale_a in _existing_a[:-max_keep]:
                try:
                    _stale_a.unlink()
                except OSError as _e:
                    print(f'[WARN] could not evict stale archive file '
                          f'{_stale_a}: {_e}')
    except Exception as _e:
        print(f'[WARN] could not archive {path.name} to {archive_dirname}: {_e}')


def _reload_best(pre_reload_step=None):
    if pre_reload_step is not None and PRERELOAD_SNAPSHOT_MAX_KEEP > 0:
        # Snapshot the state that TRIGGERED this reload before it gets
        # overwritten -- this is the state a post-hoc stiffness audit
        # actually wants (see the config comment above for why the other
        # two save triggers cannot be relied on to have captured it).
        _pr_path = save_checkpoint(pre_reload_step, float('nan'),
                                    tag_suffix='_prereload')
        _archive_bundle(_pr_path, 'prereload_archive',
                         max_keep=ARCHIVE_PRERELOAD_MAX_KEEP)
        _existing = sorted(
            CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_prereload.pt'),
            key=lambda p: p.stat().st_mtime,
        )
        for _stale in _existing[:-PRERELOAD_SNAPSHOT_MAX_KEEP]:
            try:
                _stale.unlink()
            except OSError as _e:
                print(f'[WARN] could not remove stale prereload snapshot '
                      f'{_stale}: {_e}')
    if not _best_ckpt_path.exists():
        return resume_step
    ckpt = torch.load(_best_ckpt_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    load_optim_state(ckpt['optimizer_state_dict'], label=' (watchdog reload)')
    s = ckpt.get('step', 0)
    p = ckpt.get('val_ppl', float('nan'))
    del ckpt
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    print(f'[watchdog] Reloaded best: step {s:,} PPL {p:.2f}')
    return s


steps_this_session = 0

# -- Schedule summary --
if LR_SCHEDULE == 'wsd':
    _warmup_end = int(WSD_WARMUP_FRAC * TOTAL_STEPS)
    _stable_end = int((WSD_WARMUP_FRAC + WSD_STABLE_FRAC) * TOTAL_STEPS)
    _sched_str = (f'WSD: warmup 0->{_warmup_end:,}, stable {_warmup_end:,}->{_stable_end:,}, '
                  f'decay {_stable_end:,}->{TOTAL_STEPS:,}, floor={WSD_LR_FLOOR:.2e}')
else:
    _sched_str = f'cosine: warmup {WARMUP_STEPS:,} steps'

print(f'\n{"="*60}')
print(f'CfC/BAOAB [{INTEGRATOR}]: steps {resume_step+1:,} -> {TOTAL_STEPS:,}')
print(f'  batch={BATCH_SIZE} x accum={GRAD_ACCUM} (eff={EFFECTIVE_BATCH})')
print(f'  block={BLOCK_SIZE}  lr={LR}  grad_clip={GRAD_CLIP}')
print(f'  schedule: {_sched_str}')
print(f'  d={model_cfg.d}  L={model_cfg.L}  M={model_cfg.n_registers}')
print(f'  fock-reg: lambda={LAMBDA_FOCK_REG}  eps={FOCK_REG_EPS}')
print(f'  integrator: {INTEGRATOR}  (cfg.integrator={model_cfg.integrator}, '
      f'analytic V_theta force={model_cfg.vtheta_analytic_force}, '
      f'thermostat T={model_cfg.langevin_T:g})')
print(f'  watchdog: threshold={GRAD_NORM_EMA_THRESHOLD} patience={GRAD_NORM_EMA_PATIENCE} '
      f'hard_trigger={GRAD_NORM_HARD_TRIGGER}')
print(f'  spike-capture: {"on" if CAPTURE_SPIKE_BATCH else "off"}'
      f'  capture_threshold={CAPTURE_SPIKE_THRESHOLD}'
      f'  keep={SPIKEBATCH_SNAPSHOT_MAX_KEEP}')
print(f'  per-group clip: default={GRAD_CLIP}  overrides={GRAD_CLIP_OVERRIDES}')
if CUDA_EMPTY_CACHE_INTERVAL and CUDA_EMPTY_CACHE_WARMUP_STEPS:
    print(f'  post-resume CUDA cache warmup: empty_cache() every '
          f'{CUDA_EMPTY_CACHE_INTERVAL} steps for the first '
          f'{CUDA_EMPTY_CACHE_WARMUP_STEPS} steps of each run_training() '
          f'call (re-arms on every fresh resume, first one starts at '
          f'step {resume_step:,})')
if REVERSE_CHANNEL:
    _rev_mode = ('stable (QK-norm + '
                 + ('soft-norm' if REVERSE_CHANNEL_SOFT_NORM else 'RMS-norm')
                 + (' + pre-LN' if REVERSE_CHANNEL_PRE_LN else '') + ')')
    print(f'  reverse channel: {_rev_mode}  warmup={REVERSE_CHANNEL_WARMUP_STEPS} forwards')
else:
    print('  reverse channel: OFF')
print(f'{"="*60}\n')


# 2026-08-30 (companion note SS37): assign_clip_group / per_group_grad_norms
# / clip_grads_per_group moved out to grad_clip_utils.py so Cell 6d's replay
# helpers can import them directly instead of silently depending on Cell 6
# having already run first (see that module's docstring for why).
from grad_clip_utils import (
    GradClipConfig, per_group_grad_norms, clip_grads_per_group, assign_clip_group,
)

_GRAD_CLIP_CFG = GradClipConfig(
    default_clip=GRAD_CLIP,
    overrides=GRAD_CLIP_OVERRIDES,
    watchdog_exclude_groups=frozenset(WATCHDOG_EXCLUDE_GROUPS),
)

# 2026-09-07 (companion note SS45.3-45.4): replay_clip_ablation showed E/P's
# current sum_then_clip order already disagrees with clip_then_sum on the
# applied update's *direction* by ~46-47 degrees even at today's
# default_clip=1.0 (worsening to ~60 degrees once tightened), and the
# pattern reproduces near-identically across both captured near-trigger
# bundles (52940, 55919) despite their very different layer-profile shapes
# (SS44) -- pointing to a structural one-or-few-outlier-microbatch pattern,
# not spike-specific noise. This wires clip_then_sum in for the groups
# below: each microbatch's own gradient for those groups is clipped to
# CLIP_THEN_SUM_THRESHOLD *before* being added into the running total,
# instead of accumulating raw across all GRAD_ACCUM microbatches and
# clipping once after (which every other group still does, unchanged).
# Threshold 0.3 chosen not for magnitude parity with today's
# sum_then_clip(default_clip=1.0) output (SS45.4: the ratio between the two
# orders isn't constant across thresholds, so parity isn't a single clean
# number) but because clip_then_sum(0.3) measured ~0.54 on both replayed
# bundles -- already smaller than today's 1.0 (tightens the step, the
# original ask) while still in the direction-divergent regime (cos~0.55,
# i.e. genuinely a different, outlier-robust direction rather than just a
# rescale of the same one). Set to None/empty set to disable and fall back
# to sum_then_clip for every group, matching pre-SS45.3 behavior exactly.
# Only takes effect when PER_GROUP_CLIP is True (it operates on named
# clip groups, which don't exist in the flat-clip branch below).
CLIP_THEN_SUM_GROUPS = {'E', 'P'}
CLIP_THEN_SUM_THRESHOLD = 0.3

_CLIP_THEN_SUM_PARAMS = {}  # group_key -> list[nn.Parameter], built once below
if PER_GROUP_CLIP and CLIP_THEN_SUM_GROUPS:
    for _n, _p in model.named_parameters():
        if not _p.requires_grad:
            continue
        _key, _ = assign_clip_group(_n, _GRAD_CLIP_CFG)
        if _key in CLIP_THEN_SUM_GROUPS:
            _CLIP_THEN_SUM_PARAMS.setdefault(_key, []).append(_p)
    _n_cts_params = sum(len(v) for v in _CLIP_THEN_SUM_PARAMS.values())
    print(f'[clip-then-sum] active for groups {sorted(_CLIP_THEN_SUM_PARAMS)} '
          f'({_n_cts_params} param(s) total), threshold={CLIP_THEN_SUM_THRESHOLD}')

# SS49.8/SS50: resolve log_tau once, here, rather than every step. None on
# any variant without a per-register creation temperature (v1 protocol,
# or tau_create_init=None), in which case the projection below is skipped.
_log_tau_param = dict(model.named_parameters()).get('creation_gate_qkv.log_tau')
_LOG_TAU_MIN = math.log(TAU_CREATE_MIN) if TAU_CREATE_MIN is not None else None
if _log_tau_param is not None and _LOG_TAU_MIN is not None:
    with torch.no_grad():
        _tau_now = _log_tau_param.detach().float().exp()
        _n_below = int((_tau_now < TAU_CREATE_MIN).sum())
    print(f'[tau-floor] projecting log_tau to tau >= {TAU_CREATE_MIN} after '
          f'each optim.step(); current min={float(_tau_now.min()):.4f} '
          f'(register {int(_tau_now.argmin())}), median='
          f'{float(_tau_now.median()):.4f}, {_n_below} of {_tau_now.numel()} '
          f'below the floor'
          + (' -- ACTIVE IMMEDIATELY, this will perturb the model'
             if _n_below else ' -- no-op at current values'))
elif _log_tau_param is None:
    print('[tau-floor] no creation_gate_qkv.log_tau on this model; skipped')

_last_pg_norms = {}
_last_spike_step = -10**9


def run_training(start_step, total_steps):
    """Run training steps [start_step, total_steps).

    2026-08-30 (companion note SS37): this used to be a bare top-level
    'for step in range(resume_step, TOTAL_STEPS):' loop. Interrupting it
    (Runtime -> Interrupt execution) to go inspect spike captures left the
    interrupted iteration's tensors (x, y, loss, ...) alive as ordinary
    notebook globals -- and, worse, kept the *entire* forward/backward
    graph for that half-finished step pinned in GPU memory, since nothing
    ever went out of scope. On this run that reliably OOM'd every
    replay_spike_batch() call afterward (78+ GiB 'allocated', not just
    reserved-but-cached) until a manual gc.collect()+empty_cache() dance
    cleared it. Resuming also required a full checkpoint-save -> restart
    runtime -> resume_ckpt override dance, because Cell 5 unconditionally
    rebuilds `model` from scratch and this cell's own resume block always
    reloads from whatever's on disk.

    As a real function, an interrupt here unwinds this function's stack
    frame (caught by the try/except around the call below) -- every
    per-step local (x, y, loss, xb, yb, ...) is freed within that same
    frame teardown, same as a normal return. To keep training after
    inspecting captures, just call this again with the step it reports:

        next_step = run_training(next_step, TOTAL_STEPS)

    No rebuild, no checkpoint reload, no runtime restart required -- those
    are now only needed when you actually want to pick up new *code*
    (e.g. a fresh git pull of this notebook), not just to look around.
    `save_manual_checkpoint(next_step)` below is still there for that case.
    """
    global run_ntp, run_vreg, run_fock_reg, n_run, n_skipped, best_val_ppl
    global _grad_norm_ema, _grad_norm_above_thresh, steps_this_session
    global _last_pg_norms, _last_spike_step, step
    global _autosave_ckpt_done_this_process

    for step in range(start_step, total_steps):
        lr_now = lr_schedule(step)
        for g in optim.param_groups:
            g['lr'] = lr_now

        optim.zero_grad(set_to_none=True)
        # Phase 1 spike-batch capture: snapshot the RNG state this step's
        # forward+backward is about to consume (the Langevin thermostat noise
        # draw depends on it) and start a fresh microbatch list. Cheap enough
        # to do unconditionally every step; discarded if this step is clean.
        _step_rng_cpu = torch.get_rng_state()
        _step_rng_cuda = torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None
        _this_step_batches = []
        accum_ntp = 0.0
        accum_vreg = 0.0
        accum_fock_reg = 0.0
        accum_rep = 0.0
        # clip_then_sum running total, keyed by id(param) -- see SS45.3-45.4
        # and the CLIP_THEN_SUM_GROUPS config comment above. Empty dict is a
        # true no-op when _CLIP_THEN_SUM_PARAMS is empty (feature disabled).
        _cts_running = {}
        for _acc in range(GRAD_ACCUM):
            xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
            _this_step_batches.append((xb.copy(), yb.copy()))
            x = torch.from_numpy(xb).to(DEVICE)
            y = torch.from_numpy(yb).to(DEVICE)
            loss, loss_ntp, v_reg, fock_reg = forward_with_vreg(
                x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
            if REGISTER_REPULSION:
                _rep = model.pop_repulsion_loss()
                loss = loss + _rep
                accum_rep += float(_rep.detach()) / GRAD_ACCUM
            (loss / GRAD_ACCUM).backward()
            accum_ntp      += loss_ntp.item()       / GRAD_ACCUM
            accum_vreg     += float(v_reg.detach()) / GRAD_ACCUM
            accum_fock_reg += float(fock_reg.detach()) / GRAD_ACCUM

            # clip_then_sum (SS45.3-45.4): right after this microbatch's own
            # backward() and before the next microbatch's backward() adds
            # onto it, .grad for CLIP_THEN_SUM_GROUPS' params holds exactly
            # this microbatch's own contribution. nn.utils.clip_grad_norm_
            # clips that JOINTLY across every param in a clip-group (same
            # mechanics clip_grads_per_group uses post-hoc for every other
            # group, and same as replay_clip_ablation's flattened-per-group
            # norm -- important for correctness if a group ever holds more
            # than one tensor, not just today's single-param E/P). Fold the
            # (now-clipped) result into the running total, then zero .grad
            # so the normal accumulation semantics below, which every other
            # group still relies on, never see it.
            for _gparams in _CLIP_THEN_SUM_PARAMS.values():
                if not any(_p.grad is not None for _p in _gparams):
                    continue
                nn.utils.clip_grad_norm_(_gparams, CLIP_THEN_SUM_THRESHOLD)
                for _p in _gparams:
                    if _p.grad is None:
                        continue
                    _pid = id(_p)
                    _contrib = _p.grad.detach().clone()
                    if _pid in _cts_running:
                        _cts_running[_pid] += _contrib
                    else:
                        _cts_running[_pid] = _contrib
                    _p.grad.zero_()

        # Splice the clip_then_sum total back into .grad for these params,
        # replacing the (already-zeroed) result of normal accumulation --
        # everything downstream (GRAD_CENTRALIZATION, clip_grads_per_group's
        # post-hoc per-group safety clip, the watchdog, spike-batch capture,
        # logging) reads .grad exactly as before and needs no other change;
        # it just now sees this group's already-outlier-bounded total
        # instead of the raw microbatch sum.
        for _gparams in _CLIP_THEN_SUM_PARAMS.values():
            for _p in _gparams:
                _pid = id(_p)
                if _pid in _cts_running:
                    _p.grad = _cts_running[_pid]

        if GRAD_CENTRALIZATION:
            for p in model.parameters():
                if p.grad is not None and p.grad.dim() >= 2:
                    p.grad.sub_(p.grad.mean(dim=tuple(range(1, p.grad.dim())), keepdim=True))

        if PER_GROUP_CLIP:
            grad_norm, _last_pg_norms = clip_grads_per_group(model, _GRAD_CLIP_CFG)
        else:
            _last_pg_norms = per_group_grad_norms(model, _GRAD_CLIP_CFG) if GRAD_SPIKE_DEBUG else {}
            if model.V_phi is not None:
                nn.utils.clip_grad_norm_(model.V_phi.parameters(), GRAD_CLIP_VPHI)
            grad_norm = nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], GRAD_CLIP)

        if GRAD_SPIKE_DEBUG:
            _tot_preclip = float(grad_norm)
            if (_tot_preclip > GRAD_SPIKE_THRESHOLD
                    and (step - _last_spike_step) >= GRAD_SPIKE_COOLDOWN):
                _last_spike_step = step
                if _last_pg_norms:
                    _top = sorted(_last_pg_norms.items(),
                                  key=lambda kv: kv[1], reverse=True)[:8]
                    _brk = '  '.join(f'{k}={v:.1f}' for k, v in _top)
                else:
                    _brk = '(enable PER_GROUP_CLIP for breakdown)'
                print(f'\n[spike] step {step+1}: pre-clip total grad={_tot_preclip:.1f}  '
                      f'ntp={accum_ntp:.3f}  v_reg={accum_vreg:.4f}  fock_reg={accum_fock_reg:.4f}')
                print(f'[spike]   top groups: {_brk}')
                _log_write(json.dumps({
                    'step': step + 1, 'event': 'grad_spike',
                    'pre_clip_grad_norm': round(_tot_preclip, 2),
                    'ntp': round(accum_ntp, 4), 'v_reg': round(accum_vreg, 4),
                    'fock_reg': round(accum_fock_reg, 4),
                    'top_groups': {k: round(v, 2) for k, v in _top} if _last_pg_norms else {},
                }) + '\n')

        # Phase 1 spike-batch capture (companion note SS33.3, gate widened in
        # SS36 to CAPTURE_SPIKE_THRESHOLD -- deliberately independent of the
        # reload-triggering GRAD_NORM_HARD_TRIGGER below): if this step crosses
        # CAPTURE_SPIKE_THRESHOLD, snapshot NOW -- model.parameters() still hold
        # the values the forward+backward above actually saw, because
        # optim.step() (which mutates them) has not run yet. By the time the
        # watchdog block further down fires (if it fires at all -- most
        # captures under this lower threshold will NOT trip the hard reload),
        # it is too late: optim.step() has already applied the update, which
        # is why the existing '_prereload' snapshot alone cannot reproduce it.
        _raw_gn_pre = float(grad_norm)
        if (CAPTURE_SPIKE_BATCH and CAPTURE_SPIKE_THRESHOLD is not None
                and _raw_gn_pre > CAPTURE_SPIKE_THRESHOLD):
            _top_sb = (sorted(_last_pg_norms.items(), key=lambda kv: kv[1], reverse=True)[:8]
                       if _last_pg_norms else [])
            _spike_bundle = {
                'step': step + 1,
                'pre_clip_grad_norm': round(_raw_gn_pre, 2),
                'ntp': round(accum_ntp, 4), 'v_reg': round(accum_vreg, 4),
                'fock_reg': round(accum_fock_reg, 4),
                'top_groups': {k: round(v, 2) for k, v in _top_sb},
                'model_state_dict': {k: v.detach().cpu().clone()
                                      for k, v in model.state_dict().items()},
                'batches': _this_step_batches,
                'grad_accum': GRAD_ACCUM,
                'rng_state_cpu': _step_rng_cpu,
                'rng_state_cuda': _step_rng_cuda,
            }
            _sb_path = CKPT_DIR / f'{CKPT_PREFIX}_step{step+1}_spikebatch.pt'
            torch.save(_spike_bundle, _sb_path)
            _archive_bundle(_sb_path, 'spikebatch_archive',
                             max_keep=ARCHIVE_SPIKEBATCH_MAX_KEEP,
                             min_grad_norm=ARCHIVE_MIN_GRAD_NORM,
                             grad_norm=_raw_gn_pre)
            _existing_sb = sorted(
                CKPT_DIR.glob(f'{CKPT_PREFIX}_step*_spikebatch.pt'),
                key=lambda p: p.stat().st_mtime,
            )
            for _stale_sb in _existing_sb[:-SPIKEBATCH_SNAPSHOT_MAX_KEEP]:
                try:
                    _stale_sb.unlink()
                except OSError as _e:
                    print(f'[WARN] could not remove stale spikebatch snapshot '
                          f'{_stale_sb}: {_e}')
            print(f'[spike-capture] saved pre-step (weights, '
                  f'{len(_this_step_batches)} microbatch(es), rng) -> {_sb_path.name}')

        if torch.isfinite(grad_norm) and math.isfinite(accum_ntp):
            optim.step()
            for bank in model.V_theta.banks:
                if hasattr(bank, 'clamp_params'):
                    bank.clamp_params()
            # SS49.8/SS50: project log_tau back into [log(TAU_CREATE_MIN), inf).
            # Sits here, next to clamp_params(), because it is the same kind
            # of thing: a feasible-set projection applied after the update,
            # not a change to the forward function. No-op at current values.
            if TAU_CREATE_MIN is not None and _log_tau_param is not None:
                with torch.no_grad():
                    _log_tau_param.clamp_(min=_LOG_TAU_MIN)
        else:
            n_skipped += 1
            optim.zero_grad(set_to_none=True)

        # Post-resume CUDA allocator warmup mitigation (see the
        # CUDA_EMPTY_CACHE_WARMUP_STEPS config comment above). Cheap
        # relative to a step's own compute, but not free, hence bounded to
        # only the first CUDA_EMPTY_CACHE_WARMUP_STEPS iterations of THIS
        # run_training() call rather than every step forever.
        if (CUDA_EMPTY_CACHE_INTERVAL and CUDA_EMPTY_CACHE_WARMUP_STEPS
                and (step - start_step) < CUDA_EMPTY_CACHE_WARMUP_STEPS
                and (step - start_step) % CUDA_EMPTY_CACHE_INTERVAL == 0):
            torch.cuda.empty_cache()

        # -- Wall-clock safety net against Colab's ~24h hard session
        #    cutoff (companion note SS46) -- checked every step (a single
        #    /proc/uptime read, negligible cost) so it fires as soon as
        #    possible after crossing the threshold rather than waiting for
        #    some other periodic interval; fires at most once per process.
        if (AUTOSAVE_WALLCLOCK_HOURS is not None
                and not _autosave_ckpt_done_this_process
                and _vm_uptime_seconds() >= AUTOSAVE_WALLCLOCK_HOURS * 3600):
            print(f'\n[autosave] VM uptime past {AUTOSAVE_WALLCLOCK_HOURS}h -- '
                  f'saving a safety checkpoint at step {step + 1} before '
                  f'Colab can tear this session down unannounced.')
            # A failure here (e.g. the CUDA-session corruption from
            # companion note SS45.x, which made torch.save itself hang/
            # error) must not take the training loop down with it -- this
            # is a best-effort safety net, not a step that training
            # depends on. Marked done either way so a persistently broken
            # save can't retry-and-stall every single step for the rest
            # of the run; if it fails, you still find out immediately from
            # the printed traceback and can save manually instead.
            try:
                save_manual_checkpoint(step + 1)
            except Exception as _e:
                print(f'[autosave][WARN] safety checkpoint failed, '
                      f'continuing training anyway: {_e}')
            _autosave_ckpt_done_this_process = True

        # -- Watchdog --
        _raw_gn = float(grad_norm)
        _grad_norm_ema = (1 - GRAD_NORM_EMA_ALPHA) * _grad_norm_ema + GRAD_NORM_EMA_ALPHA * _raw_gn
        if _grad_norm_ema > GRAD_NORM_EMA_THRESHOLD:
            _grad_norm_above_thresh += 1
        else:
            _grad_norm_above_thresh = 0

        if _grad_norm_above_thresh >= GRAD_NORM_EMA_PATIENCE:
            print(f'\n[watchdog] EMA grad_norm={_grad_norm_ema:.1f} > {GRAD_NORM_EMA_THRESHOLD} '
                  f'for {_grad_norm_above_thresh} steps at step {step+1}.')
            if _last_pg_norms:
                _top = sorted(_last_pg_norms.items(), key=lambda kv: kv[1], reverse=True)[:5]
                print('[watchdog] top group norms (pre-clip): '
                      + ', '.join(f'{k}={v:.1f}' for k, v in _top))
            _log_write(json.dumps({
                'step': step + 1, 'event': 'watchdog_reload',
                'ema_grad_norm': round(_grad_norm_ema, 2),
                'above_thresh_steps': _grad_norm_above_thresh,
            }) + '\n')
            _reload_best(step + 1)
            _grad_norm_ema = 0.0
            _grad_norm_above_thresh = 0
            n_skipped += 1
        elif GRAD_NORM_HARD_TRIGGER is not None and _raw_gn > GRAD_NORM_HARD_TRIGGER:
            # Fast path: a single step this far above threshold (e.g. the
            # 2401.6 / 1815.9 spikes seen 2026-08-23) can already do lasting
            # damage before the slow EMA (alpha=0.05, patience=200 consecutive
            # steps) would ever fire. Reload immediately, no smoothing.
            print(f'\n[watchdog-hard] single-step grad_norm={_raw_gn:.1f} > '
                  f'{GRAD_NORM_HARD_TRIGGER} at step {step+1}.')
            if _last_pg_norms:
                _top = sorted(_last_pg_norms.items(), key=lambda kv: kv[1], reverse=True)[:5]
                print('[watchdog-hard] top group norms (pre-clip): '
                      + ', '.join(f'{k}={v:.1f}' for k, v in _top))
            _log_write(json.dumps({
                'step': step + 1, 'event': 'watchdog_hard_reload',
                'raw_grad_norm': round(_raw_gn, 2),
                'top_groups': {k: round(v, 2) for k, v in _top} if _last_pg_norms else {},
            }) + '\n')
            _reload_best(step + 1)
            _grad_norm_ema = 0.0
            _grad_norm_above_thresh = 0
            n_skipped += 1

        run_ntp += accum_ntp
        run_vreg += accum_vreg
        run_fock_reg += accum_fock_reg
        n_run += 1
        steps_this_session += 1

        if (step + 1) % LOG_INTERVAL == 0:
            avg_ntp = run_ntp / n_run
            avg_vreg = run_vreg / n_run
            avg_fock_reg = run_fock_reg / n_run
            run_ntp, run_vreg, run_fock_reg, n_run = 0.0, 0.0, 0.0, 0
            elapsed = time.time() - t0
            sec_per_step = elapsed / steps_this_session
            remaining = (TOTAL_STEPS - step - 1) * sec_per_step
            alphas = model.xi_alpha_values()
            alpha_str = ','.join(f'{a:.3f}' for a in alphas)
            _top_grp = ''
            _dc_ratio = None
            if PER_GROUP_CLIP and _last_pg_norms:
                _k, _v = max(_last_pg_norms.items(), key=lambda kv: kv[1])
                _top_grp = f'top[{_k}]={_v:.1f}  '
                # dc_ratio (companion note SS38.7): depth_code's own group
                # norm vs. the next-largest group. Discovered post hoc from
                # the 7 captured spike events -- every smooth-cascade one had
                # dc_ratio < 1.8, both localized-blowup ones had > 2.2 -- but
                # that was only ever checked AT capture time (7 points total).
                # Logging it on every LOG_INTERVAL step here (cheap: the
                # breakdown is already computed above for clipping) is what
                # lets a future mining pass check whether it's actually a
                # leading indicator during ordinary training, not just a
                # spike-time coincidence.
                _dc_norm = _last_pg_norms.get('override:depth_code')
                if _dc_norm is not None:
                    _others = [v for k, v in _last_pg_norms.items()
                               if k != 'override:depth_code']
                    _second = max(_others) if _others else 0.0
                    _dc_ratio = _dc_norm / _second if _second > 0 else float('inf')
                    _top_grp += f'dc_ratio={_dc_ratio:.2f}  '

            # weight-space stiffness proxy (companion note SS39.4/SS39.5):
            # SS39's per-row attribution and V_theta exponent-occupancy tests
            # both failed to discriminate the localized-mode spike captures
            # from the smooth-cascade ones -- the localized events were, if
            # anything, the *flatter* ones across rows, and occupancy showed
            # no separation at all. That argues the discriminator (if it
            # exists) lives in the WEIGHTS feeding the low-rank precision
            # factor, not in any particular batch. sigma_max(B_proj.weight)
            # bounds how large ||B_proj(xi)||_F can get for ANY unit-norm xi,
            # *before* _bound_lowrank's runtime tanh cap (SS28/model_aniso_
            # gaussian_vtheta.py) ever engages -- unlike everything tried in
            # SS39, it is a pure function of the current weights, with zero
            # dependence on which tokens are in this step's batch. An SVD of
            # a (K*d*rank, in_d) matrix is cheap relative to a training step
            # but is real compute (unlike dc_ratio, which just reads numbers
            # already computed for clipping), so -- same as dc_ratio -- this
            # only runs at LOG_INTERVAL cadence, not every step.
            _bproj_sigma_max = None
            _bproj_sigma_by_bank = None
            try:
                _vt_banks = model.V_theta.bank.banks
                _sigmas = []
                with torch.no_grad():
                    for _bk in _vt_banks:
                        _bp = getattr(_bk, 'B_proj', None)
                        if _bp is None:
                            continue
                        _sigmas.append(float(torch.linalg.matrix_norm(
                            _bp.weight.detach(), ord=2)))
                if _sigmas:
                    _bproj_sigma_by_bank = [round(s, 4) for s in _sigmas]
                    _bproj_sigma_max = max(_sigmas)
                    _top_grp += f'bproj_sig={_bproj_sigma_max:.2f}  '
            except Exception as _e:
                # best-effort diagnostic (e.g. a non-aniso V_theta variant
                # without .bank.banks) -- must never break the training loop.
                if GRAD_SPIKE_DEBUG:
                    print(f'[warn] b_proj sigma_max diagnostic failed: {_e}')

            # 2026-09-09 (companion note SS49.5/SS49.9): the creation gate's
            # per-register temperatures are learned and nothing keeps them
            # comparable to each other. Register 14's has been drifting to
            # the bottom of the pool (tau 5.3409 -> 5.2104 over steps
            # 68,313-71,985 against a flat pool median of ~6.45, rank
            # 30 -> 32 of 32), which makes its scaled scores an order of
            # magnitude larger than every other register's and -- via
            # dL/dlog_tau = -sum(s_tilde * dL/ds_tilde) -- hands it 97.8%
            # (step 70,522) then 100.0% (step 71,194) of log_tau's whole
            # gradient. Both measured gradients are POSITIVE, so descent
            # lowers log_tau further: the drift is self-reinforcing.
            # This is a property of the WEIGHTS alone -- no batch needed,
            # one exp() over 32 elements -- so unlike bproj_sig above it is
            # essentially free even at this cadence.
            # Stated prediction, so this can falsify as well as confirm: if
            # SS49.5 is right, tau_argmin stays pinned at 14 and tau_min
            # keeps declining against a flat tau_median. If tau_argmin
            # wanders between registers, the divergence is NOT the
            # persistent single-register phenomenon SS49 describes.
            # 2026-09-10: this arm runs creation_qk_norm=True, so log_tau
            # is NOT REGISTERED (SS10.2/SS10.4: the clamped per-register
            # logit_scale REPLACES the temperature rather than multiplying
            # it). The monitored quantity therefore becomes
            #     sigma_k = min(exp(lambda_k), logit_scale_max)
            # instead of tau_k, and THE SIGN FLIPS: sharpening now shows up
            # as sigma rising toward the ceiling, not tau falling toward
            # zero. So watch sig_max, not sig_min.
            # Register_Temperature SS12 prediction 4, restated for this arm:
            # single-register dominance should be ABSENT, because no
            # register's scaled scores can exceed logit_scale_max and so
            # none can dominate the covariance by an order of magnitude.
            # sig_max == CREATION_LOGIT_SCALE_MAX exactly is the signal to
            # watch for: that register has hit the clamp, which zeroes its
            # gradient and freezes its sharpness permanently -- an
            # ABSORBING state (SS10.5), not a soft bound. One register
            # pinned there while the pool sits far below is this arm's
            # analogue of the SS49 runaway, and the fallback if it happens
            # is to project lambda back into the box after optim.step()
            # (like TAU_CREATE_MIN does) instead of clamping in forward.
            # Both branches are kept so this notebook still runs with
            # CREATION_QK_NORM=False.
            _tau_min = _tau_median = None
            _tau_argmin = None
            _sig_max = _sig_median = _sig_argmax = None
            try:
                _gate = model.creation_gate_qkv
                _ls = getattr(_gate, 'logit_scale', None)
                if _ls is not None:
                    _sig = _ls.detach().float().exp().clamp(
                        max=getattr(_gate, 'logit_scale_max', float('inf')))
                    _sig_argmax = int(torch.argmax(_sig))
                    _sig_max = float(_sig[_sig_argmax])
                    _sig_median = float(_sig.median())
                    _top_grp += f'sig_max={_sig_max:.3f}@r{_sig_argmax}  '
                else:
                    _lt = _gate.log_tau.detach().float()
                    _tau = _lt.exp().clamp(min=1e-4)
                    _tau_argmin = int(torch.argmin(_tau))
                    _tau_min = float(_tau[_tau_argmin])
                    _tau_median = float(_tau.median())
                    _top_grp += f'tau_min={_tau_min:.3f}@r{_tau_argmin}  '
            except Exception as _e:
                # e.g. a v1 creation protocol with neither parameter.
                if GRAD_SPIKE_DEBUG:
                    print(f'[warn] creation-gate scale monitor failed: {_e}')

            _rep_str = f'rep={accum_rep:.4f}  ' if REGISTER_REPULSION else ''
            _mem_alloc_gb = torch.cuda.memory_allocated() / 1e9
            _mem_resv_gb = torch.cuda.memory_reserved() / 1e9
            _mem_peak_gb = torch.cuda.max_memory_allocated() / 1e9
            torch.cuda.reset_peak_memory_stats()
            print(
                f'step {step+1:7d}/{TOTAL_STEPS}  '
                f'ntp={avg_ntp:.4f}  v_reg={avg_vreg:.4f}  fock_reg={avg_fock_reg:.4f}  '
                f'lr={lr_now:.2e}  grad={float(grad_norm):.2f}  {_rep_str}{_top_grp}'
                f'gamma={model.gamma.item():.3f}  alpha=[{alpha_str}]  '
                f'mem_alloc={_mem_alloc_gb:.1f}GB  mem_resv={_mem_resv_gb:.1f}GB  '
                f'mem_peak={_mem_peak_gb:.1f}GB  '
                f'{elapsed:.0f}s  (~{remaining/3600:.1f}h remaining)')
            _log_write(json.dumps({
                'step': step + 1, 'train_loss': avg_ntp, 'v_reg': avg_vreg,
                'fock_reg': avg_fock_reg,
                'lr': lr_now, 'grad_norm': float(grad_norm),
                'gamma': model.gamma.item(), 'xi_alphas': alphas,
                'reg_repulsion': accum_rep,
                'dc_ratio': (round(_dc_ratio, 4)
                             if _dc_ratio is not None and math.isfinite(_dc_ratio)
                             else _dc_ratio),
                'b_proj_sigma_max': (round(_bproj_sigma_max, 4)
                                     if _bproj_sigma_max is not None else None),
                'b_proj_sigma_by_bank': _bproj_sigma_by_bank,
                'tau_min': (round(_tau_min, 5) if _tau_min is not None else None),
                'tau_argmin': _tau_argmin,
                'tau_median': (round(_tau_median, 5)
                               if _tau_median is not None else None),
                'logit_scale_max_val': (round(_sig_max, 5)
                                        if _sig_max is not None else None),
                'logit_scale_argmax': _sig_argmax,
                'logit_scale_median': (round(_sig_median, 5)
                                       if _sig_median is not None else None),
                'mem_alloc_gb': round(_mem_alloc_gb, 3),
                'mem_reserved_gb': round(_mem_resv_gb, 3),
                'mem_peak_gb': round(_mem_peak_gb, 3),
                'elapsed_sec': elapsed, 'sec_per_step': sec_per_step,
            }) + '\n')

        if (step + 1) % EVAL_INTERVAL == 0:
            val_loss = evaluate()
            val_ppl = math.exp(val_loss)
            is_best = val_ppl < best_val_ppl
            if is_best:
                best_val_ppl = val_ppl
            elapsed = time.time() - t0
            marker = '*** NEW BEST ***' if is_best else ''
            print(f'>>> EVAL step {step+1:,}  val_loss={val_loss:.4f}  '
                  f'val_ppl={val_ppl:.2f}  best={best_val_ppl:.2f}  '
                  f'{marker}  ({elapsed:.0f}s)')
            _log_write(json.dumps({
                'step': step + 1, 'val_loss': val_loss,
                'val_ppl': val_ppl, 'best_ppl': best_val_ppl,
            }) + '\n')
            if is_best:
                save_checkpoint(step + 1, val_loss, tag_suffix='_best')

        if (step + 1) in set(CKPT_STEPS):
            if (step + 1) % EVAL_INTERVAL != 0:
                val_loss = evaluate()
                val_ppl = math.exp(val_loss)
            save_checkpoint(step + 1, val_loss)

        if PROBE_MAX_STEPS is not None and (step + 1) >= PROBE_MAX_STEPS:
            if (step + 1) % EVAL_INTERVAL != 0:
                val_loss = evaluate()
                val_ppl = math.exp(val_loss)
            save_checkpoint(step + 1, val_loss, tag_suffix='_probe_stop')
            print(f'\n[probe] PROBE_MAX_STEPS={PROBE_MAX_STEPS} reached at step '
                  f'{step+1:,}; stopping this side-by-side probe (val_ppl='
                  f'{val_ppl:.2f}). Saved {CKPT_PREFIX}_step{step+1}_probe_stop.pt')
            _log_write(json.dumps({
                'step': step + 1, 'event': 'probe_max_steps_stop',
                'val_ppl': val_ppl,
            }) + '\n')
            break

        if CAUSAL_PROBE_INTERVAL > 0 and (step + 1) % CAUSAL_PROBE_INTERVAL == 0:
            _cp_passed, _cp_delta = run_causal_probe(step + 1)
            _log_write(json.dumps({
                'step': step + 1,
                'causal_probe_passed': _cp_passed,
                'causal_probe_max_delta': _cp_delta,
            }) + '\n')

        if TRAINED_LEAK_PROBE_INTERVAL > 0 and (step + 1) % TRAINED_LEAK_PROBE_INTERVAL == 0:
            _tlp_result = run_trained_leak_probe(step + 1)
            _log_write(json.dumps(_tlp_result) + '\n')

    _log_fh[0].close()
    print(f'\nTraining complete. Best PPL: {best_val_ppl:.2f}')
    return total_steps


def save_manual_checkpoint(next_step_num, val_loss_placeholder=4.6):
    """Force-save current in-memory model/optimizer state under a
    distinct '_manual' filename, without waiting for the next scheduled or
    best checkpoint. `next_step_num` should be whatever run_training() most
    recently returned (i.e. the step training will resume from).

    This is only needed before an *actual* runtime restart (e.g. to pick
    up freshly-pulled notebook code) -- interrupting to inspect captures
    and calling run_training(next_step, TOTAL_STEPS) again no longer needs
    it (companion note SS37).
    """
    return save_checkpoint(next_step_num, val_loss_placeholder, tag_suffix='_manual')


# 2026-09-06: widened from `except KeyboardInterrupt:` alone (companion note
# SS42 follow-up). A genuine CUDA OOM (not an interrupt) used to propagate
# past this handler as a bare traceback -- no friendly resume message, and
# (worse) the crashed frame's locals stayed pinned via IPython's own
# sys.last_traceback until manually cleared. `step` itself is unaffected
# either way (it's `global`, so it's already valid in the notebook
# namespace the moment the exception is raised, whether or not it's caught
# here) -- but catching OOM explicitly means the recovery (clearing the
# stuck traceback + emptying the CUDA cache) happens automatically instead
# of requiring a manual `sys.last_traceback = None; gc.collect();
# torch.cuda.empty_cache()` dance every time.
try:
    next_step = run_training(resume_step, TOTAL_STEPS)
except (KeyboardInterrupt, torch.cuda.OutOfMemoryError) as e:
    print(f'\n[run_training] interrupted ({type(e).__name__}) at step '
          f'{step + 1:,}. Call run_training({step + 1}, TOTAL_STEPS) to '
          f'resume in-place (no rebuild / no checkpoint reload / no '
          f'runtime restart needed).')
    sys.last_traceback = None
    gc.collect()
    torch.cuda.empty_cache()
    next_step = step + 1
else:
    print(f'\n[run_training] finished cleanly; next_step={next_step} '
          f'(== TOTAL_STEPS means training is fully done).')

In [ ]:
# == Cell 6b: Stiffness diagnostic — how much is the CfC actually saving? ==
#
# The explicit (Verlet) layer step is stable only while omega*dt < 2, where
# omega = sqrt(K/m) is the local V_theta curvature seen by one coordinate of
# one token.  This probe measures the distribution of omega*dt across a real
# batch, so the instability can be observed *directly* rather than inferred
# from the gradient norm after the fact.
#
# Read it as: any mass above omega*dt = 2 is a coordinate the Verlet
# integrator is provably amplifying, and that the CfC propagator rotates
# instead.  Safe to run against any arm -- it temporarily borrows the
# harmonic linearisation even when training under 'verlet'.

import contextlib

def stiffness_report(mdl, x, dt=None):
    """Distribution of omega*dt over layers, tokens and dimensions."""
    dt = float(mdl.cfg.dt if dt is None else dt)
    if not hasattr(mdl.V_theta, 'harmonic_terms'):
        raise RuntimeError('V_theta has no harmonic_terms(); need the '
                           'anisotropic Gaussian family.')

    seen = []
    _orig = mdl.V_theta.harmonic_terms

    def _recording(xis, h):
        k_diag, s = _orig(xis, h)
        seen.append(k_diag.detach().float().flatten().cpu())
        return k_diag, s

    _saved = (mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force)
    mdl.V_theta.harmonic_terms = _recording
    mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force = 'baoab_cfc', True
    try:
        was_training = mdl.training
        mdl.eval()
        with torch.enable_grad():
            mdl(x)
    finally:
        mdl.V_theta.harmonic_terms = _orig
        mdl.cfg.integrator, mdl.cfg.vtheta_analytic_force = _saved
        if was_training:
            mdl.train()

    k = torch.cat(seen)
    m = float(mdl.compute_mass(x).mean())
    wdt = (k.clamp(min=0) / m).sqrt() * dt
    n_total = int(wdt.numel())
    # The tails are what matter, so keep the exact max but subsample for the
    # quantiles: torch.quantile refuses inputs beyond ~16M elements, and
    # L*B*T*d reaches that at d=384, L=16 with a large auto-probed batch.
    wdt_max = float(wdt.max())
    if n_total > 4_000_000:
        idx = torch.randint(0, n_total, (4_000_000,))
        wdt_q = wdt[idx]
    else:
        wdt_q = wdt
    q = torch.tensor([0.5, 0.9, 0.99, 0.999])
    qs = torch.quantile(wdt_q.double(), q.double()).float()
    qs = torch.cat([qs, torch.tensor([wdt_max])])
    return {
        'n_samples': n_total,
        'mean_mass': m,
        'median': float(qs[0]), 'p90': float(qs[1]), 'p99': float(qs[2]),
        'p999': float(qs[3]), 'max': float(qs[4]),
        'frac_unstable': float((wdt > 2.0).float().mean()),
        'frac_marginal': float((wdt > 1.0).float().mean()),
    }


_rng_s = np.random.default_rng(0)
_xb, _ = get_batch(train_ids, min(BATCH_SIZE, 4), BLOCK_SIZE, _rng_s)
_rep = stiffness_report(model, torch.from_numpy(_xb).to(DEVICE))

print(f'omega*dt over {_rep["n_samples"]:,} (layer, token, dim) samples '
      f'  [mean mass {_rep["mean_mass"]:.3f}]')
print(f'  median {_rep["median"]:.4f}   p90 {_rep["p90"]:.4f}   '
      f'p99 {_rep["p99"]:.4f}   p99.9 {_rep["p999"]:.4f}   '
      f'max {_rep["max"]:.4f}')
print(f'  fraction with omega*dt > 1 (marginal): {_rep["frac_marginal"]:.3e}')
print(f'  fraction with omega*dt > 2 (Verlet-unstable): '
      f'{_rep["frac_unstable"]:.3e}')
if _rep['max'] > 2.0:
    print('  => the explicit step is UNSTABLE on some coordinates right now; '
          'these are exactly what baoab_cfc integrates exactly instead.')
else:
    print('  => no coordinate exceeds the explicit stability bound at this '
          'checkpoint (re-run later in training: wells sharpen over time).')


In [ ]:
# == Cell 6b-2: sigma_max(B_k)^2 diagnostic -- for bracketing PRECISION_LR_MAX ==
#
# stiffness_report (Cell 6b) reports omega*dt from k_diag, the DIAGONAL
# harmonic model -- exactly what baoab_cfc already integrates exactly, so
# it is immune to unbounded B_k growth by construction (SS24, companion
# note).  The channel that is NOT immune is the anisotropic low-rank
# correction B_k B_k^T, which baoab_cfc still demotes to an explicit kick
# (SS28.2/SS28.6) and which PRECISION_LR_MAX (SS29.3) / baoab_cfc_lowrank
# (SS29.2) exist to fix.  This cell measures that raw quantity directly --
# sigma_max(B_k)^2, per well, per xi-channel, per layer, BEFORE it is
# mixed with a_k or gated by the Gaussian bump g_k -- so a
# PRECISION_LR_MAX budget can be bracketed (SS31.4) between this
# checkpoint's distribution and a spike-regime checkpoint's, instead of
# guessed.  Mirrors the sigma_lr_* percentiles added to SCAF's
# StiffnessProbe (semsimula-scaf, SS31.3).
#
# Usage: load the checkpoint of interest into `model` (the existing
# reload/checkpoint-loading cell already does this), then re-run this
# cell.  Comparing a healthy checkpoint (e.g. this run's step-27,000 best)
# against a spike-regime one (the `_prereload` snapshots saved at the
# hard-trigger steps) is the SS31.2 bracketing protocol.

def sigma_lr_report(mdl, x):
    """Distribution of the raw sigma_max(B_k)^2 over layers, wells and
    xi-channels -- the quantity PRECISION_LR_MAX caps directly. Deliberately
    NOT combined with a_k or g_k (unlike stiffness_report's omega*dt):
    bracketing a PRECISION_LR_MAX budget needs the raw per-well spectral
    norm, not a quantity already mixed with the bump weight or the
    diagonal precision.
    """
    if not hasattr(mdl.V_theta, 'context_components'):
        raise RuntimeError('V_theta has no context_components(); need the '
                           'anisotropic Gaussian family.')

    seen = []
    _orig = mdl.V_theta.context_components

    def _recording(xis):
        comps = _orig(xis)
        # comps: list of (mu, a, w, B) tuples, one per xi-channel (see
        # AnisotropicMultiContextGaussianVTheta.context_components).
        for (_mu, _a, _w, B) in comps:
            if B.shape[-1] == 0:
                continue
            # sigma_max(B_k)^2 = (largest singular value of B_k)^2, taken
            # from the SVD of B_k directly rather than eigvalsh(B_k^T B_k):
            # forming the Gram squares the condition number and can make
            # the symmetric-eigen driver fail to converge on degenerate
            # wells (same failure mode fixed in cfc_baoab.lowrank_modes).
            sigma_max_sq = torch.linalg.svdvals(B)[..., 0] ** 2  # (..., K)
            seen.append(sigma_max_sq.detach().float().flatten().cpu())
        return comps

    mdl.V_theta.context_components = _recording
    try:
        was_training = mdl.training
        mdl.eval()
        with torch.enable_grad():
            mdl(x)
    finally:
        mdl.V_theta.context_components = _orig
        if was_training:
            mdl.train()

    if not seen:
        return {'n_samples': 0}
    s = torch.cat(seen)
    n_total = int(s.numel())
    s_max = float(s.max())
    if n_total > 4_000_000:
        idx = torch.randint(0, n_total, (4_000_000,))
        s_q = s[idx]
    else:
        s_q = s
    q = torch.tensor([0.5, 0.9, 0.99, 0.999])
    qs = torch.quantile(s_q.double(), q.double()).float()
    qs = torch.cat([qs, torch.tensor([s_max])])
    return {
        'n_samples': n_total,
        'p50': float(qs[0]), 'p90': float(qs[1]), 'p99': float(qs[2]),
        'p999': float(qs[3]), 'max': float(qs[4]),
    }


_rng_slr = np.random.default_rng(0)
_xb_slr, _ = get_batch(train_ids, min(BATCH_SIZE, 4), BLOCK_SIZE, _rng_slr)
_rep_slr = sigma_lr_report(model, torch.from_numpy(_xb_slr).to(DEVICE))

if _rep_slr['n_samples'] == 0:
    print('sigma_lr_report: no low-rank wells found (rank=0 or no '
          'context_components on this V_theta).')
else:
    print(f'sigma_max(B_k)^2 over {_rep_slr["n_samples"]:,} '
          f'(layer, xi-channel, well) samples:')
    print(f'  p50 {_rep_slr["p50"]:.3f}   p90 {_rep_slr["p90"]:.3f}   '
          f'p99 {_rep_slr["p99"]:.3f}   p99.9 {_rep_slr["p999"]:.3f}   '
          f'max {_rep_slr["max"]:.3f}')
    print('  => record this alongside the checkpoint step; compare a '
          'healthy checkpoint against a `_prereload` (spike-regime) '
          'snapshot to bracket PRECISION_LR_MAX per companion-note SS31.4 '
          '(set it above the healthy p95-p99, below the spike-regime '
          'p99/max, then bias slightly upward since the Frobenius cap is '
          'conservative by up to a factor ANISO_RANK).')


In [ ]:
# == Cell 6b-3: bracket PRECISION_LR_MAX -- healthy vs the captured spike
# bundles (companion note SS41.7 item 1 / SS31.2-SS31.4 protocol) ==
#
# sigma_lr_report (Cell 6b-2) needs `model` to already hold the weights of
# interest. Rather than trust "whatever happens to be loaded right now" for
# the spike-regime reading, this loads each `_spikebatch.pt` bundle's own
# `model_state_dict` -- bit-identical to what replay_precision_cap_ablation /
# replay_integrator_ablation (Cell 6d) actually replayed -- and the healthy
# `_best.pt` checkpoint, one at a time, restoring the live model's weights
# in a `finally` block. Same non-pollution invariant as the Cell 6d replay
# helpers, so this is safe to run against the live, still-resumable
# training state.
#
# Requires Cell 6b-2 to have run at least once (defines sigma_lr_report).

def bracket_precision_lr_max(step_tags=(47116, 48507, 48917), mdl=None,
                              healthy_ckpt_name=None, n_batch=4):
    mdl = mdl if mdl is not None else model
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _rng_b = np.random.default_rng(0)
    _xb_b, _ = get_batch(train_ids, min(BATCH_SIZE, n_batch), BLOCK_SIZE, _rng_b)
    _x_b = torch.from_numpy(_xb_b).to(DEVICE)

    reports = {}
    try:
        # -- healthy baseline --
        _healthy_path = (CKPT_DIR / healthy_ckpt_name if healthy_ckpt_name
                          else CKPT_DIR / f'{CKPT_PREFIX}_best.pt')
        _hd = torch.load(_healthy_path, map_location='cpu', weights_only=False)
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in _hd['model_state_dict'].items()},
            strict=False)
        reports[f'healthy (step {_hd.get("step", "?"):,})'] = sigma_lr_report(mdl, _x_b)
        del _hd

        # -- each captured spike bundle --
        for step_tag in step_tags:
            _path = CKPT_DIR / f'{CKPT_PREFIX}_step{step_tag}_spikebatch.pt'
            _bd = torch.load(_path, map_location='cpu', weights_only=False)
            mdl.load_state_dict(
                {k: v.to(DEVICE) for k, v in _bd['model_state_dict'].items()},
                strict=False)
            reports[f'spike step {step_tag}'] = sigma_lr_report(mdl, _x_b)
            del _bd
    finally:
        mdl.load_state_dict(_saved_sd)

    print(f'{"state":<22} {"p50":>9} {"p90":>9} {"p99":>9} {"p99.9":>9} {"max":>12}')
    for label, r in reports.items():
        print(f'{label:<22} {r["p50"]:9.3f} {r["p90"]:9.3f} {r["p99"]:9.3f} '
              f'{r["p999"]:9.3f} {r["max"]:12.3f}')
    print('\n=> per companion-note SS31.4: set PRECISION_LR_MAX above the '
          'healthy row\'s p90-p99, below the spike rows\' p99/max, then bias '
          f'slightly upward (the Frobenius cap in _bound_lowrank is '
          f'conservative by up to a factor ANISO_RANK={ANISO_RANK}).')
    return reports


_brackets = bracket_precision_lr_max()


In [ ]:
# == Cell 6b-4: B_k SINGULAR-VALUE SPECTRUM -- the ANISO_RANK decision ====
#
# sigma_lr_report (Cell 6b-2) computes torch.linalg.svdvals(B) and then
# keeps only [..., 0] -- sigma_max. The rest of the spectrum is computed
# and thrown away, and it is exactly what decides two open questions:
#
#  Q1 (ANISO_RANK). Is rank 4 saturated or degenerate? _bound_lowrank caps
#      ||B_k||_F <= sqrt(PRECISION_LR_MAX)=1.0, and with ambient uncapped
#      sigma_max(B_k)^2 ~ 283 that cap is BINDING HARD (tanh(17/1) ~ 1), so
#      sum_i sigma_i^2 is pinned at 1.0 REGARDLESS of rank. Rank therefore
#      stops being an "amount of curvature" knob and becomes a
#      DISTRIBUTION knob. Under a fixed Frobenius budget:
#          flat spectrum  -> sigma_max^2 = 1/r   (rank 8: 0.125)
#          degenerate     -> sigma_max^2 = 1     (rank 8 == rank 4 == rank 1)
#      Since SS3 derives spike magnitude ~ sigma_max(B_k)^2, a spread
#      spectrum means higher rank is SAFER, not riskier. But rank is only
#      an upper bound on directions used -- if the model drives the whole
#      budget into sigma_1, more columns buy nothing and cost 23.7M params
#      (+31% model). The realized spectrum settles it.
#
#  Q2 (spike mechanism). Under a binding Frobenius cap, sigma_max^2 can
#      only grow by CONCENTRATION. So a V_theta-led spike may literally be
#      a moment of spectral collapse -- the well dumping its whole budget
#      into one direction. Comparing a healthy checkpoint's spectrum
#      against a spike bundle's tests that directly.
#
# METRIC: participation ratio  PR = (sum_i s_i^2)^2 / sum_i s_i^4,
# the standard effective-rank measure. PR = 1 -> rank-1 (all budget in one
# direction); PR = r -> perfectly flat. Reported per (layer, channel, well)
# then summarised as percentiles.
#
# DECISION RULE, stated in advance (ANISO_RANK=4):
#   PR p50 >= 3.0   -> budget saturated; rank 8 has a real case AND would
#                      halve sigma_max^2 under the same cap. Both axes agree.
#   PR p50 <= 2.0   -> the model is not using the rank it has. Rank 8 is
#                      wasted parameters; the cheaper fix is a spectral
#                      FLATNESS incentive at rank 4.
#   2.0 < p50 < 3.0 -> ambiguous; weight by the Q2 result and by cost.
#
# This is weight-space + one neutral batch: no spike bundle required for
# Q1. Use the CURRENT best checkpoint for Q1; use spike bundles for Q2.

def sigma_lr_spectrum_report(mdl, x):
    """Full singular-value spectrum of B_k per (layer, xi-channel, well).

    Same hook mechanism as sigma_lr_report (Cell 6b-2) -- monkeypatches
    context_components so it sees the REALISED B (post-_bound_lowrank),
    i.e. the same B the forward pass used -- but keeps every singular
    value instead of just the largest.

    Returns percentiles of the participation ratio, the mean normalised
    spectrum sigma_i/sigma_1, and the Frobenius norm distribution (so you
    can confirm the cap is actually binding, which is what makes rank a
    pure redistribution knob).
    """
    if not hasattr(mdl.V_theta, 'context_components'):
        raise RuntimeError('V_theta has no context_components(); need the '
                           'anisotropic Gaussian family.')

    svals = []
    _orig = mdl.V_theta.context_components

    def _recording(xis):
        comps = _orig(xis)
        for (_mu, _a, _w, B) in comps:
            if B.shape[-1] == 0:
                continue
            # (..., K, d, r) -> (..., K, r), descending
            s = torch.linalg.svdvals(B)
            svals.append(s.detach().float().reshape(-1, s.shape[-1]).cpu())
        return comps

    mdl.V_theta.context_components = _recording
    try:
        was_training = mdl.training
        mdl.eval()
        with torch.enable_grad():
            mdl(x)
    finally:
        mdl.V_theta.context_components = _orig
        if was_training:
            mdl.train()

    if not svals:
        return {'n_samples': 0}

    s = torch.cat(svals).double()          # (N, r)
    r = s.shape[-1]
    s2 = s ** 2
    fro2 = s2.sum(-1)                      # ||B_k||_F^2 = sum sigma_i^2
    # participation ratio in [1, r]; guard the numerically-dead wells
    pr = (fro2 ** 2) / (s2 ** 2).sum(-1).clamp(min=1e-300)
    # mean normalised spectrum, averaged over wells with nonzero sigma_1
    s1 = s[:, :1].clamp(min=1e-300)
    spec = (s / s1).mean(0)

    q = torch.tensor([0.05, 0.5, 0.95], dtype=torch.float64)
    pr_q = torch.quantile(pr, q)
    fro_q = torch.quantile(fro2.sqrt(), q)
    return {
        'n_samples': int(s.shape[0]), 'rank': r,
        'pr_p05': float(pr_q[0]), 'pr_p50': float(pr_q[1]),
        'pr_p95': float(pr_q[2]), 'pr_mean': float(pr.mean()),
        'fro_p05': float(fro_q[0]), 'fro_p50': float(fro_q[1]),
        'fro_p95': float(fro_q[2]),
        'spectrum': [float(v) for v in spec],      # sigma_i / sigma_1
        'sigma_max_sq_p50': float(torch.quantile(s2[:, 0], 0.5)),
    }


def _print_spectrum(label, rep):
    if rep.get('n_samples', 0) == 0:
        print(f'{label:<26} (no low-rank wells -- rank=0?)')
        return
    r = rep['rank']
    spec = '  '.join(f'{v:.3f}' for v in rep['spectrum'])
    print(f'{label:<26} PR p50={rep["pr_p50"]:.2f}/{r}  '
          f'(p05={rep["pr_p05"]:.2f} p95={rep["pr_p95"]:.2f})   '
          f'||B||_F p50={rep["fro_p50"]:.3f}   '
          f'sigma_max^2 p50={rep["sigma_max_sq_p50"]:.4f}')
    print(f'{"":<26} mean sigma_i/sigma_1: [{spec}]')


def _resolve_bundle_path(tag, suffix, archive_dir):
    """Find `{CKPT_PREFIX}_step{tag}_{suffix}.pt`, checking the live
    CKPT_DIR first and falling back to the given archive dir if the ring
    has already evicted it. The archive copy is byte-identical (Cell 1c
    writes it via shutil.copy2), so there is no preference between them
    beyond "whichever still exists" -- returns (path, 'live'|'archive')
    or (None, None).
    Recomputes GDRIVE_ROOT-relative archive paths fresh rather than
    trusting Cell 1c's globals to still be bound, in case this cell is
    ever re-run after a kernel restart without re-running Cell 1c first.
    """
    fname = f'{CKPT_PREFIX}_step{tag}_{suffix}.pt'
    live = CKPT_DIR / fname
    if live.exists():
        return live, 'live'
    archived = archive_dir / fname
    if archived.exists():
        return archived, 'archive'
    return None, None


def spectrum_across_checkpoints(step_tags=(), mdl=None, healthy_ckpt_name=None,
                                 n_batch=4, include_prereload=()):
    """Q1 + Q2 in one pass, on a single fixed neutral batch.

    step_tags : spikebatch bundles (for Q2). Pass () for the Q1-only run.
    include_prereload : _prereload.pt step tags -- weights-only snapshots
        from watchdog reloads; they carry model_state_dict so they work
        here even though they can NOT be replayed (no batch/RNG).

    Same non-pollution invariant as bracket_precision_lr_max: the live
    model's weights are restored in a finally block, so this is safe to
    run against a resumable training session.

    Bundle lookup checks CKPT_DIR first, then falls back to Cell 1c's
    spikebatch_archive / prereload_archive -- the live ring rotates
    (SPIKEBATCH_SNAPSHOT_MAX_KEEP=12, PRERELOAD_SNAPSHOT_MAX_KEEP=5) on
    every new capture, independent of whether YOU have gotten around to
    analysing an older one yet, so a bundle worth archiving is worth
    finding there too rather than silently skipped.
    """
    mdl = mdl if mdl is not None else model
    _saved_sd = _copy.deepcopy(mdl.state_dict())
    _rng_s = np.random.default_rng(0)
    _xb_s, _ = get_batch(train_ids, min(BATCH_SIZE, n_batch), BLOCK_SIZE, _rng_s)
    _x_s = torch.from_numpy(_xb_s).to(DEVICE)

    reports = {}
    try:
        _hp = (CKPT_DIR / healthy_ckpt_name if healthy_ckpt_name
               else CKPT_DIR / f'{CKPT_PREFIX}_best.pt')
        _hd = torch.load(_hp, map_location='cpu', weights_only=False)
        mdl.load_state_dict(
            {k: v.to(DEVICE) for k, v in _hd['model_state_dict'].items()},
            strict=False)
        reports[f'BEST (step {_hd.get("step", "?"):,})'] = \
            sigma_lr_spectrum_report(mdl, _x_s)
        del _hd

        _sb_archive = GDRIVE_ROOT / 'spikebatch_archive'
        for tag in step_tags:
            _p, _src = _resolve_bundle_path(tag, 'spikebatch', _sb_archive)
            if _p is None:
                print(f'  [skip] no spikebatch bundle for step {tag} in '
                      f'CKPT_DIR or spikebatch_archive')
                continue
            if _src == 'archive':
                print(f'  [archive] step {tag} spikebatch -- evicted from '
                      f'the live ring, loaded from spikebatch_archive')
            _bd = torch.load(_p, map_location='cpu', weights_only=False)
            mdl.load_state_dict(
                {k: v.to(DEVICE) for k, v in _bd['model_state_dict'].items()},
                strict=False)
            reports[f'spike {tag} ({_bd.get("pre_clip_grad_norm", "?")})'] = \
                sigma_lr_spectrum_report(mdl, _x_s)
            del _bd

        _pr_archive = GDRIVE_ROOT / 'prereload_archive'
        for tag in include_prereload:
            _p, _src = _resolve_bundle_path(tag, 'prereload', _pr_archive)
            if _p is None:
                print(f'  [skip] no prereload for step {tag} in CKPT_DIR '
                      f'or prereload_archive')
                continue
            if _src == 'archive':
                print(f'  [archive] step {tag} prereload -- evicted from '
                      f'the live ring, loaded from prereload_archive')
            _pd = torch.load(_p, map_location='cpu', weights_only=False)
            mdl.load_state_dict(
                {k: v.to(DEVICE) for k, v in _pd['model_state_dict'].items()},
                strict=False)
            reports[f'prereload {tag}'] = sigma_lr_spectrum_report(mdl, _x_s)
            del _pd
    finally:
        mdl.load_state_dict(_saved_sd)

    print(f'\nB_k spectrum -- ANISO_RANK={ANISO_RANK}, '
          f'PRECISION_LR_MAX={PRECISION_LR_MAX} '
          f'(||B||_F cap = {None if PRECISION_LR_MAX is None else PRECISION_LR_MAX ** 0.5})')
    print('=' * 78)
    for label, rep in reports.items():
        _print_spectrum(label, rep)
    _best = next(iter(reports.values()), {})
    if _best.get('n_samples', 0):
        _pr, _r = _best['pr_p50'], _best['rank']
        print('=' * 78)
        if _pr >= 0.75 * _r:
            print(f'=> PR p50 = {_pr:.2f} of {_r}: budget SATURATED. Rank '
                  f'{2*_r} has a real case, and would also lower sigma_max^2 '
                  f'under the same Frobenius cap.')
        elif _pr <= 0.5 * _r:
            print(f'=> PR p50 = {_pr:.2f} of {_r}: budget NOT used. Rank '
                  f'{2*_r} would be wasted params; prefer a spectral '
                  f'flatness incentive at rank {_r} (or rank {_r//2}).')
        else:
            print(f'=> PR p50 = {_pr:.2f} of {_r}: ambiguous band. Weigh the '
                  f'spike-vs-best comparison above and the parameter cost.')
    return reports


# Q1 only (no bundles needed). Add spike tags for Q2, e.g.:
#   spectrum_across_checkpoints(step_tags=(81647, 82660, 81393))
_spec_reports = spectrum_across_checkpoints()


In [ ]:
# == Cell 6c: Bottleneck profile — GPU-compute vs CPU/launch ==
#
# Interrupt the training cell first.  This does not step the optimizer
# and does not change weights.  Takes ~1–2 minutes on the live A100.
#
# Three independent measurements vote on why the step is ~22s:
#   (1) GPU-compute-bound  — H100 extra FLOPs/bandwidth can help
#   (2) CPU/launch-bound   — a faster GPU buys almost nothing
#
# Prefers scaleup/debug/cfc_step_bottleneck_profile.py when present
# (pull the paper repo).  Falls back to the copy shipped in this cell
# so a stale Colab clone still works.

import importlib.util
import sys
from pathlib import Path

def _load_bottleneck_mod():
    candidates = []
    if 'REPO_ROOT' in globals():
        candidates.append(
            REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
            / 'debug' / 'cfc_step_bottleneck_profile.py')
    candidates += [
        Path.cwd() / 'debug' / 'cfc_step_bottleneck_profile.py',
        Path.cwd() / 'scaleup' / 'debug' / 'cfc_step_bottleneck_profile.py',
    ]
    for p in candidates:
        if p.exists():
            spec = importlib.util.spec_from_file_location(
                'cfc_step_bottleneck_profile', p)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            print(f'[bottleneck] loaded {p}')
            return mod
    return None

_bp = _load_bottleneck_mod()
if _bp is None:
    print('[bottleneck] debug module not on disk; using the copy in this cell')
    # --- embedded copy of cfc_step_bottleneck_profile.py (keep in sync) ---
    import math, os, re, subprocess, threading, time
    from typing import Callable, Iterable

    def _median(xs):
        ys = sorted(xs)
        n = len(ys)
        if n == 0:
            return float('nan')
        if n % 2:
            return ys[n // 2]
        return 0.5 * (ys[n // 2 - 1] + ys[n // 2])

    def _sync():
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    def _time_cuda(fn, n_warmup=1, n_repeat=3):
        for _ in range(n_warmup):
            fn(); _sync()
        samples = []
        for _ in range(n_repeat):
            _sync()
            t0 = time.perf_counter()
            fn(); _sync()
            samples.append(time.perf_counter() - t0)
        return _median(samples), samples

    def _classify_scale(ratio_hi_over_lo, hi, lo):
        expected = hi / lo
        slope = ((ratio_hi_over_lo - 1.0) / (expected - 1.0)
                 if expected > 1 else 0.0)
        if slope >= 0.70:
            return 'compute', slope
        if slope <= 0.25:
            return 'launch', slope
        return 'mixed', slope

    def _classify_overlap(efficiency):
        if efficiency <= 0.25:
            return 'compute'
        if efficiency >= 0.65:
            return 'launch'
        return 'mixed'

    def _classify_util(mean_util):
        if mean_util >= 70:
            return 'compute'
        if mean_util <= 35:
            return 'launch'
        return 'mixed'

    def _poll_smi(samples, stop_evt, interval=0.05):
        while not stop_evt.is_set():
            try:
                out = subprocess.check_output(
                    ['nvidia-smi',
                     '--query-gpu=utilization.gpu,utilization.memory',
                     '--format=csv,noheader,nounits'],
                    text=True, timeout=2,
                ).strip().splitlines()[0]
                gpu_u, mem_u = [float(x.strip()) for x in out.split(',')[:2]]
                samples.append((gpu_u, mem_u))
            except Exception:
                return
            stop_evt.wait(interval)

    def _parse_self_cuda_seconds(table):
        m = re.search(
            r'Self (?:CUDA|device) time total:\s*([0-9.]+)\s*(us|ms|s)',
            table, flags=re.I)
        if not m:
            return None
        val = float(m.group(1))
        return val * {'us': 1e-6, 'ms': 1e-3, 's': 1.0}[m.group(2).lower()]

    def _calibrate_gemm_count(device, target_s, dim=4096):
        a = torch.randn(dim, dim, device=device, dtype=torch.float32)
        b = torch.randn(dim, dim, device=device, dtype=torch.float32)
        _sync()
        t0 = time.perf_counter()
        for _ in range(4):
            _ = a @ b
        _sync()
        per = max((time.perf_counter() - t0) / 4.0, 1e-4)
        n = min(max(4, int(math.ceil(target_s / per))), 400)
        return a, b, n, per

    def _enqueue_gemms(a, b, n, stream):
        with torch.cuda.stream(stream):
            acc = a
            for _ in range(n):
                acc = acc @ b
            stream_result = acc.sum()
        return stream_result

    def run_bottleneck_profile(
        model, *, forward_fn, make_batch, batch_size, grad_accum,
        batch_sizes=None, n_warmup=1, n_repeat=3, device='cuda',
        profile_kernels=True,
    ):
        if not torch.cuda.is_available():
            raise RuntimeError('Needs CUDA; CPU cannot split compute vs launch.')
        model.train()
        sizes = list(batch_sizes) if batch_sizes is not None else [1, 2, batch_size]
        sizes = sorted({s for s in sizes if 1 <= s <= batch_size})
        if batch_size not in sizes:
            sizes.append(batch_size); sizes.sort()
        cached = {}
        for bs in sizes:
            x, y = make_batch(bs)
            cached[bs] = (x.detach().clone(), y.detach().clone())

        def microbatch(bs, zero=True):
            x, y = cached[bs]
            loss = forward_fn(x, y)
            loss.backward()
            if zero:
                model.zero_grad(set_to_none=True)
            return loss

        print('=' * 64)
        print('CfC step bottleneck profile')
        cfg = getattr(model, 'cfg', None)
        print(f'  integrator={getattr(cfg, "integrator", "?")}  '
              f'd={getattr(cfg, "d", "?")}  L={getattr(cfg, "L", "?")}  '
              f'bs={batch_size}  accum={grad_accum}')
        print('  interrupt training first; this does not step the optimizer')
        print('=' * 64)
        votes = []

        print('\n[1] Batch scaling (one microbatch, fwd+bwd)')
        scale = {}
        for bs in sizes:
            med, samples = _time_cuda(
                lambda bs=bs: microbatch(bs), n_warmup=n_warmup, n_repeat=n_repeat)
            scale[bs] = med
            extra = (f'   ({bs}/{sizes[0]} = {med / scale[sizes[0]]:.2f})'
                     if bs != sizes[0] else '')
            samp = ' '.join(f'{s:.2f}' for s in samples)
            print(f'  bs={bs:<3d}  median {med:6.2f}s  samples [{samp}]{extra}')
        lo, hi = sizes[0], sizes[-1]
        ratio = scale[hi] / scale[lo] if scale[lo] > 0 else float('inf')
        scale_cls, slope = _classify_scale(ratio, hi, lo)
        votes.append(scale_cls)
        print(f'  linear slope captured: {slope:.2f}  '
              f'(1.0 = time ∝ batch,  0.0 = time flat)')
        print(f'  => {scale_cls.upper()}')

        print(f'\n[2] Side-stream GEMM overlap (bs={batch_size})')
        T_step = scale[batch_size]
        a, b, n_gemm, per = _calibrate_gemm_count(device, T_step)
        print(f'  calibrated {n_gemm} x {a.shape[0]} GEMMs  (~{per*1000:.1f} ms each)')

        def gemm_pile():
            side = torch.cuda.Stream()
            res = _enqueue_gemms(a, b, n_gemm, side)
            torch.cuda.current_stream().wait_stream(side)
            return res

        T_gemm, _ = _time_cuda(gemm_pile, n_warmup=0, n_repeat=2)
        print(f'  T_gemm alone  {T_gemm:.2f}s')

        def both():
            side = torch.cuda.Stream()
            res = _enqueue_gemms(a, b, n_gemm, side)
            microbatch(batch_size)
            torch.cuda.current_stream().wait_stream(side)
            return res

        T_both, _ = _time_cuda(both, n_warmup=0, n_repeat=2)
        overlap = (T_step + T_gemm - T_both) / min(T_step, T_gemm)
        overlap = max(0.0, min(1.2, overlap))
        overlap_cls = _classify_overlap(overlap)
        votes.append(overlap_cls)
        print(f'  T_step={T_step:.2f}s  T_gemm={T_gemm:.2f}s  T_both={T_both:.2f}s')
        print(f'  overlap efficiency = {overlap:.2f}  '
              f'(1.0 = full overlap / idle SMs,  0.0 = no spare SMs)')
        print(f'  => {overlap_cls.upper()}')
        del a, b

        print(f'\n[3] nvidia-smi during one full step ({grad_accum} microbatches)')
        smi, stop_evt = [], threading.Event()
        th = threading.Thread(target=_poll_smi, args=(smi, stop_evt), daemon=True)
        th.start(); time.sleep(0.15)

        def full_step():
            model.zero_grad(set_to_none=True)
            for _ in range(grad_accum):
                microbatch(batch_size, zero=False)
            model.zero_grad(set_to_none=True)

        _sync(); t0 = time.perf_counter(); full_step(); _sync()
        T_full = time.perf_counter() - t0
        stop_evt.set(); th.join(timeout=2.0)
        mean_u = None
        if smi:
            gpu_u = [g for g, _ in smi]
            mem_u = [m for _, m in smi]
            mean_u = sum(gpu_u) / len(gpu_u)
            p50_u = _median(gpu_u)
            p90_u = sorted(gpu_u)[max(0, int(0.9 * (len(gpu_u) - 1)))]
            util_cls = _classify_util(mean_u)
            votes.append(util_cls)
            print(f'  wall {T_full:.1f}s   samples={len(smi)}  '
                  f'GPU util mean {mean_u:.0f}%  p50 {p50_u:.0f}%  '
                  f'p90 {p90_u:.0f}%  mem-util mean {sum(mem_u)/len(mem_u):.0f}%')
            print(f'  => {util_cls.upper()}')
        else:
            print('  nvidia-smi unavailable; skipping this vote')
            print(f'  wall {T_full:.1f}s for the full step')

        if profile_kernels:
            print('\n[4] Profiler dump (one microbatch, supporting only)')
            try:
                from torch.profiler import ProfilerActivity, profile
                _sync(); t0 = time.perf_counter()
                with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
                             record_shapes=False, with_stack=False) as prof:
                    microbatch(batch_size); _sync()
                wall = time.perf_counter() - t0
                try:
                    table = prof.key_averages().table(
                        sort_by='self_cuda_time_total', row_limit=15)
                except Exception:
                    table = prof.key_averages().table(
                        sort_by='self_device_time_total', row_limit=15)
                self_cuda = _parse_self_cuda_seconds(table)
                if self_cuda is not None and wall > 0:
                    print(f'  wall {wall:.2f}s   Self-CUDA {self_cuda:.2f}s   '
                          f'busy={self_cuda / wall:.0%}')
                print(table)
            except Exception as exc:
                print(f'  profiler skipped: {type(exc).__name__}: {exc}')

        counts = {k: votes.count(k) for k in ('compute', 'launch', 'mixed')}
        if counts['compute'] > counts['launch'] and counts['compute'] >= 2:
            verdict = 'compute'
        elif counts['launch'] > counts['compute'] and counts['launch'] >= 2:
            verdict = 'launch'
        else:
            verdict = 'mixed'
        print('\n' + '=' * 64)
        if verdict == 'compute':
            print('VERDICT: (1) GPU-compute-bound')
            print('  The SMs are busy.  H100 extra FLOPs / HBM bandwidth can')
            print('  plausibly cut wall time by ~1.5-2.5x at the same 4x8.')
        elif verdict == 'launch':
            print('VERDICT: (2) CPU / launch-bound')
            print('  The GPU is idle between many small kernels.  A faster')
            print('  GPU buys almost nothing (maybe 1.0-1.2x).  The real')
            print('  levers are fewer sequential launches: cut GRAD_ACCUM,')
            print('  shrink ANISO_RANK, or fuse the per-layer Python step.')
        else:
            print('VERDICT: mixed — both (1) and (2) contribute')
            print('  H100 may help, but not by 2x.  Expect something closer')
            print('  to 1.2-1.6x unless launches are also reduced.')
        print(f'  votes: {votes}')
        print('=' * 64)
        model.zero_grad(set_to_none=True)
        return {'verdict': verdict, 'votes': votes}

    def run_from_notebook():
        import numpy as np
        rng = np.random.default_rng(12345)
        if not bool(globals().get('REGISTER_REPULSION', False)):
            def forward_fn(x, y):
                loss, *_ = forward_with_vreg(
                    x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                return loss
        else:
            def forward_fn(x, y):
                loss, *_ = forward_with_vreg(
                    x, y, LAMBDA_V, LAMBDA_FOCK_REG, FOCK_REG_EPS)
                return loss + model.pop_repulsion_loss()

        def make_batch(bs):
            xb, yb = get_batch(train_ids, bs, BLOCK_SIZE, rng)
            return (torch.from_numpy(xb).to(DEVICE),
                    torch.from_numpy(yb).to(DEVICE))

        return run_bottleneck_profile(
            model, forward_fn=forward_fn, make_batch=make_batch,
            batch_size=int(BATCH_SIZE), grad_accum=int(GRAD_ACCUM),
            device=str(DEVICE))

    class _M: pass
    _bp = _M()
    _bp.run_from_notebook = run_from_notebook

_bp.run_from_notebook()


## Fock v2.1 component diagnosticsStandalone probe -- safe to run any time against the live model or afreshly loaded checkpoint. It answers two questions:1. **Structural health** -- is each Fock piece being *used well*?2. **PPL attribution** -- how much does each piece actually *buy*?

In [ ]:
# == Cell 7: Component diagnostics =====================================
import torch
_bp = CKPT_DIR / f'{CKPT_PREFIX}_best.pt'
_bd = torch.load(_bp, map_location=DEVICE, weights_only=False)
model.load_state_dict(_bd['model_state_dict'], strict=False)
model.eval()
print(f"Probe target -> {_bp.name}  step {_bd.get('step')}  PPL {_bd.get('val_ppl'):.2f}")
del _bd
import gc; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

import gc, math, sys, torch, numpy as np
for _a in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _a): setattr(sys, _a, None)
model.zero_grad(set_to_none=True)
try: optim.zero_grad(set_to_none=True)
except Exception: pass
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    _free, _total = torch.cuda.mem_get_info()
    print(f'GPU free {_free/1e9:.1f} / {_total/1e9:.1f} GB before probe')

PROBE_BS = 2
_rng = np.random.default_rng(1234)
def _mk(n, bs):
    return [(torch.from_numpy(a).to(DEVICE), torch.from_numpy(b).to(DEVICE))
            for a, b in (get_batch(val_ids, bs, BLOCK_SIZE, _rng) for _ in range(n))]
def _eval_on(batches):
    model.eval(); losses = []
    for x, y in batches:
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(float(loss.item()))
        del loss
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return float(np.mean(losses))

# --- 1. structural health ---
model.eval(); model.set_fock_capture(True)
_hx, _hy = _mk(1, PROBE_BS)[0]
with torch.enable_grad():
    _out = model(_hx, _hy)
del _out
rep = model.fock_component_report()
_cols = ['layer','active_frac','reg_cos_sim','create_entropy','create_alpha_max',
         'rev_entropy','rev_scale','qforce_ratio','destroy_mean']
print('='*72); print('Fock v2.1 STRUCTURAL HEALTH'); print('='*72)
print('  '.join(f'{c[:10]:>10}' for c in _cols))
for dd in rep['per_layer']:
    print('  '.join(f'{str(dd.get(c)):>10}' if isinstance(dd.get(c),(bool,type(None)))
                    else f'{float(dd.get(c)):>10.3f}' for c in _cols))
print('-'*72); print('summary:', {k: round(v,3) for k,v in rep['summary'].items()})
for f in rep.get('flags', []): print('  * '+f)
del _hx, _hy, rep; gc.collect()
if DEVICE == 'cuda': torch.cuda.empty_cache()

# --- 2. PPL attribution ---
_pb = _mk(40, PROBE_BS)
base = _eval_on(_pb); base_ppl = math.exp(base); rows = [('full model', base)]
if getattr(model, 'reverse_channel_scale', None) is not None:
    _s = model.reverse_channel_scale.detach().clone()
    with torch.no_grad(): model.reverse_channel_scale.zero_()
    rows.append(('  - reverse channel', _eval_on(_pb)))
    with torch.no_grad(): model.reverse_channel_scale.copy_(_s)
_thr = model.cfg.register_salience_threshold
try:
    model.cfg.register_salience_threshold = 1e9
    rows.append(('  - registers (all)', _eval_on(_pb)))
finally:
    model.cfg.register_salience_threshold = _thr
print('\n'+'='*72)
print(f"{'arm':<22}{'loss':>10}{'ppl':>10}{'dPPL':>10}")
for n, l in rows:
    p = math.exp(l); print(f'{n:<22}{l:>10.4f}{p:>10.2f}{p-base_ppl:>+10.2f}')

In [ ]:
# == Cell 8: Training curve ============================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

eval_entries = []
alpha_entries = []
if log_path.exists():
    with open(log_path) as f:
        for line in f:
            try:
                e = json.loads(line)
                if 'val_ppl' in e and 'event' not in e:
                    eval_entries.append(e)
                if 'xi_alphas' in e and 'event' not in e:
                    alpha_entries.append(e)
            except Exception:
                pass

if eval_entries:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    ax = axes[0]
    steps_arr = [e['step'] for e in eval_entries]
    ppls = [e['val_ppl'] for e in eval_entries]
    ax.plot(steps_arr, ppls, 'o-',
            label=f'Aniso-Gaussian r={ANISO_RANK} + fock-reg (OWT d=384)',
            linewidth=1.5, color='#C62828')
    ax.axhline(y=9.14, color='green', linestyle='--', alpha=0.7,
               label='Aniso-Gaussian TinyStories best (9.14)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Val PPL')
    ax.set_title(f'Aniso-Gaussian + Fock-Reg -- OpenWebText d=384')
    ax.legend()
    ax.grid(True, alpha=0.3)

    if alpha_entries:
        ax = axes[1]
        a_steps = [e['step'] for e in alpha_entries]
        n_ch = len(alpha_entries[0]['xi_alphas'])
        for k in range(n_ch):
            ax.plot(a_steps, [e['xi_alphas'][k] for e in alpha_entries],
                    'o-', label=f'alpha_{k+1}', markersize=2, linewidth=1.5)
        ax.set_xlabel('Step')
        ax.set_ylabel('alpha_k')
        ax.set_title('Fock coupling strengths (alpha_k)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-0.05, 1.05)

    plt.tight_layout()
    fig.savefig(RESULTS_DIR / 'training_curve_aniso_gaussian_owt.png', dpi=150)
    plt.show()
    print(f'Saved: {RESULTS_DIR / "training_curve_aniso_gaussian_owt.png"}')
else:
    print('No eval data to plot.')